<a href="https://colab.research.google.com/github/sgevatschnaider/estadisticas-para-ciencia-de-datos/blob/main/src/classroom/probabilidad/notebooks/Proceso_Gaussianos_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Índice Interactivo: Procesos Gaussianos
from IPython.display import display, HTML
import html

# =========================================================
# 1. INTRODUCCIÓN
# =========================================================

intro_html_content = """
<div class="content-block" style="margin-top:0;">
    <h2>Resumen General</h2>
    <p>
        Este índice interactivo introduce los <strong>Procesos Gaussianos</strong> como una de las herramientas más elegantes
        de la estadística bayesiana y del aprendizaje automático para modelar funciones desconocidas. La idea central no
        consiste únicamente en asignar incertidumbre a un conjunto finito de parámetros, como ocurre en muchos modelos
        clásicos, sino en asignar una estructura probabilística a la función completa que relaciona entradas y salidas.
        De ese modo, los Procesos Gaussianos permiten trabajar con predicción, interpolación, cuantificación de
        incertidumbre y optimización en contextos donde no se quiere imponer desde el inicio una forma funcional rígida.
    </p>
    <p>
        A lo largo del material se construye el puente conceptual que va desde la distribución normal multivariante hasta
        la definición formal de un Proceso Gaussiano. También se estudian las distribuciones finito-dimensionales, los
        kernels y su interpretación geométrica, el condicionamiento que da lugar a la regresión con GP, la selección de
        hiperparámetros, la relación con otros métodos y las principales limitaciones prácticas. Finalmente, se presentan
        aplicaciones relevantes en regresión probabilística, optimización bayesiana, series temporales, geostatística,
        robótica, control, detección de anomalías y modelos sustitutos para simuladores costosos.
    </p>
    <p>
        El objetivo de este índice es ofrecer una estructura clara, rigurosa y visualmente ordenada para estudiar el
        tema, manteniendo una presentación completa del contenido y una navegación cómoda a través de secciones
        desplegables.
    </p>
</div>
"""

# =========================================================
# 2. SECCIONES
# =========================================================

secciones_principales_data = [
    {
        "id": "sec-0",
        "titulo": "0. Motivación: por qué modelar funciones aleatorias",
        "contenido": """
            <h4>De regresión paramétrica a regresión no paramétrica</h4>
            <p>
                En un modelo de regresión paramétrica clásico se supone que la relación entre la entrada y la salida tiene
                una forma funcional fijada por un número finito de parámetros. Por ejemplo, en una regresión lineal se
                escribe
            </p>
            <blockquote>
                \\[
                y = \\beta_0 + \\beta_1 x + \\varepsilon
                \\]
            </blockquote>
            <p>
                y la incertidumbre recae sobre los coeficientes \\(\\beta_0\\) y \\(\\beta_1\\). Este enfoque es útil cuando
                existe una teoría previa que justifica la forma del modelo o cuando la estructura lineal constituye una
                aproximación razonable. Sin embargo, en muchos problemas reales no sabemos de antemano si la relación es
                lineal, cuadrática, periódica, muy suave, rugosa o con comportamiento local complejo. En esos casos se
                vuelve natural pasar a una perspectiva no paramétrica, en la cual no se fija una forma rígida de la
                función antes de observar los datos.
            </p>
            <p>
                En ese contexto, la meta deja de ser únicamente estimar un pequeño conjunto de parámetros. El problema
                consiste ahora en inferir una función desconocida a partir de observaciones parciales. La idea de los
                Procesos Gaussianos es tratar esa función como un objeto aleatorio. Es decir, en vez de decir “quiero
                ajustar una curva específica con ciertos parámetros”, se plantea “quiero modelar una distribución de
                probabilidad sobre funciones posibles”.
            </p>

            <h4>Incertidumbre sobre funciones, no solo sobre parámetros</h4>
            <p>
                Si escribimos un modelo del tipo
            </p>
            <blockquote>
                \\[
                y = f(x) + \\varepsilon,
                \\]
            </blockquote>
            <p>
                el desafío consiste en aprender la función \\(f\\). En un enfoque bayesiano, antes de observar datos, esa
                función es desconocida y se la dota de una distribución a priori. Después de incorporar observaciones, la
                distribución se actualiza y produce una distribución posterior sobre funciones. Esa es una de las ideas
                más potentes del enfoque: la incertidumbre no se concentra solo en parámetros internos del modelo, sino en
                la función completa que podría explicar los datos.
            </p>
            <p>
                Esto permite responder no solo cuál es la predicción más probable en un punto dado, sino también qué tan
                incierto es el modelo en cada región del dominio. Cerca de zonas con muchas observaciones, la incertidumbre
                tiende a reducirse. En regiones alejadas de los datos, la incertidumbre crece. Esta capacidad de medir la
                confianza de la predicción constituye una de las fortalezas más importantes de los Procesos Gaussianos.
            </p>

            <h4>Ejemplos: interpolación, predicción y optimización</h4>
            <p>
                En problemas de interpolación, un GP permite reconstruir una función continua a partir de pocas
                observaciones, generando una curva suave acompañada por bandas de incertidumbre. En predicción, permite
                estimar valores no observados con una distribución predictiva completa, no solamente con un valor puntual.
                En optimización, especialmente en <em>Bayesian Optimization</em>, el GP se utiliza como un modelo sustituto
                de una función costosa de evaluar, por ejemplo una simulación física, un experimento de laboratorio o el
                ajuste de hiperparámetros de un algoritmo de aprendizaje automático.
            </p>
            <p>
                La motivación general es entonces clara: cuando queremos aprender una relación funcional flexible,
                cuantificar incertidumbre y aprovechar hipótesis estructurales sobre la suavidad o correlación de la
                función, los Procesos Gaussianos ofrecen una herramienta probabilística particularmente rica.
            </p>
        """
    },
    {
        "id": "sec-1",
        "titulo": "1. De la Normal Multivariante al Proceso Gaussiano",
        "contenido": """
            <h4>Recordatorio de la normal multivariante</h4>
            <p>
                La distribución normal multivariante describe el comportamiento conjunto de un vector aleatorio finito. Si
            </p>
            <blockquote>
                \\[
                X \\sim \\mathcal{N}(\\mu, \\Sigma),
                \\]
            </blockquote>
            <p>
                entonces \\(\\mu\\) es el vector de medias y \\(\\Sigma\\) es la matriz de covarianza. El vector
                \\(\\mu\\) fija la localización central de la distribución, mientras que \\(\\Sigma\\) determina la
                dispersión, la orientación geométrica y la relación de dependencia entre componentes.
            </p>
            <p>
                Desde el punto de vista geométrico, \\(\\mu\\) puede interpretarse como el centro de masa probabilístico
                del vector aleatorio, mientras que \\(\\Sigma\\) controla la forma de los elipsoides de nivel. Los
                autovalores de \\(\\Sigma\\) indican cuánta variabilidad existe en distintas direcciones, y los
                autovectores señalan las direcciones principales de esa variación.
            </p>

            <h4>Interpretación geométrica de \\(\\mu\\) y \\(\\Sigma\\)</h4>
            <p>
                Si \\(\\Sigma\\) es diagonal, la variabilidad de cada componente se entiende de manera separada y los ejes
                principales están alineados con las coordenadas originales. Si aparecen términos fuera de la diagonal,
                entonces existe covariación entre componentes y la nube gaussiana se inclina en el espacio. Esa estructura
                es la que permite pasar de una visión puramente marginal a una comprensión verdaderamente conjunta del
                comportamiento aleatorio.
            </p>
            <p>
                Esta interpretación geométrica será fundamental luego, porque en los Procesos Gaussianos el kernel jugará
                un papel análogo al de \\(\\Sigma\\), pero ahora en un contexto funcional.
            </p>

            <h4>Pasar de un vector finito a una familia indexada por \\(x\\)</h4>
            <p>
                El salto conceptual hacia un Proceso Gaussiano consiste en dejar de pensar en un vector aleatorio finito y
                pasar a una familia de variables aleatorias indexadas por una entrada \\(x\\). En lugar de trabajar con
                \\((X_1, X_2, \\dots, X_n)\\), se considera una colección
            </p>
            <blockquote>
                \\[
                \\{f(x) : x \\in \\mathcal{X}\\}.
                \\]
            </blockquote>
            <p>
                Cada valor de \\(x\\) determina una variable aleatoria \\(f(x)\\). El problema ya no es describir la ley de
                un vector fijo, sino definir una ley coherente sobre una familia potencialmente infinita de evaluaciones de
                una función desconocida.
            </p>

            <h4>Definición intuitiva: un GP como “una gaussiana sobre funciones”</h4>
            <p>
                La intuición más difundida es que un Proceso Gaussiano es una generalización infinita de la distribución
                normal multivariante. En la MVN se asigna una ley gaussiana a un vector finito. En un GP se asigna una
                estructura gaussiana a la función completa, en el sentido de que cualquier subconjunto finito de
                evaluaciones de esa función debe tener una distribución normal multivariante.
            </p>
            <p>
                Por eso se dice, de forma intuitiva, que un GP es “una gaussiana sobre funciones”. Antes de ver datos, no
                se elige una sola curva, sino una familia de curvas posibles, cada una con una determinada probabilidad
                inducida por una función media y una función de covarianza.
            </p>
        """
    },
    {
        "id": "sec-2",
        "titulo": "2. Definición formal de Proceso Gaussiano",
        "contenido": """
            <h4>Proceso estocástico \\(\\{f(x): x \\in \\mathcal{X}\\}\\)</h4>
            <p>
                Un proceso estocástico es una colección de variables aleatorias indexadas por un conjunto. En el caso de
                los Procesos Gaussianos, se considera la familia
            </p>
            <blockquote>
                \\[
                \\{f(x) : x \\in \\mathcal{X}\\}.
                \\]
            </blockquote>
            <p>
                Cada entrada \\(x\\) genera una variable aleatoria \\(f(x)\\), y la relación probabilística entre estas
                variables se organiza mediante una función de media y una función de covarianza.
            </p>

            <h4>Definición formal</h4>
            <p>
                Se dice que \\(f\\) sigue un Proceso Gaussiano si
            </p>
            <blockquote>
                \\[
                f \\sim GP(m,k),
                \\]
            </blockquote>
            <p>
                con
            </p>
            <blockquote>
                \\[
                m(x) = \\mathbb{E}[f(x)],
                \\qquad
                k(x,x') = \\operatorname{Cov}(f(x), f(x')).
                \\]
            </blockquote>
            <p>
                La propiedad esencial es que para cualquier conjunto finito de puntos \\(x_1, \\dots, x_n\\), el vector
            </p>
            <blockquote>
                \\[
                (f(x_1), \\dots, f(x_n))
                \\]
            </blockquote>
            <p>
                debe tener distribución normal multivariante.
            </p>

            <h4>Relación entre función media y función de covarianza</h4>
            <p>
                La función media \\(m(x)\\) representa el valor esperado de la función en cada punto del dominio. Si no se
                dispone de una información previa fuerte, es habitual tomar \\(m(x)=0\\), no porque se suponga que la
                función real vale cero, sino porque gran parte de la estructura relevante puede quedar codificada en la
                covarianza y luego corregida por los datos.
            </p>
            <p>
                La función de covarianza \\(k(x,x')\\), también llamada <em>kernel</em>, mide cuánto se parecen
                probabilísticamente los valores de la función en dos entradas distintas. Si \\(k(x,x')\\) es alto para
                puntos cercanos, el modelo expresa la creencia de que entradas próximas generan salidas similares. Si la
                covarianza cae rápido con la distancia, se expresa una correlación local más débil.
            </p>
            <p>
                Así como \\((\\mu,\\Sigma)\\) caracterizan la distribución normal multivariante, la pareja \\((m,k)\\)
                caracteriza al Proceso Gaussiano.
            </p>
        """
    },
    {
        "id": "sec-3",
        "titulo": "3. Distribuciones finito-dimensionales",
        "contenido": """
            <h4>Para \\(x_1, \\dots, x_n\\)</h4>
            <p>
                Aunque un GP está definido sobre una colección potencialmente infinita de puntos, gran parte de su
                comprensión se apoya en su restricción a subconjuntos finitos del dominio. Para cualquier conjunto
                \\(x_1, \\dots, x_n\\), se tiene
            </p>
            <blockquote>
                \\[
                (f(x_1), \\dots, f(x_n)) \\sim \\mathcal{N}(\\mu, K),
                \\]
            </blockquote>
            <p>
                donde el vector de medias está dado por
            </p>
            <blockquote>
                \\[
                \\mu_i = m(x_i),
                \\]
            </blockquote>
            <p>
                y la matriz de covarianza por
            </p>
            <blockquote>
                \\[
                K_{ij} = k(x_i, x_j).
                \\]
            </blockquote>

            <h4>Construcción de la matriz de Gram</h4>
            <p>
                La matriz \\(K\\) se conoce como matriz de Gram asociada al kernel y al conjunto de puntos seleccionado.
                Cada entrada \\((i,j)\\) mide la covarianza entre los valores funcionales en \\(x_i\\) y \\(x_j\\). Esta
                matriz resume la estructura de dependencia que el GP induce sobre el vector de evaluaciones finitas.
            </p>
            <p>
                Por construcción, \\(K\\) debe ser simétrica y semidefinida positiva. La condición de semidefinición
                positiva garantiza que cualquier combinación lineal de las variables aleatorias correspondientes tenga
                varianza no negativa, lo cual es indispensable para que la matriz pueda interpretarse como una verdadera
                matriz de covarianza.
            </p>

            <h4>GP como generalización infinita de la MVN</h4>
            <p>
                La normal multivariante actúa sobre un vector finito. El GP extiende esa lógica a una familia arbitraria
                de evaluaciones funcionales. Cada vez que se selecciona un subconjunto finito de puntos del dominio, se
                obtiene una MVN coherente con todas las demás elecciones finitas. Esa consistencia permite hablar de una
                distribución sobre funciones sin necesidad de enumerar todas sus posibles entradas.
            </p>
            <p>
                Desde esta perspectiva, el GP puede entenderse como una familia coherente de distribuciones gaussianas
                finito-dimensionales. Esa es la razón matemática por la que se dice que generaliza la MVN al caso
                funcional.
            </p>
        """
    },
    {
        "id": "sec-4",
        "titulo": "4. Kernels y estructura de covarianza",
        "contenido": """
            <h4>Qué es un kernel</h4>
            <p>
                Un kernel es una función
            </p>
            <blockquote>
                \\[
                k : \\mathcal{X} \\times \\mathcal{X} \\to \\mathbb{R},
                \\]
            </blockquote>
            <p>
                que mide semejanza, dependencia o proximidad probabilística entre dos entradas. En un Proceso Gaussiano,
                el kernel determina la covarianza entre los valores de la función en diferentes puntos del dominio y, por
                lo tanto, define qué clases de funciones resultan plausibles antes de observar datos.
            </p>

            <h4>Propiedad fundamental: simetría y semidefinición positiva</h4>
            <p>
                Para que una función pueda actuar como kernel válido de un GP debe cumplir dos propiedades esenciales. En
                primer lugar, debe ser simétrica:
            </p>
            <blockquote>
                \\[
                k(x,x') = k(x',x).
                \\]
            </blockquote>
            <p>
                En segundo lugar, para cualquier conjunto finito de puntos \\(x_1, \\dots, x_n\\), la matriz
                \\(K=(k(x_i,x_j))\\) debe ser semidefinida positiva, es decir,
            </p>
            <blockquote>
                \\[
                c^T K c \\ge 0
                \\]
            </blockquote>
            <p>
                para todo vector \\(c \\in \\mathbb{R}^n\\). Esta condición asegura que la matriz sea una covarianza
                válida.
            </p>

            <h4>Ejemplos de kernels</h4>
            <p>
                El kernel RBF o <em>squared exponential</em> es uno de los más utilizados:
            </p>
            <blockquote>
                \\[
                k(x,x') = \\sigma_f^2 \\exp\\left(-\\frac{\\|x-x'\\|^2}{2\\ell^2}\\right).
                \\]
            </blockquote>
            <p>
                Induce funciones muy suaves y expresa una correlación que decrece con la distancia.
            </p>
            <p>
                El kernel de Matérn ofrece una familia más flexible en cuanto a regularidad. Según el valor de su
                parámetro \\(\\nu\\), las trayectorias pueden ser más o menos suaves.
            </p>
            <p>
                El kernel lineal
            </p>
            <blockquote>
                \\[
                k(x,x') = \\sigma_b^2 + \\sigma_v^2 x^T x'
                \\]
            </blockquote>
            <p>
                codifica relaciones afines y se vincula directamente con la regresión lineal bayesiana.
            </p>
            <p>
                El kernel periódico se usa cuando el fenómeno presenta repetición cíclica:
            </p>
            <blockquote>
                \\[
                k(x,x') =
                \\sigma_f^2
                \\exp\\left(
                -\\frac{2\\sin^2(\\pi |x-x'|/p)}{\\ell^2}
                \\right).
                \\]
            </blockquote>

            <h4>Qué información codifica un kernel</h4>
            <p>
                El kernel codifica hipótesis estructurales sobre la función desconocida. Puede expresar suavidad,
                periodicidad, correlación local, amplitud de oscilación y escala de variación. En consecuencia, elegir un
                kernel no es un detalle técnico menor: es una forma de introducir conocimiento previo sobre la geometría y
                la regularidad del fenómeno que se está modelando.
            </p>
        """
    },
    {
        "id": "sec-5",
        "titulo": "5. Geometría e interpretación probabilística",
        "contenido": """
            <h4>Media como tendencia central de la función</h4>
            <p>
                En un GP, la función media \\(m(x)\\) representa la tendencia central del conjunto de funciones que el
                modelo considera plausibles a priori. Si \\(m(x)=0\\), la distribución queda centrada en torno a cero,
                pero la variedad de formas posibles sigue siendo determinada por el kernel. Si se dispone de información
                previa estructurada, la media puede elegirse para reflejar una tendencia global determinada.
            </p>

            <h4>Covarianza como geometría de semejanza entre entradas</h4>
            <p>
                La covarianza no solo mide dispersión. También define una geometría probabilística sobre el espacio de
                entradas. Dos puntos \\(x\\) y \\(x'\\) se consideran próximos desde la perspectiva del modelo cuando
                \\(k(x,x')\\) es alto. En ese caso, el GP presupone que los valores de la función en esos dos puntos
                tienden a moverse juntos. Si el kernel asigna una covarianza pequeña, la dependencia probabilística entre
                ambos valores es débil.
            </p>
            <p>
                De esta manera, el kernel funciona como análogo funcional de la matriz \\(\\Sigma\\) de la normal
                multivariante. Así como \\(\\Sigma\\) organiza la geometría de una distribución gaussiana finita, el
                kernel organiza la geometría del espacio de funciones probables.
            </p>

            <h4>Bandas de incertidumbre</h4>
            <p>
                Cuando se calcula la distribución posterior en un conjunto de puntos, se obtiene una media predictiva y
                una varianza predictiva para cada uno de ellos. En visualizaciones es habitual representar la media junto
                con bandas del tipo
            </p>
            <blockquote>
                \\[
                m_*(x) \\pm 2\\sigma_*(x),
                \\]
            </blockquote>
            <p>
                donde \\(\\sigma_*(x)\\) representa la desviación estándar posterior. Estas bandas muestran cómo cambia la
                confianza del modelo a lo largo del dominio: se angostan donde hay evidencia suficiente y se expanden en
                regiones alejadas de las observaciones.
            </p>

            <h4>Muestras de funciones desde un GP</h4>
            <p>
                Una manera muy intuitiva de entender un GP consiste en seleccionar un conjunto finito de puntos del
                dominio, construir la matriz de Gram y muestrear vectores gaussianos multivariados. Si luego esos puntos
                se unen visualmente, aparecen curvas que pueden interpretarse como funciones extraídas del GP. Cambiar el
                kernel cambia la forma de esas curvas. Algunos kernels inducen trayectorias extremadamente suaves, otros
                generan mayor rugosidad, otros periodicidad y otros estructuras casi lineales.
            </p>
        """
    },
    {
        "id": "sec-6",
        "titulo": "6. Condicionamiento y regresión con GP",
        "contenido": """
            <h4>Observaciones ruidosas</h4>
            <p>
                En problemas de regresión, la función latente no suele observarse de manera exacta. Lo que se observa son
                datos ruidosos del tipo
            </p>
            <blockquote>
                \\[
                y_i = f(x_i) + \\varepsilon_i,
                \\]
            </blockquote>
            <p>
                donde el ruido se modela normalmente como
            </p>
            <blockquote>
                \\[
                \\varepsilon_i \\sim \\mathcal{N}(0,\\sigma_n^2).
                \\]
            </blockquote>
            <p>
                Esto implica que la covarianza de las observaciones debe incorporar un término adicional
                \\(\\sigma_n^2 I\\), que representa la varianza del ruido de medición.
            </p>

            <h4>Distribución conjunta entre observados y no observados</h4>
            <p>
                Si \\(X\\) denota los puntos observados y \\(X_*\\) los puntos en los que queremos predecir, la
                distribución conjunta previa entre los valores observados y los valores latentes futuros puede escribirse
                como
            </p>
            <blockquote>
                \\[
                \\begin{pmatrix}
                y \\\\
                f_*
                \\end{pmatrix}
                \\sim
                \\mathcal{N}
                \\left(
                \\begin{pmatrix}
                m(X) \\\\
                m(X_*)
                \\end{pmatrix},
                \\begin{pmatrix}
                K(X,X)+\\sigma_n^2 I & K(X,X_*) \\\\
                K(X_*,X) & K(X_*,X_*)
                \\end{pmatrix}
                \\right).
                \\]
            </blockquote>
            <p>
                Toda la lógica de la regresión con Procesos Gaussianos se apoya en esta distribución conjunta.
            </p>

            <h4>Distribución condicional</h4>
            <p>
                Al condicionar respecto de los datos observados, se obtiene una distribución posterior para los valores
                no observados. Como la distribución conjunta es gaussiana, la distribución condicional sigue siendo
                gaussiana:
            </p>
            <blockquote>
                \\[
                f_* \\mid X,y,X_* \\sim \\mathcal{N}(\\mu_*,\\Sigma_*).
                \\]
            </blockquote>

            <h4>Media posterior y covarianza posterior</h4>
            <p>
                La media posterior está dada por
            </p>
            <blockquote>
                \\[
                \\mu_* =
                m(X_*)
                + K(X_*,X)
                \\bigl(K(X,X)+\\sigma_n^2 I\\bigr)^{-1}
                (y-m(X)),
                \\]
            </blockquote>
            <p>
                mientras que la covarianza posterior es
            </p>
            <blockquote>
                \\[
                \\Sigma_* =
                K(X_*,X_*)
                - K(X_*,X)
                \\bigl(K(X,X)+\\sigma_n^2 I\\bigr)^{-1}
                K(X,X_*).
                \\]
            </blockquote>
            <p>
                La media posterior actúa como estimación funcional aprendida a partir de los datos, y la covarianza
                posterior cuantifica la incertidumbre remanente.
            </p>

            <h4>Interpretación: aprender una función a partir de datos</h4>
            <p>
                Desde una perspectiva conceptual, la regresión con GP consiste en actualizar una distribución previa sobre
                funciones con información observada. La presencia de datos atrae la media posterior hacia los valores
                observados y reduce la incertidumbre alrededor de ellos. En zonas alejadas de las observaciones, el modelo
                tiende a regresar hacia la media previa y a recuperar mayor incertidumbre.
            </p>
        """
    },
    {
        "id": "sec-7",
        "titulo": "7. Selección de hiperparámetros",
        "contenido": """
            <h4>Longitud de escala</h4>
            <p>
                Uno de los hiperparámetros más importantes es la longitud de escala \\(\\ell\\). Este parámetro controla
                la rapidez con la que decrece la correlación al movernos en el espacio de entradas. Si \\(\\ell\\) es
                grande, el modelo favorece funciones que cambian lentamente y presentan un comportamiento suave. Si
                \\(\\ell\\) es pequeña, el modelo permite variaciones locales más rápidas.
            </p>

            <h4>Varianza del proceso</h4>
            <p>
                La varianza del proceso \\(\\sigma_f^2\\) controla la amplitud vertical típica de las funciones generadas
                por el GP. Un valor alto permite oscilaciones más pronunciadas, mientras que un valor pequeño restringe la
                magnitud de las variaciones funcionales.
            </p>

            <h4>Varianza del ruido</h4>
            <p>
                La varianza del ruido \\(\\sigma_n^2\\) representa cuánto error atribuimos a las observaciones. Si este
                parámetro es muy pequeño, el modelo tenderá a ajustarse fuertemente a los datos. Si es grande, el modelo
                interpretará buena parte de las variaciones observadas como ruido y será menos agresivo al seguirlas.
            </p>

            <h4>Log marginal likelihood</h4>
            <p>
                Una forma estándar de seleccionar estos hiperparámetros consiste en maximizar la verosimilitud marginal de
                los datos, o su logaritmo:
            </p>
            <blockquote>
                \\[
                \\log p(y \\mid X, \\theta)
                =
                -\\frac{1}{2}y^T K_\\theta^{-1} y
                -\\frac{1}{2}\\log |K_\\theta|
                -\\frac{n}{2}\\log(2\\pi).
                \\]
            </blockquote>
            <p>
                Esta expresión equilibra ajuste y complejidad. El primer término recompensa la capacidad del modelo de
                explicar los datos observados. El segundo penaliza configuraciones excesivamente complejas mediante el
                determinante de la matriz de covarianza.
            </p>

            <h4>Sobreajuste vs suavidad</h4>
            <p>
                La selección de hiperparámetros está estrechamente relacionada con el equilibrio entre flexibilidad y
                regularización. Una longitud de escala demasiado pequeña y un ruido demasiado bajo pueden llevar a un
                modelo que siga incluso el ruido aleatorio, produciendo sobreajuste. En cambio, una longitud de escala
                excesivamente grande o un ruido muy alto pueden generar funciones demasiado lisas, incapaces de capturar
                estructuras relevantes de la señal.
            </p>
        """
    },
    {
        "id": "sec-8",
        "titulo": "8. Relación con otros métodos",
        "contenido": """
            <h4>Regresión lineal bayesiana</h4>
            <p>
                La regresión lineal bayesiana puede verse como un caso particular de GP. Si se parte de una familia de
                funciones lineales en parámetros y se asigna una distribución gaussiana a esos parámetros, la función
                inducida sobre las salidas presenta una covarianza determinada por las funciones base. En muchos casos,
                esto conduce a un kernel lineal.
            </p>

            <h4>Splines</h4>
            <p>
                Los splines también buscan ajustar funciones suaves penalizando la rugosidad. Comparten con los GP la idea
                de regularizar la forma funcional. Sin embargo, los GP incorporan esta idea dentro de un marco
                probabilístico completo, lo cual permite trabajar de manera natural con distribuciones predictivas y
                cuantificación de incertidumbre.
            </p>

            <h4>Kernel methods</h4>
            <p>
                Los métodos de kernel, como la regresión kernel ridge o las máquinas de soporte vectorial, comparten con
                los GP el uso de funciones de semejanza definidas sobre el espacio de entradas. En ambos casos, el kernel
                permite trabajar implícitamente en espacios de características de alta dimensión. La diferencia es que los
                GP ofrecen una interpretación plenamente bayesiana y una distribución posterior sobre funciones.
            </p>

            <h4>RKHS como puente teórico</h4>
            <p>
                El espacio de Hilbert con kernel reproducible, o RKHS, constituye un puente matemático muy importante
                entre la perspectiva geométrica de los kernels y los métodos de aprendizaje funcional. Aunque las
                trayectorias típicas de un GP no tienen por qué pertenecer al RKHS asociado, este espacio sigue siendo
                central para comprender normas inducidas por el kernel, regularidad y relaciones con métodos
                deterministas de regularización.
            </p>

            <h4>Diferencia entre GP y redes neuronales</h4>
            <p>
                Una red neuronal parametriza una función mediante capas, pesos y no linealidades, y aprende ajustando esos
                parámetros por optimización. Un GP, en cambio, no fija de entrada una función particular, sino una
                distribución completa sobre funciones. Su fortaleza principal es la cuantificación explícita de la
                incertidumbre y la elegancia probabilística del condicionamiento gaussiano.
            </p>
            <p>
                Existe además una relación teórica profunda, ya que ciertas redes neuronales de ancho infinito convergen a
                Procesos Gaussianos bajo condiciones apropiadas. Aun así, en la práctica ambos enfoques suelen destacarse
                en contextos diferentes.
            </p>
        """
    },
    {
        "id": "sec-9",
        "titulo": "9. Limitaciones prácticas",
        "contenido": """
            <h4>Costo computacional</h4>
            <p>
                Una de las limitaciones más conocidas de los Procesos Gaussianos es su costo computacional. Para entrenar
                un GP con \\(n\\) observaciones es necesario invertir, o al menos factorizar, una matriz de tamaño
                \\(n \\times n\\). Esto conduce normalmente a un costo temporal del orden
            </p>
            <blockquote>
                \\[
                O(n^3),
                \\]
            </blockquote>
            <p>
                y a un costo de memoria del orden
            </p>
            <blockquote>
                \\[
                O(n^2).
                \\]
            </blockquote>
            <p>
                Este comportamiento es manejable en problemas pequeños o medianos, pero se vuelve problemático en
                escenarios de gran escala.
            </p>

            <h4>Problemas en gran escala</h4>
            <p>
                En bases de datos masivas, almacenar la matriz de Gram completa puede resultar inviable, y su
                factorización numérica puede ser demasiado costosa. Además, la optimización de hiperparámetros exige
                repetir estas operaciones varias veces, lo cual incrementa aún más la carga computacional.
            </p>
            <p>
                También pueden surgir problemas numéricos cuando la matriz es casi singular, por ejemplo si hay entradas
                muy cercanas o si ciertos hiperparámetros producen una estructura de covarianza demasiado rígida. Por eso
                es frecuente agregar un pequeño término diagonal adicional para mejorar la estabilidad numérica.
            </p>

            <h4>Aproximaciones sparse / inducing points</h4>
            <p>
                Para superar estas limitaciones se han desarrollado aproximaciones <em>sparse</em> que resumen el
                comportamiento del proceso mediante un conjunto reducido de puntos representativos, conocidos como
                <em>inducing points</em>. La idea general es aproximar la dependencia global utilizando muchas menos
                variables auxiliares que el número total de observaciones.
            </p>
            <p>
                Existen también otras estrategias, como métodos locales, aproximaciones variacionales, kernels
                estructurados y técnicas de álgebra numérica escalable. Estas extensiones permiten trasladar la intuición
                de los GP a problemas más grandes, aunque introducen aproximaciones adicionales.
            </p>
        """
    },
    {
        "id": "sec-10",
        "titulo": "10. Aplicaciones",
        "contenido": """
            <h4>Regresión probabilística</h4>
            <p>
                La aplicación más directa de los GP es la regresión probabilística. En lugar de producir una sola curva
                ajustada, el modelo entrega una media predictiva y una distribución de incertidumbre. Esto es muy valioso
                cuando la toma de decisiones depende no solo del valor esperado, sino también del riesgo o confianza de la
                predicción.
            </p>

            <h4>Bayesian Optimization</h4>
            <p>
                En optimización bayesiana, el GP actúa como modelo sustituto de una función costosa de evaluar. La
                incertidumbre predictiva se combina con una función de adquisición para decidir cuál es el siguiente punto
                más informativo que conviene explorar. Esta estrategia permite optimizar funciones caras con un número
                reducido de evaluaciones.
            </p>

            <h4>Series temporales</h4>
            <p>
                Los GP pueden modelar series temporales mediante kernels que combinen suavidad, tendencia, periodicidad y
                variación local. Esto los vuelve muy útiles para fenómenos con estacionalidad o estructuras temporales
                complejas.
            </p>

            <h4>Geostatística y datos espaciales</h4>
            <p>
                En geostatística, los GP aparecen en forma de campos aleatorios gaussianos y técnicas como kriging. Se
                utilizan para interpolar variables espaciales como temperatura, precipitación, contaminación, humedad del
                suelo o propiedades geológicas, aprovechando la dependencia entre ubicaciones cercanas.
            </p>

            <h4>Robótica y control</h4>
            <p>
                En robótica, los GP se emplean para modelar dinámicas desconocidas, aprender funciones de costo, construir
                mapas probabilísticos y apoyar estrategias de control bajo incertidumbre. Resultan especialmente útiles
                cuando cada interacción con el sistema físico es costosa y debe aprovecharse al máximo.
            </p>

            <h4>Detección de anomalías</h4>
            <p>
                Si un GP modela el comportamiento esperado de una señal, entonces observaciones que se desvían
                significativamente de la distribución predictiva pueden interpretarse como anómalas. Esto tiene
                aplicaciones en monitoreo industrial, mantenimiento predictivo, finanzas y sistemas ciberfísicos.
            </p>

            <h4>Surrogate models de simuladores costosos</h4>
            <p>
                En ingeniería y simulación numérica, es frecuente trabajar con modelos muy caros de evaluar. Un GP puede
                funcionar como <em>surrogate model</em> o metamodelo, aproximando la salida del simulador con un costo
                computacional mucho menor. Esto facilita el análisis de sensibilidad, la exploración de escenarios y la
                optimización.
            </p>
            <p>
                La amplitud de aplicaciones de los Procesos Gaussianos refleja la riqueza de su estructura: combinan
                flexibilidad funcional, inferencia bayesiana, cuantificación de incertidumbre y una interpretación
                geométrica muy potente a través de los kernels.
            </p>
        """
    }
]

# =========================================================
# 3. GENERACIÓN HTML
# =========================================================

def generar_tarjetas_acordeon(datos):
    bloques = []
    for seccion in datos:
        bloques.append(f"""
        <div class="topic-card" id="{html.escape(seccion['id'])}">
            <div class="topic-header">
                <span class="topic-title">{html.escape(seccion['titulo'])}</span>
                <i class="fas fa-chevron-down expand-icon"></i>
            </div>
            <div class="topic-content">
                {seccion['contenido']}
            </div>
        </div>
        """)
    return "\n".join(bloques)

def generar_navegacion(datos):
    chips = []
    for seccion in datos:
        chips.append(
            f'<button class="nav-chip" type="button" data-target="{html.escape(seccion["id"])}">{html.escape(seccion["titulo"])}</button>'
        )
    return "\n".join(chips)

contenido_dinamico_html = generar_tarjetas_acordeon(secciones_principales_data)
navegacion_html = generar_navegacion(secciones_principales_data)

# =========================================================
# 4. PLANTILLA
# =========================================================

plantilla_profesional = r"""
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{{MAIN_TITLE}}</title>

  <script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-MML-AM_CHTML" async></script>
  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
  <link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css" rel="stylesheet">

  <style>
    :root {
      --bg-primary: linear-gradient(135deg, #f0f9ff 0%, #e0f2fe 100%);
      --bg-secondary: rgba(255, 255, 255, 0.85);
      --bg-tertiary: rgba(248, 250, 252, 0.8);
      --text-primary: #0c4a6e;
      --text-secondary: #075985;
      --accent-primary: #0ea5e9;
      --accent-secondary: #38bdf8;
      --accent-gradient: linear-gradient(135deg, var(--accent-primary) 0%, var(--accent-secondary) 100%);
      --border-color: rgba(226, 232, 240, 0.8);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.08);
      --border-radius: 20px;
      --transition: all 0.4s cubic-bezier(0.25, 0.8, 0.25, 1);
      --nav-chip-bg: rgba(255,255,255,0.8);
      --progress-bg: rgba(255,255,255,0.45);
    }

    [data-theme="dark"] {
      --bg-primary: linear-gradient(135deg, #0c4a6e 0%, #1e3a8a 100%);
      --bg-secondary: rgba(30, 58, 138, 0.85);
      --bg-tertiary: rgba(30, 64, 175, 0.7);
      --text-primary: #f0f9ff;
      --text-secondary: #dbeafe;
      --accent-primary: #38bdf8;
      --accent-secondary: #7dd3fc;
      --border-color: rgba(255, 255, 255, 0.15);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.20);
      --nav-chip-bg: rgba(30, 58, 138, 0.65);
      --progress-bg: rgba(255,255,255,0.15);
    }

    * {
      margin: 0;
      padding: 0;
      box-sizing: border-box;
    }

    html {
      scroll-behavior: smooth;
    }

    body {
      font-family: 'Inter', sans-serif;
      line-height: 1.8;
      background: var(--bg-primary);
      color: var(--text-primary);
      transition: var(--transition);
      min-height: 100vh;
      position: relative;
      overflow-x: hidden;
    }

    .particles {
      position: fixed;
      top: 0;
      left: 0;
      width: 100%;
      height: 100%;
      pointer-events: none;
      z-index: -1;
    }

    .particle {
      position: absolute;
      border-radius: 50%;
      animation: float 25s infinite linear;
      opacity: 0;
      background: rgba(255, 255, 255, 0.55);
      box-shadow: 0 0 15px rgba(255,255,255,0.25);
    }

    @keyframes float {
      0% {
        transform: translateY(100vh) rotate(0deg);
        opacity: 0;
      }
      10%, 90% {
        opacity: 0.55;
      }
      100% {
        transform: translateY(-10vh) rotate(360deg);
        opacity: 0;
      }
    }

    .progress-wrapper {
      position: fixed;
      top: 0;
      left: 0;
      width: 100%;
      height: 6px;
      background: var(--progress-bg);
      z-index: 1200;
      backdrop-filter: blur(10px);
    }

    .progress-bar {
      height: 100%;
      width: 0%;
      background: var(--accent-gradient);
      transition: width 0.15s ease;
    }

    .theme-toggle {
      position: fixed;
      top: 1.2rem;
      right: 1.2rem;
      width: 60px;
      height: 60px;
      border: 1px solid var(--border-color);
      border-radius: 50%;
      background: var(--bg-secondary);
      backdrop-filter: blur(15px);
      box-shadow: var(--shadow-card);
      cursor: pointer;
      display: flex;
      align-items: center;
      justify-content: center;
      font-size: 1.35rem;
      color: var(--text-primary);
      transition: var(--transition);
      z-index: 1100;
    }

    .theme-toggle:hover {
      transform: scale(1.15) rotate(180deg);
    }

    .container {
      max-width: 1100px;
      margin: 0 auto;
      padding: 2rem;
      z-index: 1;
      position: relative;
    }

    .header {
      text-align: center;
      margin-bottom: 2rem;
      position: relative;
      padding-top: 1.2rem;
    }

    .header-badge {
      display: inline-flex;
      align-items: center;
      gap: 0.6rem;
      background: var(--bg-secondary);
      border: 1px solid var(--border-color);
      border-radius: 999px;
      padding: 0.7rem 1rem;
      backdrop-filter: blur(15px);
      box-shadow: var(--shadow-card);
      margin-bottom: 1rem;
      color: var(--text-secondary);
      font-size: 0.95rem;
    }

    .main-title {
      font-size: clamp(2.2rem, 5vw, 3.8rem);
      font-weight: 800;
      background: var(--accent-gradient);
      -webkit-background-clip: text;
      -webkit-text-fill-color: transparent;
      background-clip: text;
      margin-bottom: 1rem;
      line-height: 1.15;
    }

    .subtitle {
      color: var(--text-secondary);
      max-width: 900px;
      margin: 0 auto;
      font-size: 1.03rem;
    }

    .content-block,
    .quick-nav,
    .control-bar,
    .topic-card {
      background: var(--bg-secondary);
      backdrop-filter: blur(20px);
      border-radius: var(--border-radius);
      box-shadow: var(--shadow-card);
      border: 2px solid var(--border-color);
    }

    .content-block {
      padding: 2rem;
      margin-top: 2rem;
    }

    .content-block h2 {
      color: var(--text-primary);
      margin-bottom: 1rem;
      font-size: 1.55rem;
    }

    .content-block p,
    .content-block li,
    .subtitle {
      color: var(--text-secondary);
    }

    .quick-nav {
      margin-top: 2rem;
      padding: 1.4rem;
    }

    .quick-nav h3 {
      margin-bottom: 1rem;
      color: var(--text-primary);
      font-size: 1.2rem;
    }

    .nav-grid {
      display: flex;
      flex-wrap: wrap;
      gap: 0.75rem;
    }

    .nav-chip {
      padding: 0.65rem 0.95rem;
      border-radius: 999px;
      background: var(--nav-chip-bg);
      color: var(--text-primary);
      border: 1px solid var(--border-color);
      font-size: 0.92rem;
      transition: var(--transition);
      box-shadow: 0 8px 18px rgba(0,0,0,0.06);
      cursor: pointer;
      font-family: inherit;
      text-align: left;
    }

    .nav-chip:hover {
      transform: translateY(-2px);
      background: var(--bg-tertiary);
    }

    .control-bar {
      margin-top: 1.5rem;
      padding: 1rem 1.2rem;
      display: flex;
      gap: 0.8rem;
      flex-wrap: wrap;
      justify-content: center;
    }

    .control-btn {
      border: 1px solid var(--border-color);
      background: var(--nav-chip-bg);
      color: var(--text-primary);
      border-radius: 999px;
      padding: 0.75rem 1rem;
      cursor: pointer;
      font-family: inherit;
      font-size: 0.95rem;
      transition: var(--transition);
    }

    .control-btn:hover {
      transform: translateY(-2px);
      background: var(--bg-tertiary);
    }

    .lesson-container {
      display: flex;
      flex-direction: column;
      gap: 1.3rem;
      margin-top: 1.6rem;
    }

    .topic-card {
      overflow: hidden;
      transition: var(--transition);
      scroll-margin-top: 90px;
    }

    .topic-card:hover {
      transform: translateY(-2px);
    }

    .topic-header {
      cursor: pointer;
      padding: 1.5rem 2rem;
      display: flex;
      justify-content: space-between;
      align-items: center;
      gap: 1rem;
      user-select: none;
    }

    .topic-title {
      font-size: 1.22rem;
      font-weight: 700;
      color: var(--text-primary);
      line-height: 1.4;
    }

    .expand-icon {
      font-size: 1.15rem;
      color: var(--text-secondary);
      transition: var(--transition);
      flex-shrink: 0;
    }

    .topic-card.open .expand-icon {
      transform: rotate(180deg);
    }

    .topic-content {
      max-height: 0;
      overflow: hidden;
      transition: max-height 1.2s ease, padding 1.2s ease;
      background: var(--bg-tertiary);
    }

    .topic-card.open .topic-content {
      max-height: 6500px;
      padding: 1.6rem 2rem 1.8rem 2rem;
      border-top: 1px solid var(--border-color);
    }

    .topic-content h4 {
      color: var(--text-primary);
      margin-top: 1.5rem;
      margin-bottom: 0.55rem;
      font-size: 1.08rem;
      border-left: 4px solid var(--accent-primary);
      padding-left: 12px;
    }

    .topic-content h4:first-child {
      margin-top: 0;
    }

    .topic-content p,
    .topic-content li {
      color: var(--text-secondary);
      line-height: 1.8;
      margin-bottom: 0.95rem;
    }

    .topic-content strong {
      color: var(--text-primary);
      font-weight: 700;
    }

    .topic-content code {
      font-family: 'JetBrains Mono', monospace;
      background-color: var(--border-color);
      padding: 0.2rem 0.4rem;
      border-radius: 6px;
      font-size: 0.9em;
    }

    .topic-content blockquote {
      text-align: center;
      border-left: 4px solid var(--accent-primary);
      margin: 1.4rem 0;
      font-style: normal;
      color: var(--text-secondary);
      background: rgba(127, 140, 141, 0.05);
      border-radius: 0 10px 10px 0;
      padding: 1rem 1.5rem;
      overflow-x: auto;
    }

    footer {
      text-align: center;
      margin-top: 4rem;
      padding-top: 2rem;
      border-top: 1px solid var(--border-color);
    }

    footer p {
      color: var(--text-secondary);
      font-size: 0.95rem;
      opacity: 0.9;
    }

    @media (max-width: 768px) {
      .container {
        padding: 1rem;
      }

      .topic-header {
        padding: 1.1rem 1.3rem;
      }

      .topic-card.open .topic-content {
        padding: 1.1rem 1.3rem 1.3rem 1.3rem;
      }

      .theme-toggle {
        width: 54px;
        height: 54px;
        top: 1rem;
        right: 1rem;
      }

      .nav-grid {
        gap: 0.55rem;
      }

      .nav-chip {
        font-size: 0.88rem;
      }
    }
  </style>
</head>

<body data-theme="dark">
  <div class="progress-wrapper">
    <div class="progress-bar" id="progressBar"></div>
  </div>

  <div class="particles" id="particles-container"></div>

  <div class="theme-toggle" id="themeToggleButton" title="Cambiar tema">
    <i class="fas fa-moon" id="theme-icon"></i>
  </div>

  <div class="container">
    <header class="header">
      <div class="header-badge">
        <i class="fas fa-wave-square"></i>
        <span>Índice interactivo completo · Procesos Gaussianos</span>
      </div>
      <h1 class="main-title">{{MAIN_TITLE}}</h1>
      <p class="subtitle">
        Desde la normal multivariante hasta la regresión con GP, los kernels, la geometría probabilística, la selección
        de hiperparámetros, las limitaciones computacionales y las aplicaciones modernas.
      </p>
    </header>

    {{INTRO_HTML}}

    <section class="quick-nav">
      <h3>Navegación rápida</h3>
      <div class="nav-grid">
        {{NAV_HTML}}
      </div>
    </section>

    <section class="control-bar">
      <button class="control-btn" id="expandAllBtn" type="button"><i class="fas fa-plus-circle"></i> Expandir todo</button>
      <button class="control-btn" id="collapseAllBtn" type="button"><i class="fas fa-minus-circle"></i> Contraer todo</button>
      <button class="control-btn" id="topBtn" type="button"><i class="fas fa-arrow-up"></i> Volver arriba</button>
    </section>

    <div class="lesson-container">
      {{DYNAMIC_HTML}}
    </div>

    <footer>
      <p>{{FOOTER_TEXT}}</p>
    </footer>
  </div>

  <script>
    (function() {
      const themeToggleButton = document.getElementById('themeToggleButton');
      const themeIcon = document.getElementById('theme-icon');
      const bodyEl = document.body;
      const expandAllBtn = document.getElementById('expandAllBtn');
      const collapseAllBtn = document.getElementById('collapseAllBtn');
      const topBtn = document.getElementById('topBtn');
      const progressBar = document.getElementById('progressBar');

      function setTheme(theme) {
        bodyEl.setAttribute('data-theme', theme);
        try {
          localStorage.setItem('gp_theme', theme);
        } catch (e) {}
        if (themeIcon) {
          themeIcon.className = theme === 'dark' ? 'fas fa-sun' : 'fas fa-moon';
        }
      }

      if (themeToggleButton) {
        themeToggleButton.addEventListener('click', () => {
          const newTheme = (bodyEl.getAttribute('data-theme') || 'dark') === 'dark' ? 'light' : 'dark';
          setTheme(newTheme);
        });
      }

      let preferredTheme = 'dark';
      try {
        preferredTheme = localStorage.getItem('gp_theme') || 'dark';
      } catch (e) {}
      setTheme(preferredTheme);

      const cards = Array.from(document.querySelectorAll('.topic-card'));

      cards.forEach(card => {
        const header = card.querySelector('.topic-header');
        if (header) {
          header.addEventListener('click', () => {
            card.classList.toggle('open');
          });
        }
      });

      if (expandAllBtn) {
        expandAllBtn.addEventListener('click', () => {
          cards.forEach(card => card.classList.add('open'));
        });
      }

      if (collapseAllBtn) {
        collapseAllBtn.addEventListener('click', () => {
          cards.forEach(card => card.classList.remove('open'));
        });
      }

      if (topBtn) {
        topBtn.addEventListener('click', () => {
          window.scrollTo({ top: 0, behavior: 'smooth' });
        });
      }

      const navButtons = Array.from(document.querySelectorAll('.nav-chip'));
      navButtons.forEach(btn => {
        btn.addEventListener('click', () => {
          const targetId = btn.getAttribute('data-target');
          const target = document.getElementById(targetId);
          if (target) {
            target.classList.add('open');
            setTimeout(() => {
              target.scrollIntoView({ behavior: 'smooth', block: 'start' });
            }, 120);
          }
        });
      });

      const firstTopic = document.querySelector('.topic-card');
      if (firstTopic) {
        firstTopic.classList.add('open');
      }

      const container = document.getElementById('particles-container');
      if (container) {
        for (let i = 0; i < 34; i++) {
          const p = document.createElement('div');
          p.className = 'particle';
          p.style.left = Math.random() * 100 + 'vw';
          const size = (Math.random() * 6 + 3);
          p.style.width = size + 'px';
          p.style.height = size + 'px';
          p.style.animationDelay = Math.random() * -20 + 's';
          p.style.animationDuration = (15 + Math.random() * 12) + 's';
          container.appendChild(p);
        }
      }

      function updateProgress() {
        const scrollTop = document.documentElement.scrollTop || document.body.scrollTop;
        const scrollHeight = document.documentElement.scrollHeight - document.documentElement.clientHeight;
        const progress = scrollHeight > 0 ? (scrollTop / scrollHeight) * 100 : 0;
        progressBar.style.width = progress + '%';
      }

      window.addEventListener('scroll', updateProgress);
      updateProgress();
    })();
  </script>
</body>
</html>
"""

# =========================================================
# 5. ENSAMBLADO
# =========================================================

final_html = (
    plantilla_profesional
    .replace("{{MAIN_TITLE}}", "Índice: Procesos Gaussianos")
    .replace("{{INTRO_HTML}}", intro_html_content)
    .replace("{{NAV_HTML}}", navegacion_html)
    .replace("{{DYNAMIC_HTML}}", contenido_dinamico_html)
    .replace("{{FOOTER_TEXT}}", "Material elaborado por el profesor Sergio Gevatschnaider.")
)

display(HTML(final_html))

In [ ]:
# @title Material de estudio: Procesos Gaussianos - Punto 0
from IPython.display import display, HTML
import html

# =========================================================
# 1. CONTENIDO DEL MATERIAL
# =========================================================

titulo_principal = "0. Motivación: por qué modelar funciones aleatorias"

resumen_inicial = r"""
<p>
    Este material desarrolla la motivación conceptual para introducir procesos gaussianos como una herramienta
    para modelar funciones desconocidas cuando no queremos imponer desde el inicio una forma funcional rígida.
    La idea central consiste en pasar desde la incertidumbre sobre un número finito de parámetros hacia una
    incertidumbre sobre funciones completas, manteniendo una interpretación probabilística, geométrica y útil
    para interpolación, predicción y optimización.
</p>
"""

secciones_material = [
    {
        "id": "mat-1",
        "titulo": "Relación entre entrada y salida en estadística y aprendizaje automático",
        "contenido": r"""
        <p>
            Una de las tareas centrales de la estadística y del aprendizaje automático consiste en comprender la relación
            entre una variable de entrada \(x\) y una variable de salida \(y\). En los problemas más simples, se supone
            que esa relación puede describirse mediante una fórmula con pocos parámetros. Por ejemplo, en una regresión
            lineal clásica se escribe
        </p>
        <blockquote>
            \[
            y = \beta_0 + \beta_1 x + \varepsilon,
            \]
        </blockquote>
        <p>
            donde \(\beta_0\) y \(\beta_1\) son parámetros desconocidos y \(\varepsilon\) representa el ruido o
            error aleatorio. En este enfoque, la incertidumbre está concentrada en un número finito de cantidades, a saber,
            los parámetros \(\beta_0\) y \(\beta_1\). Una vez estimados esos parámetros, la función que relaciona
            entrada y salida queda esencialmente determinada. Este modo de pensar es extremadamente útil cuando existe una
            razón teórica fuerte para creer que la relación subyacente es lineal o puede aproximarse adecuadamente mediante
            una familia simple de funciones. Sin embargo, en muchos problemas reales la forma funcional no es conocida de
            antemano, y forzar una estructura demasiado rígida puede conducir a errores sistemáticos.
        </p>
        """
    },
    {
        "id": "mat-2",
        "titulo": "Límite de los modelos paramétricos",
        "contenido": r"""
        <p>
            Para entender esta limitación, conviene observar que un modelo paramétrico impone desde el comienzo una clase
            específica de funciones. Si se usa una regresión lineal, la familia admisible está formada por rectas; si se
            usa una regresión polinómica de grado dos, la familia admisible está formada por parábolas; si se usa un
            polinomio de grado \(m\), la familia admisible está formada por expresiones del tipo
        </p>
        <blockquote>
            \[
            f(x) = \beta_0 + \beta_1 x + \beta_2 x^2 + \cdots + \beta_m x^m.
            \]
        </blockquote>
        <p>
            En todos estos casos, el problema consiste en elegir un vector finito de parámetros
            \((\beta_0, \beta_1, \dots, \beta_m)\). Esto hace que el modelo sea interpretable y manejable, pero
            también significa que todo comportamiento posible debe caber dentro de esa familia elegida de antemano. Si la
            función real no se parece a ninguna de esas formas, el modelo fallará no porque los datos sean malos, sino
            porque la hipótesis estructural era demasiado restrictiva.
        </p>
        """
    },
    {
        "id": "mat-3",
        "titulo": "Pasar de incertidumbre sobre parámetros a incertidumbre sobre funciones",
        "contenido": r"""
        <p>
            Esta observación conduce a una idea más flexible y más profunda: en lugar de postular una forma funcional fija
            y desconocer solo los parámetros, podemos considerar que la función misma es un objeto incierto. Dicho de otro
            modo, en vez de pensar que el mundo está gobernado por una única función determinista de forma conocida salvo
            por algunos coeficientes, podemos tratar a la función como una variable aleatoria. Esta es la intuición que
            motiva los procesos gaussianos. Allí no se parte de una ecuación cerrada como
        </p>
        <blockquote>
            \[
            y = \beta_0 + \beta_1 x + \varepsilon,
            \]
        </blockquote>
        <p>
            sino de la idea de que existe una función desconocida \(f\) tal que
        </p>
        <blockquote>
            \[
            y = f(x) + \varepsilon,
            \]
        </blockquote>
        <p>
            y de que nuestra incertidumbre recae sobre la forma completa de \(f\), no únicamente sobre un pequeño
            conjunto de parámetros numéricos.
        </p>
        """
    },
    {
        "id": "mat-4",
        "titulo": "La amplitud del cambio conceptual",
        "contenido": r"""
        <p>
            Este cambio conceptual es fundamental. Cuando se habla de incertidumbre sobre parámetros, se está diciendo que
            la forma general de la función ya está decidida y que solo falta ajustar algunos números. Cuando se habla de
            incertidumbre sobre funciones, se está diciendo algo mucho más amplio: aún no sabemos si la relación es casi
            lineal, suavemente curva, altamente oscilatoria o localmente estable con cambios bruscos en otras regiones. En
            vez de seleccionar una sola forma rígida, se construye una distribución de probabilidad sobre posibles
            funciones. El objetivo del aprendizaje ya no es escoger una única ecuación dentro de una familia estrecha, sino
            actualizar una creencia probabilística sobre un espacio amplio de comportamientos funcionales a medida que se
            observan datos.
        </p>
        """
    },
    {
        "id": "mat-5",
        "titulo": "Regresión no paramétrica y estructura del Proceso Gaussiano",
        "contenido": r"""
        <p>
            Desde este punto de vista, la regresión no paramétrica no significa ausencia de estructura, sino una estructura
            de otro tipo. En la regresión paramétrica, la complejidad del modelo está controlada por un número finito de
            parámetros. En la regresión no paramétrica, la complejidad no está fijada de antemano por una cantidad pequeña
            de coeficientes, sino por supuestos más globales sobre suavidad, regularidad y dependencia entre valores de la
            función en distintos puntos. En un proceso gaussiano, por ejemplo, no se especifica una fórmula cerrada para
            \(f\), pero sí se define una media \(m(x)\) y una función de covarianza o kernel \(k(x,x')\). La notación
            estándar es
        </p>
        <blockquote>
            \[
            f \sim GP(m,k).
            \]
        </blockquote>
        <p>
            Esto significa que para cualquier conjunto finito de puntos \(x_1, \dots, x_n\), el vector
        </p>
        <blockquote>
            \[
            (f(x_1), f(x_2), \dots, f(x_n))
            \]
        </blockquote>
        <p>
            tiene distribución normal multivariante con vector de medias
        </p>
        <blockquote>
            \[
            \mu = (m(x_1), m(x_2), \dots, m(x_n))
            \]
        </blockquote>
        <p>
            y matriz de covarianza
        </p>
        <blockquote>
            \[
            K_{ij} = k(x_i, x_j).
            \]
        </blockquote>
        <p>
            La idea es que el kernel codifica la relación entre valores de la función en distintos puntos del dominio. Si
            \(k(x,x')\) es grande cuando \(x\) y \(x'\) están cerca, el modelo expresa la creencia de que la función
            varía de manera suave. Si el kernel permite oscilaciones periódicas, entonces se está incorporando la
            posibilidad de comportamiento cíclico. La flexibilidad del modelo proviene de que no se elige una fórmula
            explícita para la función, sino una estructura probabilística para su comportamiento global.
        </p>
        """
    },
    {
        "id": "mat-6",
        "titulo": "Por qué esta perspectiva es más rica que la regresión clásica",
        "contenido": r"""
        <p>
            Esta perspectiva es especialmente poderosa porque permite representar incertidumbre de una manera mucho más rica
            que en la regresión clásica. En un modelo paramétrico lineal, después de estimar \(\beta_0\) y \(\beta_1\),
            la predicción en un nuevo punto \(x_*\) suele escribirse como
        </p>
        <blockquote>
            \[
            \hat{y}_* = \hat{\beta}_0 + \hat{\beta}_1 x_*.
            \]
        </blockquote>
        <p>
            Puede acompañarse esa predicción con un intervalo de confianza o de predicción, pero la forma funcional sigue
            siendo una recta. En cambio, con un proceso gaussiano, antes de observar datos se tiene una distribución sobre
            funciones posibles, y después de observar datos se obtiene una distribución posterior sobre funciones
            compatibles con la evidencia. Esto implica que en cada punto nuevo \(x_*\) no solo se produce una predicción
            puntual, sino también una medida de incertidumbre local. En términos intuitivos, el modelo no responde
            únicamente “la salida esperada aquí es tal valor”, sino también “y este es el grado de confianza que tengo en
            esa estimación”.
        </p>
        """
    },
    {
        "id": "mat-7",
        "titulo": "Ejemplo de motivación: ajuste flexible con incertidumbre",
        "contenido": r"""
        <p>
            Para ver por qué esto es importante, imaginemos un conjunto de observaciones
            \((x_1,y_1), \dots, (x_n,y_n)\) tomadas de un fenómeno físico. Supongamos que los puntos observados sugieren
            una curva suave, pero no hay teoría suficiente para afirmar que la relación sea lineal o cuadrática. Si se
            utiliza un modelo paramétrico demasiado simple, la curva ajustada puede ignorar rasgos locales relevantes. Si
            se utiliza un polinomio de grado alto, puede aparecer sobreajuste, con oscilaciones artificiales entre los
            puntos. Un proceso gaussiano ofrece una alternativa elegante: permite interpolar o suavizar los datos
            respetando una noción de regularidad impuesta por el kernel, y además cuantifica cómo crece la incertidumbre en
            las zonas donde hay pocos datos.
        </p>
        """
    },
    {
        "id": "mat-8",
        "titulo": "Interpolación como consecuencia probabilística",
        "contenido": r"""
        <p>
            La interpolación es una de las aplicaciones más intuitivas. Supongamos que se observa la temperatura en ciertas
            horas del día y se quiere estimar el valor en horas no medidas. Si los tiempos observados están cerca unos de
            otros, es razonable esperar que la función temperatura-tiempo sea relativamente suave. Un proceso gaussiano con
            un kernel apropiado puede usar esa idea para generar una curva interpolante. Lo importante es que la
            interpolación no aparece como un mero truco geométrico, sino como una consecuencia probabilística del supuesto
            de que valores cercanos en el tiempo tienen correlación alta. Si \(x\) y \(x'\) representan tiempos
            próximos, un kernel de la forma
        </p>
        <blockquote>
            \[
            k(x,x') = \sigma^2 \exp\left(-\frac{(x-x')^2}{2\ell^2}\right)
            \]
        </blockquote>
        <p>
            expresa justamente esa intuición. El parámetro \(\sigma^2\) controla la escala vertical de variación de la
            función, mientras que \(\ell\) controla la longitud de escala, es decir, qué tan rápido decae la correlación
            cuando los puntos se separan. Si \(\ell\) es grande, la función tenderá a ser muy suave; si \(\ell\) es
            pequeño, el modelo permitirá cambios más abruptos.
        </p>
        """
    },
    {
        "id": "mat-9",
        "titulo": "Predicción en nuevas entradas",
        "contenido": r"""
        <p>
            La predicción es otra motivación esencial. En muchos problemas no se quiere únicamente reconstruir valores
            faltantes entre observaciones, sino anticipar el comportamiento futuro o estimar salidas en nuevas entradas.
            Por ejemplo, si \(x\) representa la dosis de un medicamento y \(y\) la respuesta fisiológica, puede ser
            difícil justificar una fórmula paramétrica precisa antes de ver los datos. Un proceso gaussiano permite
            aprender una relación funcional flexible a partir de observaciones previas y producir predicciones para nuevas
            dosis. Lo notable es que el resultado no es solo una curva estimada, sino una distribución predictiva. En un
            nuevo punto \(x_*\), la predicción suele resumirse mediante una media posterior \(m_n(x_*)\) y una
            varianza posterior \(s_n^2(x_*)\). La media posterior representa la mejor estimación del valor funcional,
            mientras que la varianza posterior representa la incertidumbre asociada. En regiones bien cubiertas por los
            datos, la varianza posterior tiende a ser pequeña; en regiones alejadas de las observaciones, tiende a
            aumentar. Esto refleja una idea epistemológica importante: el modelo sabe distinguir entre lo que ha aprendido
            con evidencia y lo que sigue siendo incierto.
        </p>
        """
    },
    {
        "id": "mat-10",
        "titulo": "Optimización de funciones costosas",
        "contenido": r"""
        <p>
            La optimización constituye una tercera motivación de gran relevancia. En muchos contextos, evaluar una función
            es costoso. Puede tratarse de una simulación numérica compleja, de un experimento de laboratorio o del
            entrenamiento completo de un modelo de aprendizaje automático para una configuración dada de hiperparámetros. En
            esos casos, no conviene probar puntos al azar ni recorrer exhaustivamente el espacio de posibilidades. Lo que
            se desea es construir una estrategia inteligente para decidir dónde evaluar a continuación. Los procesos
            gaussianos son muy útiles aquí porque proporcionan una aproximación probabilística de la función objetivo. Si
            \(f(x)\) es la cantidad que queremos maximizar o minimizar y solo conocemos sus valores en algunos puntos, un
            proceso gaussiano puede servir como modelo sustituto de esa función. A partir de la media posterior y la
            incertidumbre posterior, se diseña una regla de decisión que equilibra exploración y explotación. Explotar
            significa evaluar donde la media estimada es favorable; explorar significa evaluar donde la incertidumbre es
            grande y podría esconderse un valor aún mejor. Esta es la base de la optimización bayesiana.
        </p>
        """
    },
    {
        "id": "mat-11",
        "titulo": "Exploración y explotación",
        "contenido": r"""
        <p>
            Para comprender mejor esta idea, imaginemos que queremos encontrar el valor de \(x\) que maximiza una función
            desconocida \(f(x)\), pero cada evaluación de \(f\) requiere varias horas de cómputo. Si solo siguiéramos
            la media estimada, podríamos quedar atrapados en una región prometedora sin descubrir otras mejores. Si solo
            siguiéramos la incertidumbre, perderíamos tiempo explorando zonas poco relevantes. El proceso gaussiano permite
            combinar ambos aspectos. El modelo no solo resume lo que parece probable, sino también lo que aún podría
            sorprender. Esta capacidad de integrar conocimiento e ignorancia de forma cuantitativa es una de las razones
            más profundas por las cuales resulta natural modelar funciones aleatorias.
        </p>
        """
    },
    {
        "id": "mat-12",
        "titulo": "Puente con la normal multivariante",
        "contenido": r"""
        <p>
            En un plano más conceptual, la motivación para introducir procesos gaussianos también puede entenderse como una
            ampliación de la geometría probabilística que ya aparece en la normal multivariante. En una distribución normal
            multivariante, un vector aleatorio \(X\) queda caracterizado por un vector de medias \(\mu\) y una matriz
            de covarianza \(\Sigma\). La covarianza describe cómo varían conjuntamente las componentes y determina la
            geometría de los elipsoides de densidad. En un proceso gaussiano, esa lógica se traslada del espacio de
            vectores finitos al espacio de funciones. La media \(m(x)\) juega el papel de tendencia central funcional, y
            el kernel \(k(x,x')\) reemplaza a la matriz de covarianza, describiendo cómo se relacionan los valores de la
            función en distintos puntos del dominio. Así, un proceso gaussiano puede verse como la versión
            infinita-dimensional de la normal multivariante. Esta continuidad teórica explica por qué su estudio encaja de
            manera natural después de la media, la covarianza, la dependencia y la normal multivariante.
        </p>
        """
    },
    {
        "id": "mat-13",
        "titulo": "Cierre",
        "contenido": r"""
        <p>
            En definitiva, modelar funciones aleatorias es una respuesta a una limitación central de los enfoques
            rígidamente paramétricos. Cuando no se conoce la forma funcional y, al mismo tiempo, se desea trabajar de
            manera probabilística y cuantificar incertidumbre, resulta insuficiente poner una distribución solo sobre un
            conjunto de coeficientes. Lo que hace falta es una distribución sobre funciones. Los procesos gaussianos
            ofrecen precisamente esa posibilidad de una forma matemáticamente elegante, geométricamente interpretable y
            computacionalmente útil. Permiten interpolar datos, predecir valores en nuevos puntos y optimizar funciones
            costosas, todo ello dentro de un marco unificado en el que la incertidumbre no desaparece, sino que se vuelve
            una parte explícita del modelo. Por eso constituyen una herramienta tan poderosa para pasar de una visión
            estática de la regresión a una visión más flexible, más rica y más realista del aprendizaje a partir de datos.
        </p>
        """
    }
]

# =========================================================
# 2. GENERACIÓN HTML DINÁMICO
# =========================================================

def generar_bloques_material(secciones):
    bloques = []
    for sec in secciones:
        bloques.append(f"""
        <div class="topic-card" id="{html.escape(sec['id'])}">
            <div class="topic-header">
                <span class="topic-title">{html.escape(sec['titulo'])}</span>
                <span class="expand-icon">⌄</span>
            </div>
            <div class="topic-content">
                {sec['contenido']}
            </div>
        </div>
        """)
    return "\n".join(bloques)

material_html = generar_bloques_material(secciones_material)

# =========================================================
# 3. PLANTILLA FINAL
# =========================================================

html_final = f"""
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{titulo_principal}</title>

    <script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-MML-AM_CHTML" async></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">

    <style>
        :root {{
            --bg-primary: linear-gradient(135deg, #f0f9ff 0%, #e0f2fe 100%);
            --bg-secondary: rgba(255, 255, 255, 0.85);
            --bg-tertiary: rgba(248, 250, 252, 0.8);
            --text-primary: #0c4a6e;
            --text-secondary: #075985;
            --accent-primary: #0ea5e9;
            --accent-secondary: #38bdf8;
            --accent-gradient: linear-gradient(135deg, var(--accent-primary) 0%, var(--accent-secondary) 100%);
            --border-color: rgba(226, 232, 240, 0.8);
            --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.08);
            --border-radius: 20px;
            --transition: all 0.4s cubic-bezier(0.25, 0.8, 0.25, 1);
            --progress-bg: rgba(255,255,255,0.45);
        }}

        [data-theme="dark"] {{
            --bg-primary: linear-gradient(135deg, #0c4a6e 0%, #1e3a8a 100%);
            --bg-secondary: rgba(30, 58, 138, 0.85);
            --bg-tertiary: rgba(30, 64, 175, 0.7);
            --text-primary: #f0f9ff;
            --text-secondary: #dbeafe;
            --accent-primary: #38bdf8;
            --accent-secondary: #7dd3fc;
            --border-color: rgba(255, 255, 255, 0.15);
            --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.20);
            --progress-bg: rgba(255,255,255,0.15);
        }}

        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }}

        html {{
            scroll-behavior: smooth;
        }}

        body {{
            font-family: 'Inter', sans-serif;
            line-height: 1.8;
            background: var(--bg-primary);
            color: var(--text-primary);
            transition: var(--transition);
            min-height: 100vh;
            position: relative;
            overflow-x: hidden;
            padding-bottom: 3rem;
        }}

        .particles {{
            position: fixed;
            inset: 0;
            pointer-events: none;
            z-index: -1;
        }}

        .particle {{
            position: absolute;
            border-radius: 50%;
            animation: float 25s infinite linear;
            opacity: 0;
            background: rgba(255, 255, 255, 0.55);
            box-shadow: 0 0 15px rgba(255,255,255,0.25);
        }}

        @keyframes float {{
            0% {{
                transform: translateY(100vh) rotate(0deg);
                opacity: 0;
            }}
            10%, 90% {{
                opacity: 0.55;
            }}
            100% {{
                transform: translateY(-10vh) rotate(360deg);
                opacity: 0;
            }}
        }}

        .progress-wrapper {{
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 6px;
            background: var(--progress-bg);
            z-index: 1200;
            backdrop-filter: blur(10px);
        }}

        .progress-bar {{
            height: 100%;
            width: 0%;
            background: var(--accent-gradient);
            transition: width 0.15s ease;
        }}

        .theme-toggle {{
            position: fixed;
            top: 1.2rem;
            right: 1.2rem;
            width: 60px;
            height: 60px;
            border: 1px solid var(--border-color);
            border-radius: 50%;
            background: var(--bg-secondary);
            backdrop-filter: blur(15px);
            box-shadow: var(--shadow-card);
            cursor: pointer;
            display: flex;
            align-items: center;
            justify-content: center;
            font-size: 1.35rem;
            color: var(--text-primary);
            transition: var(--transition);
            z-index: 1100;
        }}

        .theme-toggle:hover {{
            transform: scale(1.12) rotate(180deg);
        }}

        .container {{
            max-width: 1100px;
            margin: 0 auto;
            padding: 2rem;
            position: relative;
            z-index: 1;
        }}

        .hero,
        .summary-card,
        .topic-card {{
            background: var(--bg-secondary);
            backdrop-filter: blur(20px);
            border-radius: var(--border-radius);
            box-shadow: var(--shadow-card);
            border: 2px solid var(--border-color);
        }}

        .hero {{
            padding: 2rem;
            margin-top: 1rem;
            margin-bottom: 1.5rem;
            text-align: center;
        }}

        .hero-badge {{
            display: inline-flex;
            align-items: center;
            gap: 0.5rem;
            background: var(--bg-tertiary);
            border: 1px solid var(--border-color);
            border-radius: 999px;
            padding: 0.65rem 1rem;
            margin-bottom: 1rem;
            color: var(--text-secondary);
            font-size: 0.95rem;
            font-weight: 600;
        }}

        .main-title {{
            font-size: clamp(2.2rem, 5vw, 3.6rem);
            font-weight: 800;
            background: var(--accent-gradient);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            background-clip: text;
            margin-bottom: 1rem;
            line-height: 1.12;
        }}

        .subtitle {{
            color: var(--text-secondary);
            font-size: 1.03rem;
            max-width: 900px;
            margin: 0 auto;
        }}

        .summary-card {{
            padding: 1.6rem 1.8rem;
            margin-bottom: 1.6rem;
        }}

        .summary-card h2 {{
            color: var(--text-primary);
            font-size: 1.45rem;
            margin-bottom: 0.9rem;
        }}

        .summary-card p {{
            color: var(--text-secondary);
            margin-bottom: 1rem;
        }}

        .lesson-container {{
            display: flex;
            flex-direction: column;
            gap: 1.2rem;
        }}

        .topic-card {{
            overflow: hidden;
            transition: var(--transition);
        }}

        .topic-card:hover {{
            transform: translateY(-2px);
        }}

        .topic-header {{
            cursor: pointer;
            padding: 1.35rem 1.7rem;
            display: flex;
            justify-content: space-between;
            align-items: center;
            gap: 1rem;
            user-select: none;
        }}

        .topic-title {{
            font-size: 1.15rem;
            font-weight: 700;
            color: var(--text-primary);
            line-height: 1.4;
        }}

        .expand-icon {{
            font-size: 1.45rem;
            line-height: 1;
            color: var(--text-secondary);
            transition: var(--transition);
            flex-shrink: 0;
        }}

        .topic-card.open .expand-icon {{
            transform: rotate(180deg);
        }}

        .topic-content {{
            max-height: 0;
            overflow: hidden;
            transition: max-height 0.85s ease, padding 0.85s ease;
            background: var(--bg-tertiary);
            padding: 0 1.7rem;
        }}

        .topic-card.open .topic-content {{
            padding: 1.4rem 1.7rem 1.7rem 1.7rem;
            border-top: 1px solid var(--border-color);
        }}

        .topic-content p,
        .topic-content li {{
            color: var(--text-secondary);
            margin-bottom: 1rem;
        }}

        .topic-content blockquote {{
            text-align: center;
            border-left: 4px solid var(--accent-primary);
            margin: 1.3rem 0;
            color: var(--text-secondary);
            background: rgba(127, 140, 141, 0.05);
            border-radius: 0 10px 10px 0;
            padding: 1rem 1.3rem;
            overflow-x: auto;
        }}

        .footer {{
            text-align: center;
            margin-top: 2.8rem;
            padding-top: 1.6rem;
            border-top: 1px solid var(--border-color);
            color: var(--text-secondary);
            font-size: 0.95rem;
        }}

        @media (max-width: 768px) {{
            .container {{
                padding: 1rem;
            }}

            .hero,
            .summary-card {{
                padding: 1.2rem;
            }}

            .topic-header {{
                padding: 1rem 1.2rem;
            }}

            .topic-content {{
                padding: 0 1.2rem;
            }}

            .topic-card.open .topic-content {{
                padding: 1rem 1.2rem 1.2rem 1.2rem;
            }}

            .theme-toggle {{
                width: 54px;
                height: 54px;
                top: 1rem;
                right: 1rem;
            }}
        }}
    </style>
</head>

<body data-theme="dark">
    <div class="progress-wrapper">
        <div class="progress-bar" id="progressBar"></div>
    </div>

    <div class="particles" id="particles-container"></div>

    <div class="theme-toggle" id="themeToggleButton" title="Cambiar tema">
        <span id="themeIcon">☾</span>
    </div>

    <div class="container">
        <section class="hero">
            <div class="hero-badge">Material de estudio · Procesos Gaussianos</div>
            <h1 class="main-title">{html.escape(titulo_principal)}</h1>
            <p class="subtitle">
                Desarrollo completo del punto 0 con estética visual tipo glassmorphism, cambio de tema,
                subtítulos deslizantes y fórmulas renderizadas con MathJax, listo para Google Colab.
            </p>
        </section>

        <section class="summary-card">
            <h2>Resumen inicial</h2>
            {resumen_inicial}
        </section>

        <section class="lesson-container">
            {material_html}
        </section>

        <div class="footer">
            Material elaborado por el profesor Sergio Gevatschnaider.
        </div>
    </div>

    <script>
        (function() {{
            const bodyEl = document.body;
            const themeToggleButton = document.getElementById('themeToggleButton');
            const themeIcon = document.getElementById('themeIcon');
            const progressBar = document.getElementById('progressBar');
            const cards = Array.from(document.querySelectorAll('.topic-card'));

            function setTheme(theme) {{
                bodyEl.setAttribute('data-theme', theme);
                try {{
                    localStorage.setItem('gp_material_theme', theme);
                }} catch (e) {{}}
                if (themeIcon) {{
                    themeIcon.textContent = theme === 'dark' ? '☀' : '☾';
                }}
            }}

            let preferredTheme = 'dark';
            try {{
                preferredTheme = localStorage.getItem('gp_material_theme') || 'dark';
            }} catch (e) {{}}
            setTheme(preferredTheme);

            if (themeToggleButton) {{
                themeToggleButton.addEventListener('click', () => {{
                    const current = bodyEl.getAttribute('data-theme') || 'dark';
                    setTheme(current === 'dark' ? 'light' : 'dark');
                }});
            }}

            function typesetMath(target) {{
                if (window.MathJax && window.MathJax.Hub) {{
                    try {{
                        MathJax.Hub.Queue(["Typeset", MathJax.Hub, target || document.body]);
                    }} catch (e) {{}}
                }}
            }}

            function openCard(card) {{
                const content = card.querySelector('.topic-content');
                if (!content) return;
                card.classList.add('open');
                content.style.maxHeight = content.scrollHeight + 80 + 'px';
                typesetMath(content);
                setTimeout(() => {{
                    content.style.maxHeight = content.scrollHeight + 80 + 'px';
                }}, 250);
            }}

            function closeCard(card) {{
                const content = card.querySelector('.topic-content');
                if (!content) return;
                content.style.maxHeight = content.scrollHeight + 'px';
                requestAnimationFrame(() => {{
                    card.classList.remove('open');
                    content.style.maxHeight = '0px';
                }});
            }}

            function toggleCard(card) {{
                if (card.classList.contains('open')) {{
                    closeCard(card);
                }} else {{
                    openCard(card);
                }}
            }}

            function refreshOpenCards() {{
                cards.forEach(card => {{
                    if (card.classList.contains('open')) {{
                        const content = card.querySelector('.topic-content');
                        if (content) {{
                            content.style.maxHeight = content.scrollHeight + 80 + 'px';
                        }}
                    }}
                }});
            }}

            cards.forEach(card => {{
                const header = card.querySelector('.topic-header');
                if (header) {{
                    header.addEventListener('click', () => toggleCard(card));
                }}
            }});

            if (cards.length > 0) {{
                openCard(cards[0]);
            }}

            const container = document.getElementById('particles-container');
            if (container) {{
                for (let i = 0; i < 36; i++) {{
                    const p = document.createElement('div');
                    p.className = 'particle';
                    p.style.left = Math.random() * 100 + 'vw';
                    const size = (Math.random() * 7 + 3);
                    p.style.width = size + 'px';
                    p.style.height = size + 'px';
                    p.style.animationDelay = Math.random() * -20 + 's';
                    p.style.animationDuration = (15 + Math.random() * 12) + 's';
                    container.appendChild(p);
                }}
            }}

            function updateProgress() {{
                const scrollTop = document.documentElement.scrollTop || document.body.scrollTop;
                const scrollHeight = document.documentElement.scrollHeight - document.documentElement.clientHeight;
                const progress = scrollHeight > 0 ? (scrollTop / scrollHeight) * 100 : 0;
                progressBar.style.width = progress + '%';
            }}

            window.addEventListener('scroll', updateProgress);
            window.addEventListener('resize', refreshOpenCards);

            setTimeout(() => {{
                refreshOpenCards();
                updateProgress();
                typesetMath(document.body);
            }}, 800);

            setTimeout(() => {{
                refreshOpenCards();
                updateProgress();
            }}, 1600);
        }})();
    </script>
</body>
</html>
"""

display(HTML(html_final))

In [ ]:
# @title Material de estudio: Procesos Gaussianos - Punto 1
from IPython.display import display, HTML
import html

# =========================================================
# 1. CONTENIDO DEL MATERIAL
# =========================================================

titulo_principal = "1. De la Normal Multivariante al Proceso Gaussiano"

resumen_inicial = r"""
<p>
    Este material desarrolla el pasaje conceptual y matemático desde la distribución normal multivariante
    hacia el proceso gaussiano. La idea central es extender una distribución gaussiana definida sobre un
    vector aleatorio finito a una distribución gaussiana definida sobre una función completa. Así, la media
    y la covarianza dejan de describir solamente un vector y pasan a organizar la geometría probabilística
    de un espacio de funciones.
</p>
"""

secciones_material = [
    {
        "id": "mat-1",
        "titulo": "Introducción: de un vector gaussiano a una función gaussiana",
        "contenido": r"""
        <p>
            Para comprender qué es un proceso gaussiano, conviene comenzar por un objeto ya conocido: la normal multivariante.
            La idea central del paso de la normal multivariante al proceso gaussiano consiste en extender una distribución
            gaussiana definida sobre un vector aleatorio finito a una distribución gaussiana definida sobre una función completa.
            Ese pasaje parece, al principio, un salto grande, pero en realidad es una generalización muy natural del lenguaje
            de media, covarianza, dependencia y geometría que ya aparece en el estudio de la distribución normal en dimensión finita.
        </p>
        """
    },
    {
        "id": "mat-2",
        "titulo": "Vector aleatorio finito y definición de la normal multivariante",
        "contenido": r"""
        <p>
            Supongamos primero que tenemos un vector aleatorio de dimensión \(d\),
        </p>
        <blockquote>
            \[
            X=
            \begin{pmatrix}
            X_1\\
            X_2\\
            \vdots\\
            X_d
            \end{pmatrix}.
            \]
        </blockquote>
        <p>
            Decimos que \(X\) tiene distribución normal multivariante con vector de medias \(\mu\) y matriz de
            covarianza \(\Sigma\), y escribimos
        </p>
        <blockquote>
            \[
            X \sim N(\mu,\Sigma),
            \]
        </blockquote>
        <p>
            cuando su comportamiento probabilístico queda completamente determinado por esos dos objetos:
        </p>
        <blockquote>
            \[
            \mu = E[X]
            \]
        </blockquote>
        <p>
            y
        </p>
        <blockquote>
            \[
            \Sigma = E[(X-\mu)(X-\mu)^T].
            \]
        </blockquote>
        """
    },
    {
        "id": "mat-3",
        "titulo": "Interpretación de la media y la matriz de covarianza",
        "contenido": r"""
        <p>
            Aquí, \(\mu\) es un vector en \(R^d\) que contiene las medias de cada componente,
        </p>
        <blockquote>
            \[
            \mu=
            \begin{pmatrix}
            E[X_1]\\
            E[X_2]\\
            \vdots\\
            E[X_d]
            \end{pmatrix},
            \]
        </blockquote>
        <p>
            mientras que \(\Sigma\) es una matriz \(d \times d\) cuya entrada \((i,j)\) es
        </p>
        <blockquote>
            \[
            \Sigma_{ij}=Cov(X_i,X_j).
            \]
        </blockquote>
        <p>
            La diagonal de \(\Sigma\) contiene las varianzas \(Var(X_i)\), y las entradas fuera de la diagonal
            contienen las covarianzas entre componentes. Eso significa que \(\Sigma\) no solo describe cuánta
            dispersión hay en cada dirección coordenada, sino también cómo se mueven conjuntamente las componentes
            del vector aleatorio.
        </p>
        """
    },
    {
        "id": "mat-4",
        "titulo": "Densidad de la normal multivariante e interpretación geométrica",
        "contenido": r"""
        <p>
            Cuando \(\Sigma\) es invertible, la densidad de la normal multivariante está dada por
        </p>
        <blockquote>
            \[
            p(x)=
            \frac{1}{(2\pi)^{d/2}|\Sigma|^{1/2}}
            \exp\left(
            -\frac{1}{2}(x-\mu)^T\Sigma^{-1}(x-\mu)
            \right).
            \]
        </blockquote>
        <p>
            Esta fórmula contiene dos piezas de gran importancia. La primera es el factor de normalización
        </p>
        <blockquote>
            \[
            \frac{1}{(2\pi)^{d/2}|\Sigma|^{1/2}},
            \]
        </blockquote>
        <p>
            que asegura que la densidad integre uno. La segunda es la forma cuadrática
        </p>
        <blockquote>
            \[
            (x-\mu)^T\Sigma^{-1}(x-\mu),
            \]
        </blockquote>
        <p>
            que mide qué tan lejos está el punto \(x\) del centro \(\mu\), pero no con la distancia euclídea usual,
            sino con una distancia adaptada a la geometría de la covarianza. Esa cantidad aparece también en la
            distancia de Mahalanobis:
        </p>
        <blockquote>
            \[
            d_M(x,\mu)^2=(x-\mu)^T\Sigma^{-1}(x-\mu).
            \]
        </blockquote>
        """
    },
    {
        "id": "mat-5",
        "titulo": "Curvas de nivel, identidad y elipsoides",
        "contenido": r"""
        <p>
            Desde el punto de vista geométrico, \(\mu\) es el centro de la nube gaussiana y \(\Sigma\) determina su forma.
            Si \(\Sigma\) fuera proporcional a la identidad, por ejemplo
        </p>
        <blockquote>
            \[
            \Sigma=\sigma^2 I,
            \]
        </blockquote>
        <p>
            entonces la distribución tendría la misma dispersión en todas las direcciones, y sus curvas de nivel serían
            esferas o circunferencias, según la dimensión. En cambio, si \(\Sigma\) tiene autovalores diferentes, la
            dispersión cambia según la dirección, y las curvas de nivel se vuelven elipsoides. Esto se ve claramente a través
            de la descomposición espectral
        </p>
        <blockquote>
            \[
            \Sigma=U\Lambda U^T,
            \]
        </blockquote>
        <p>
            donde \(U\) es una matriz ortogonal cuyas columnas son autovectores de \(\Sigma\), y \(\Lambda\) es una matriz
            diagonal con los autovalores \(\lambda_1,\dots,\lambda_d\). Los autovectores indican las direcciones principales
            de dispersión, y los autovalores indican cuánta varianza hay en cada una de esas direcciones. Si una dirección
            tiene un autovalor grande, la nube se estira en ese eje; si tiene un autovalor pequeño, la nube se contrae.
            Por eso la matriz de covarianza no es un accesorio técnico: es el objeto que impone la geometría de la distribución.
        </p>
        """
    },
    {
        "id": "mat-6",
        "titulo": "Proyecciones y varianza en una dirección",
        "contenido": r"""
        <p>
            Otra manera de ver esta geometría es mediante proyecciones. Si \(u \in R^d\) es un vector fijo, la variable
            proyectada \(u^T X\) tiene media
        </p>
        <blockquote>
            \[
            E[u^T X]=u^T \mu
            \]
        </blockquote>
        <p>
            y varianza
        </p>
        <blockquote>
            \[
            Var(u^T X)=u^T \Sigma u.
            \]
        </blockquote>
        <p>
            Esta fórmula muestra que la varianza en una dirección dada depende de la covarianza completa, no solo de las
            varianzas marginales. En otras palabras, \(\Sigma\) dice cuánto “ruido” o dispersión hay cuando observamos la
            distribución desde cualquier dirección del espacio.
        </p>
        """
    },
    {
        "id": "mat-7",
        "titulo": "Del caso finito al caso indexado por un parámetro",
        "contenido": r"""
        <p>
            Hasta aquí, todo sucede en dimensión finita. El vector aleatorio \(X\) tiene un número fijo de componentes.
            Puede ser dos, cinco, cien o mil, pero sigue siendo un objeto finito. El paso conceptual decisivo hacia los
            procesos gaussianos consiste en reemplazar ese vector finito por una familia de variables aleatorias indexadas
            por un parámetro. En lugar de escribir
        </p>
        <blockquote>
            \[
            X=
            \begin{pmatrix}
            X_1\\
            X_2\\
            \vdots\\
            X_d
            \end{pmatrix},
            \]
        </blockquote>
        <p>
            imaginamos una colección
        </p>
        <blockquote>
            \[
            \{f(x):x\in X\},
            \]
        </blockquote>
        <p>
            donde \(X\) es un conjunto de entradas. Ese conjunto puede ser \(R\), \(R^p\), el tiempo, el espacio geográfico
            o cualquier dominio en el que queramos evaluar una función. En vez de tener componentes numeradas por índices
            discretos \(1,\dots,d\), ahora tenemos valores de una cantidad aleatoria etiquetados por puntos \(x\) del dominio.
        </p>
        """
    },
    {
        "id": "mat-8",
        "titulo": "Una realización del proceso como función completa",
        "contenido": r"""
        <p>
            Esta notación ya sugiere una lectura funcional. Si para cada \(x\) tenemos una variable aleatoria \(f(x)\),
            entonces una realización del proceso no es simplemente un vector, sino una función completa \(x \mapsto f(x)\).
            Por eso un proceso gaussiano puede pensarse como una distribución de probabilidad sobre funciones. No elegimos
            una única curva, sino una ley probabilística que asigna mayor o menor plausibilidad a distintas curvas posibles.
        </p>
        """
    },
    {
        "id": "mat-9",
        "titulo": "Definición estructural de proceso gaussiano",
        "contenido": r"""
        <p>
            La pregunta natural es entonces la siguiente: ¿cómo generalizar la idea de normal multivariante a este nuevo contexto?
            La respuesta es elegante. Un proceso gaussiano se define exigiendo que, si tomamos cualquier cantidad finita de puntos
        </p>
        <blockquote>
            \[
            x_1,x_2,\dots,x_n \in X,
            \]
        </blockquote>
        <p>
            entonces el vector formado por las evaluaciones del proceso en esos puntos,
        </p>
        <blockquote>
            \[
            \begin{pmatrix}
            f(x_1)\\
            f(x_2)\\
            \vdots\\
            f(x_n)
            \end{pmatrix},
            \]
        </blockquote>
        <p>
            tenga distribución normal multivariante. Esta condición vale para todo \(n\) y para toda elección finita de puntos.
            Esa es la definición estructural del proceso gaussiano.
        </p>
        <p>
            Formalmente, se escribe
        </p>
        <blockquote>
            \[
            f \sim GP(m,k),
            \]
        </blockquote>
        <p>
            donde \(m\) es la función media y \(k\) es la función de covarianza, definidas por
        </p>
        <blockquote>
            \[
            m(x)=E[f(x)]
            \]
        </blockquote>
        <p>
            y
        </p>
        <blockquote>
            \[
            k(x,x')=Cov(f(x),f(x')).
            \]
        </blockquote>
        """
    },
    {
        "id": "mat-10",
        "titulo": "Analogía exacta entre MVN y GP",
        "contenido": r"""
        <p>
            La analogía con la normal multivariante es inmediata. En dimensión finita, una gaussiana está determinada por
            un vector \(\mu\) y una matriz \(\Sigma\). En el caso funcional, un proceso gaussiano está determinado por una
            función media \(m(x)\) y una función de covarianza \(k(x,x')\). Dicho de otro modo, el proceso gaussiano reemplaza
            el vector de medias por una media definida punto por punto, y la matriz de covarianza por un kernel que asigna
            covarianza a cada par de entradas.
        </p>
        <p>
            Si elegimos puntos \(x_1,\dots,x_n\), entonces el vector
        </p>
        <blockquote>
            \[
            \begin{pmatrix}
            f(x_1)\\
            f(x_2)\\
            \vdots\\
            f(x_n)
            \end{pmatrix}
            \]
        </blockquote>
        <p>
            tiene distribución
        </p>
        <blockquote>
            \[
            N(\mu,K),
            \]
        </blockquote>
        <p>
            donde
        </p>
        <blockquote>
            \[
            \mu=
            \begin{pmatrix}
            m(x_1)\\
            m(x_2)\\
            \vdots\\
            m(x_n)
            \end{pmatrix}
            \]
        </blockquote>
        <p>
            y
        </p>
        <blockquote>
            \[
            K=
            \begin{pmatrix}
            k(x_1,x_1) & k(x_1,x_2) & \cdots & k(x_1,x_n)\\
            k(x_2,x_1) & k(x_2,x_2) & \cdots & k(x_2,x_n)\\
            \vdots & \vdots & \ddots & \vdots\\
            k(x_n,x_1) & k(x_n,x_2) & \cdots & k(x_n,x_n)
            \end{pmatrix}.
            \]
        </blockquote>
        <p>
            Esta matriz \(K\) se llama a veces matriz de Gram o matriz de covarianza inducida por el kernel. Su papel es
            exactamente análogo al de \(\Sigma\) en la normal multivariante. La única diferencia es que ahora las filas y
            columnas no representan componentes abstractas \(X_1,\dots,X_d\), sino valores de la función en diferentes
            ubicaciones del dominio.
        </p>
        """
    },
    {
        "id": "mat-11",
        "titulo": "Una muestra de una MVN y una muestra de un GP",
        "contenido": r"""
        <p>
            Aquí aparece una intuición decisiva. En una normal multivariante, una muestra del modelo es un vector. En un
            proceso gaussiano, una muestra del modelo es una función. Si uno fija una colección de puntos \(x_1,\dots,x_n\),
            obtiene una muestra finita del vector
        </p>
        <blockquote>
            \[
            (f(x_1),\dots,f(x_n)),
            \]
        </blockquote>
        <p>
            pero si deja variar \(x\) en todo el dominio, lo que está describiendo es una curva o una superficie aleatoria.
            Por eso se dice, de manera intuitiva, que un proceso gaussiano es “una gaussiana sobre funciones”. No porque
            exista una única densidad elemental sobre un espacio infinito-dimensional comparable a la densidad gaussiana usual
            en \(R^d\), sino porque toda colección finita de evaluaciones tiene ley gaussiana multivariante.
        </p>
        """
    },
    {
        "id": "mat-12",
        "titulo": "La intuición central: una familia consistente de gaussianas finitas",
        "contenido": r"""
        <p>
            Conviene detenerse un momento en esta intuición, porque es la esencia conceptual del tema. Cuando decimos que
            un vector aleatorio es gaussiano, queremos decir que cualquier combinación lineal de sus componentes tiene
            distribución normal y, equivalentemente, que el vector completo tiene estructura de normal multivariante. Cuando
            decimos que una función aleatoria sigue un proceso gaussiano, queremos decir que si observamos esa función en
            cualquier número finito de puntos, los valores observados forman un vector normal multivariante. Así, el GP puede
            entenderse como una familia consistente de distribuciones gaussianas finito-dimensionales que “encajan” entre sí
            al cambiar el conjunto de puntos observados.
        </p>
        """
    },
    {
        "id": "mat-13",
        "titulo": "Consistencia marginal",
        "contenido": r"""
        <p>
            El requisito de consistencia no es un tecnicismo menor. Supongamos que tomamos tres puntos
            \(x_1,x_2,x_3\). El vector
        </p>
        <blockquote>
            \[
            (f(x_1),f(x_2),f(x_3))
            \]
        </blockquote>
        <p>
            debe seguir una normal trivariada. Si luego ignoramos el tercer punto y miramos solo
        </p>
        <blockquote>
            \[
            (f(x_1),f(x_2)),
            \]
        </blockquote>
        <p>
            esa distribución bivariada debe coincidir con la marginal obtenida a partir de la trivariada. Un proceso
            gaussiano no es una colección arbitraria de gaussianas para distintos tamaños de vectores, sino una familia
            coherente, construida a partir de una única función media y una única función de covarianza. Esa coherencia
            es la que permite pensar al proceso como un objeto global y no como una simple acumulación de modelos finitos
            sin relación entre sí.
        </p>
        """
    },
    {
        "id": "mat-14",
        "titulo": "Geometría del kernel en el espacio de funciones",
        "contenido": r"""
        <p>
            La interpretación geométrica también se conserva al pasar a procesos gaussianos, pero ahora en un sentido más sutil.
            En dimensión finita, la matriz \(\Sigma\) define elipsoides de densidad alrededor de \(\mu\). En un GP, el kernel
            \(k(x,x')\) define cómo se correlacionan los valores de la función en distintos puntos, y por lo tanto regula la
            forma típica de las funciones que el modelo considera plausibles. Si el kernel asigna covarianzas altas a puntos
            cercanos, entonces las funciones muestreadas tenderán a ser suaves. Si la covarianza cae rápidamente con la distancia
            entre \(x\) y \(x'\), las funciones podrán cambiar más bruscamente. Si el kernel incorpora periodicidad, las
            funciones plausibles tendrán un patrón oscilatorio. Por eso el kernel es mucho más que un artificio algebraico:
            es el objeto que codifica la geometría probabilística del espacio de funciones.
        </p>
        """
    },
    {
        "id": "mat-15",
        "titulo": "Ejemplo: kernel gaussiano o de base radial",
        "contenido": r"""
        <p>
            Un ejemplo muy utilizado es el kernel gaussiano o de base radial,
        </p>
        <blockquote>
            \[
            k(x,x')=\sigma_f^2 \exp\left(-\frac{\|x-x'\|^2}{2\ell^2}\right).
            \]
        </blockquote>
        <p>
            Aquí, \(\sigma_f^2\) controla la escala vertical de la variación funcional, mientras que \(\ell\) controla la
            longitud de escala. Si \(\ell\) es grande, dos puntos alejados siguen teniendo covarianza apreciable, de modo
            que la función cambia lentamente. Si \(\ell\) es pequeño, la covarianza entre puntos separados cae rápido, y la
            función puede oscilar más. Esta sola fórmula muestra de manera concreta cómo se pasa del lenguaje geométrico de
            la covarianza en dimensión finita al lenguaje geométrico de la dependencia funcional.
        </p>
        """
    },
    {
        "id": "mat-16",
        "titulo": "Ejemplo discreto para ver la transición",
        "contenido": r"""
        <p>
            También es útil ver un ejemplo discreto para percibir la transición entre ambos mundos. Supongamos que elegimos
            tres entradas \(x_1=0\), \(x_2=1\) y \(x_3=2\), y consideramos un proceso gaussiano con media nula,
        </p>
        <blockquote>
            \[
            m(x)=0,
            \]
        </blockquote>
        <p>
            y con algún kernel \(k\). Entonces el vector
        </p>
        <blockquote>
            \[
            \begin{pmatrix}
            f(0)\\
            f(1)\\
            f(2)
            \end{pmatrix}
            \]
        </blockquote>
        <p>
            sigue una normal multivariante con media
        </p>
        <blockquote>
            \[
            \begin{pmatrix}
            0\\
            0\\
            0
            \end{pmatrix}
            \]
        </blockquote>
        <p>
            y matriz de covarianza
        </p>
        <blockquote>
            \[
            \begin{pmatrix}
            k(0,0) & k(0,1) & k(0,2)\\
            k(1,0) & k(1,1) & k(1,2)\\
            k(2,0) & k(2,1) & k(2,2)
            \end{pmatrix}.
            \]
        </blockquote>
        <p>
            Si agregamos otro punto, por ejemplo \(x_4=3\), obtenemos una normal de dimensión cuatro. Si agregamos diez puntos,
            obtenemos una normal de dimensión diez. El proceso gaussiano no cambia de naturaleza: simplemente produce gaussianas
            multivariantes de cualquier dimensión, todas enlazadas por la misma función media y el mismo kernel. En ese sentido,
            el GP puede verse como una “máquina” que, dado cualquier conjunto finito de entradas, devuelve la normal multivariante
            apropiada sobre los valores funcionales correspondientes.
        </p>
        """
    },
    {
        "id": "mat-17",
        "titulo": "Creencias previas sobre funciones",
        "contenido": r"""
        <p>
            La intuición de “gaussiana sobre funciones” se puede afinar aún más. En una distribución normal multivariante,
            uno no conoce de antemano qué vector concreto se observará; solo conoce la ley que genera vectores más o menos
            probables. De manera análoga, en un proceso gaussiano uno no conoce de antemano qué función concreta será la
            verdadera; solo conoce una ley que vuelve unas funciones más plausibles que otras. Antes de observar datos, el GP
            representa un conjunto de creencias previas sobre la forma posible de la función. Después, al incorporar
            observaciones, esa distribución se actualiza y se concentra alrededor de las funciones que explican bien los datos.
            Este punto es crucial para la regresión con GP, pero incluso antes del condicionamiento ya muestra por qué la
            construcción es tan poderosa: no obliga a fijar una familia paramétrica rígida, sino que trabaja directamente en
            el nivel funcional.
        </p>
        """
    },
    {
        "id": "mat-18",
        "titulo": "Perspectiva teórica general",
        "contenido": r"""
        <p>
            Desde una perspectiva teórica, el paso de la normal multivariante al proceso gaussiano es entonces una ampliación
            del mismo esquema conceptual. En la MVN, el objeto central es un vector aleatorio finito caracterizado por media
            y covarianza. En el GP, el objeto central es una familia indexada de variables aleatorias, o equivalentemente una
            función aleatoria, también caracterizada por media y covarianza, pero ahora en forma funcional. La noción de
            dependencia se conserva, aunque trasladada desde pares de componentes \((X_i,X_j)\) a pares de ubicaciones
            \((x,x')\). La geometría también se conserva, aunque trasladada desde el espacio euclídeo finito al espacio de
            funciones. Por eso no debe pensarse al proceso gaussiano como una teoría completamente nueva y separada, sino
            como la continuación natural del estudio de la covarianza y de la normal multivariante en un contexto infinitamente
            más rico.
        </p>
        """
    },
    {
        "id": "mat-19",
        "titulo": "Síntesis final",
        "contenido": r"""
        <p>
            En síntesis, una normal multivariante modela incertidumbre sobre un vector finito; un proceso gaussiano modela
            incertidumbre sobre una función completa. La media \(\mu\) se transforma en una función media \(m(x)\), y la
            matriz de covarianza \(\Sigma\) se transforma en una función de covarianza \(k(x,x')\). La distribución gaussiana
            sobre \(R^d\) se convierte en una familia coherente de gaussianas multivariantes sobre cualquier colección finita
            de puntos del dominio. Esa es la razón precisa por la cual resulta correcto, desde la intuición matemática,
            describir a un proceso gaussiano como “una gaussiana sobre funciones”.
        </p>
        <div class="mini-summary">
            <strong>Resumen muy compacto.</strong> La transición de la normal multivariante al proceso gaussiano consiste
            en pasar de un modelo probabilístico sobre vectores finitos,
            \[
            X \sim N(\mu,\Sigma),
            \]
            a un modelo probabilístico sobre funciones,
            \[
            f \sim GP(m,k),
            \]
            de modo que para todo conjunto finito de puntos \(x_1,\dots,x_n\), el vector de evaluaciones
            \[
            (f(x_1),\dots,f(x_n))
            \]
            sea normal multivariante con media inducida por \(m\) y covarianza inducida por \(k\). La media sigue
            describiendo el centro esperado, y la covarianza sigue describiendo la dependencia y la geometría, pero
            ahora en el nivel funcional. Por eso un proceso gaussiano puede entenderse rigurosamente como una extensión
            infinita-dimensional de la normal multivariante.
        </div>
        """
    },
]

# =========================================================
# 2. GENERACIÓN HTML DINÁMICO
# =========================================================

def generar_bloques_material(secciones):
    bloques = []
    for sec in secciones:
        bloques.append(f"""
        <div class="topic-card" id="{html.escape(sec['id'])}">
            <div class="topic-header">
                <span class="topic-title">{html.escape(sec['titulo'])}</span>
                <span class="expand-icon">⌄</span>
            </div>
            <div class="topic-content">
                {sec['contenido']}
            </div>
        </div>
        """)
    return "\n".join(bloques)

material_html = generar_bloques_material(secciones_material)

# =========================================================
# 3. PLANTILLA FINAL
# =========================================================

html_final = f"""
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{html.escape(titulo_principal)}</title>

    <script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-MML-AM_CHTML" async></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">

    <style>
        :root {{
            --bg-primary: linear-gradient(135deg, #f0f9ff 0%, #e0f2fe 100%);
            --bg-secondary: rgba(255, 255, 255, 0.85);
            --bg-tertiary: rgba(248, 250, 252, 0.8);
            --text-primary: #0c4a6e;
            --text-secondary: #075985;
            --accent-primary: #0ea5e9;
            --accent-secondary: #38bdf8;
            --accent-gradient: linear-gradient(135deg, var(--accent-primary) 0%, var(--accent-secondary) 100%);
            --border-color: rgba(226, 232, 240, 0.8);
            --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.08);
            --border-radius: 20px;
            --transition: all 0.4s cubic-bezier(0.25, 0.8, 0.25, 1);
            --progress-bg: rgba(255,255,255,0.45);
        }}

        [data-theme="dark"] {{
            --bg-primary: linear-gradient(135deg, #0c4a6e 0%, #1e3a8a 100%);
            --bg-secondary: rgba(30, 58, 138, 0.85);
            --bg-tertiary: rgba(30, 64, 175, 0.7);
            --text-primary: #f0f9ff;
            --text-secondary: #dbeafe;
            --accent-primary: #38bdf8;
            --accent-secondary: #7dd3fc;
            --border-color: rgba(255, 255, 255, 0.15);
            --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.20);
            --progress-bg: rgba(255,255,255,0.15);
        }}

        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }}

        html {{
            scroll-behavior: smooth;
        }}

        body {{
            font-family: 'Inter', sans-serif;
            line-height: 1.8;
            background: var(--bg-primary);
            color: var(--text-primary);
            transition: var(--transition);
            min-height: 100vh;
            position: relative;
            overflow-x: hidden;
            padding-bottom: 3rem;
        }}

        .particles {{
            position: fixed;
            inset: 0;
            pointer-events: none;
            z-index: -1;
        }}

        .particle {{
            position: absolute;
            border-radius: 50%;
            animation: float 25s infinite linear;
            opacity: 0;
            background: rgba(255, 255, 255, 0.55);
            box-shadow: 0 0 15px rgba(255,255,255,0.25);
        }}

        @keyframes float {{
            0% {{
                transform: translateY(100vh) rotate(0deg);
                opacity: 0;
            }}
            10%, 90% {{
                opacity: 0.55;
            }}
            100% {{
                transform: translateY(-10vh) rotate(360deg);
                opacity: 0;
            }}
        }}

        .progress-wrapper {{
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 6px;
            background: var(--progress-bg);
            z-index: 1200;
            backdrop-filter: blur(10px);
        }}

        .progress-bar {{
            height: 100%;
            width: 0%;
            background: var(--accent-gradient);
            transition: width 0.15s ease;
        }}

        .theme-toggle {{
            position: fixed;
            top: 1.2rem;
            right: 1.2rem;
            width: 60px;
            height: 60px;
            border: 1px solid var(--border-color);
            border-radius: 50%;
            background: var(--bg-secondary);
            backdrop-filter: blur(15px);
            box-shadow: var(--shadow-card);
            cursor: pointer;
            display: flex;
            align-items: center;
            justify-content: center;
            font-size: 1.35rem;
            color: var(--text-primary);
            transition: var(--transition);
            z-index: 1100;
        }}

        .theme-toggle:hover {{
            transform: scale(1.12) rotate(180deg);
        }}

        .container {{
            max-width: 1100px;
            margin: 0 auto;
            padding: 2rem;
            position: relative;
            z-index: 1;
        }}

        .hero,
        .summary-card,
        .topic-card {{
            background: var(--bg-secondary);
            backdrop-filter: blur(20px);
            border-radius: var(--border-radius);
            box-shadow: var(--shadow-card);
            border: 2px solid var(--border-color);
        }}

        .hero {{
            padding: 2rem;
            margin-top: 1rem;
            margin-bottom: 1.5rem;
            text-align: center;
        }}

        .hero-badge {{
            display: inline-flex;
            align-items: center;
            gap: 0.5rem;
            background: var(--bg-tertiary);
            border: 1px solid var(--border-color);
            border-radius: 999px;
            padding: 0.65rem 1rem;
            margin-bottom: 1rem;
            color: var(--text-secondary);
            font-size: 0.95rem;
            font-weight: 600;
        }}

        .main-title {{
            font-size: clamp(2.2rem, 5vw, 3.6rem);
            font-weight: 800;
            background: var(--accent-gradient);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            background-clip: text;
            margin-bottom: 1rem;
            line-height: 1.12;
        }}

        .subtitle {{
            color: var(--text-secondary);
            font-size: 1.03rem;
            max-width: 920px;
            margin: 0 auto;
        }}

        .summary-card {{
            padding: 1.6rem 1.8rem;
            margin-bottom: 1.6rem;
        }}

        .summary-card h2 {{
            color: var(--text-primary);
            font-size: 1.45rem;
            margin-bottom: 0.9rem;
        }}

        .summary-card p {{
            color: var(--text-secondary);
            margin-bottom: 1rem;
        }}

        .lesson-container {{
            display: flex;
            flex-direction: column;
            gap: 1.2rem;
        }}

        .topic-card {{
            overflow: hidden;
            transition: var(--transition);
        }}

        .topic-card:hover {{
            transform: translateY(-2px);
        }}

        .topic-header {{
            cursor: pointer;
            padding: 1.35rem 1.7rem;
            display: flex;
            justify-content: space-between;
            align-items: center;
            gap: 1rem;
            user-select: none;
        }}

        .topic-title {{
            font-size: 1.15rem;
            font-weight: 700;
            color: var(--text-primary);
            line-height: 1.4;
        }}

        .expand-icon {{
            font-size: 1.45rem;
            line-height: 1;
            color: var(--text-secondary);
            transition: var(--transition);
            flex-shrink: 0;
        }}

        .topic-card.open .expand-icon {{
            transform: rotate(180deg);
        }}

        .topic-content {{
            max-height: 0;
            overflow: hidden;
            transition: max-height 0.85s ease, padding 0.85s ease;
            background: var(--bg-tertiary);
            padding: 0 1.7rem;
        }}

        .topic-card.open .topic-content {{
            padding: 1.4rem 1.7rem 1.7rem 1.7rem;
            border-top: 1px solid var(--border-color);
        }}

        .topic-content p,
        .topic-content li {{
            color: var(--text-secondary);
            margin-bottom: 1rem;
        }}

        .topic-content blockquote {{
            text-align: center;
            border-left: 4px solid var(--accent-primary);
            margin: 1.3rem 0;
            color: var(--text-secondary);
            background: rgba(127, 140, 141, 0.05);
            border-radius: 0 10px 10px 0;
            padding: 1rem 1.3rem;
            overflow-x: auto;
        }}

        .mini-summary {{
            margin-top: 1.2rem;
            padding: 1rem 1.1rem;
            border-radius: 14px;
            background: rgba(14, 165, 233, 0.08);
            border: 1px solid rgba(14, 165, 233, 0.18);
            color: var(--text-secondary);
        }}

        .mini-summary strong {{
            color: var(--text-primary);
        }}

        .footer {{
            text-align: center;
            margin-top: 2.8rem;
            padding-top: 1.6rem;
            border-top: 1px solid var(--border-color);
            color: var(--text-secondary);
            font-size: 0.95rem;
        }}

        @media (max-width: 768px) {{
            .container {{
                padding: 1rem;
            }}

            .hero,
            .summary-card {{
                padding: 1.2rem;
            }}

            .topic-header {{
                padding: 1rem 1.2rem;
            }}

            .topic-content {{
                padding: 0 1.2rem;
            }}

            .topic-card.open .topic-content {{
                padding: 1rem 1.2rem 1.2rem 1.2rem;
            }}

            .theme-toggle {{
                width: 54px;
                height: 54px;
                top: 1rem;
                right: 1rem;
            }}
        }}
    </style>
</head>

<body data-theme="dark">
    <div class="progress-wrapper">
        <div class="progress-bar" id="progressBar"></div>
    </div>

    <div class="particles" id="particles-container"></div>

    <div class="theme-toggle" id="themeToggleButton" title="Cambiar tema">
        <span id="themeIcon">☾</span>
    </div>

    <div class="container">
        <section class="hero">
            <div class="hero-badge">Material de estudio · Procesos Gaussianos</div>
            <h1 class="main-title">{html.escape(titulo_principal)}</h1>
            <p class="subtitle">
                Desarrollo completo del punto 1 con estética visual tipo glassmorphism, cambio de tema,
                subtítulos deslizantes, barra de progreso y fórmulas renderizadas con MathJax, listo para Google Colab.
            </p>
        </section>

        <section class="summary-card">
            <h2>Resumen inicial</h2>
            {resumen_inicial}
        </section>

        <section class="lesson-container">
            {material_html}
        </section>

        <div class="footer">
            Material elaborado por el profesor Sergio Gevatschnaider.
        </div>
    </div>

    <script>
        (function() {{
            const bodyEl = document.body;
            const themeToggleButton = document.getElementById('themeToggleButton');
            const themeIcon = document.getElementById('themeIcon');
            const progressBar = document.getElementById('progressBar');
            const cards = Array.from(document.querySelectorAll('.topic-card'));

            function setTheme(theme) {{
                bodyEl.setAttribute('data-theme', theme);
                try {{
                    localStorage.setItem('gp_material_theme_p1', theme);
                }} catch (e) {{}}
                if (themeIcon) {{
                    themeIcon.textContent = theme === 'dark' ? '☀' : '☾';
                }}
            }}

            let preferredTheme = 'dark';
            try {{
                preferredTheme = localStorage.getItem('gp_material_theme_p1') || 'dark';
            }} catch (e) {{}}
            setTheme(preferredTheme);

            if (themeToggleButton) {{
                themeToggleButton.addEventListener('click', () => {{
                    const current = bodyEl.getAttribute('data-theme') || 'dark';
                    setTheme(current === 'dark' ? 'light' : 'dark');
                }});
            }}

            function typesetMath(target) {{
                if (window.MathJax && window.MathJax.Hub) {{
                    try {{
                        MathJax.Hub.Queue(["Typeset", MathJax.Hub, target || document.body]);
                    }} catch (e) {{}}
                }}
            }}

            function openCard(card) {{
                const content = card.querySelector('.topic-content');
                if (!content) return;
                card.classList.add('open');
                content.style.maxHeight = content.scrollHeight + 100 + 'px';
                typesetMath(content);
                setTimeout(() => {{
                    content.style.maxHeight = content.scrollHeight + 100 + 'px';
                }}, 250);
            }}

            function closeCard(card) {{
                const content = card.querySelector('.topic-content');
                if (!content) return;
                content.style.maxHeight = content.scrollHeight + 'px';
                requestAnimationFrame(() => {{
                    card.classList.remove('open');
                    content.style.maxHeight = '0px';
                }});
            }}

            function toggleCard(card) {{
                if (card.classList.contains('open')) {{
                    closeCard(card);
                }} else {{
                    openCard(card);
                }}
            }}

            function refreshOpenCards() {{
                cards.forEach(card => {{
                    if (card.classList.contains('open')) {{
                        const content = card.querySelector('.topic-content');
                        if (content) {{
                            content.style.maxHeight = content.scrollHeight + 100 + 'px';
                        }}
                    }}
                }});
            }}

            cards.forEach(card => {{
                const header = card.querySelector('.topic-header');
                if (header) {{
                    header.addEventListener('click', () => toggleCard(card));
                }}
            }});

            if (cards.length > 0) {{
                openCard(cards[0]);
            }}

            const container = document.getElementById('particles-container');
            if (container) {{
                for (let i = 0; i < 36; i++) {{
                    const p = document.createElement('div');
                    p.className = 'particle';
                    p.style.left = Math.random() * 100 + 'vw';
                    const size = (Math.random() * 7 + 3);
                    p.style.width = size + 'px';
                    p.style.height = size + 'px';
                    p.style.animationDelay = Math.random() * -20 + 's';
                    p.style.animationDuration = (15 + Math.random() * 12) + 's';
                    container.appendChild(p);
                }}
            }}

            function updateProgress() {{
                const scrollTop = document.documentElement.scrollTop || document.body.scrollTop;
                const scrollHeight = document.documentElement.scrollHeight - document.documentElement.clientHeight;
                const progress = scrollHeight > 0 ? (scrollTop / scrollHeight) * 100 : 0;
                progressBar.style.width = progress + '%';
            }}

            window.addEventListener('scroll', updateProgress);
            window.addEventListener('resize', refreshOpenCards);

            setTimeout(() => {{
                refreshOpenCards();
                updateProgress();
                typesetMath(document.body);
            }}, 800);

            setTimeout(() => {{
                refreshOpenCards();
                updateProgress();
            }}, 1600);
        }})();
    </script>
</body>
</html>
"""

display(HTML(html_final))

In [ ]:
# @title Índice Interactivo: 2. Definición formal de Proceso Gaussiano
from IPython.display import display, HTML
import html

# =========================================================
# 1. INTRODUCCIÓN
# =========================================================

intro_html_content = r"""
<div class="content-block" style="margin-top:0;">
    <h2>Resumen de la Sección</h2>
    <p>
        La noción de proceso gaussiano surge al extender el lenguaje de la normal multivariante desde vectores
        aleatorios finitos hacia familias enteras de variables aleatorias indexadas por un parámetro.
    </p>
    <p>
        En esta sección formalizaremos esta idea. Pasaremos de la intuición de una "distribución sobre funciones"
        a la definición matemática rigurosa basada en el espacio de probabilidad, las distribuciones finito-dimensionales,
        y las propiedades ineludibles que deben cumplir la función media y la función de covarianza para definir
        un proceso válido.
    </p>
</div>
"""

# =========================================================
# 2. SECCIONES (Texto 100% completo y LaTeX corregido)
# =========================================================

secciones_principales_data = [
    {
        "id": "sec-1",
        "titulo": "1. De vectores a funciones: El Proceso Estocástico",
        "contenido": r"""
            <p>
                En lugar de estudiar un único vector $X \in \mathbb{R}^d$, se considera una colección de variables aleatorias
            </p>
            <blockquote>$$\{f(x): x \in \mathcal{X}\},$$</blockquote>
            <p>
                donde $\mathcal{X}$ es un conjunto índice, también llamado espacio de entrada o dominio. Ese conjunto puede ser,
                por ejemplo, la recta real $\mathbb{R}$, un intervalo de tiempo, un espacio geográfico, o en general un subconjunto
                de $\mathbb{R}^p$. La interpretación intuitiva es que, para cada valor $x$ del dominio, la cantidad $f(x)$ es una
                variable aleatoria. Una realización concreta del proceso corresponde entonces a una función completa $x \mapsto f(x)$.
                Por eso un proceso estocástico puede entenderse como una distribución de probabilidad sobre funciones.
            </p>
            <div style="text-align: center; margin: 2rem 0;">

            </div>
            <p>
                Desde el punto de vista formal, un proceso estocástico es una familia de variables aleatorias definida sobre un
                mismo espacio de probabilidad. Más precisamente, si $(\Omega, \mathcal{F}, \mathbb{P})$ es un espacio de probabilidad,
                un proceso estocástico indexado por $\mathcal{X}$ es una aplicación
            </p>
            <blockquote>$$f: \mathcal{X} \times \Omega \to \mathbb{R}$$</blockquote>
            <p>
                tal que, para cada $x \in \mathcal{X}$, la función $\omega \mapsto f(x,\omega)$ es una variable aleatoria real.
                Fijado $\omega \in \Omega$, la aplicación $x \mapsto f(x,\omega)$ es una trayectoria o realización del proceso.
                En este lenguaje, el proceso gaussiano es una clase muy especial de proceso estocástico, caracterizada porque
                toda colección finita de sus evaluaciones tiene distribución normal multivariante.
            </p>
        """
    },
    {
        "id": "sec-2",
        "titulo": "2. La definición matemática rigurosa",
        "contenido": r"""
            <p>
                La definición formal es la siguiente. Se dice que $f$ es un proceso gaussiano con función media $m$ y función
                de covarianza $k$, y se escribe
            </p>
            <blockquote>$$f \sim \mathcal{GP}(m,k),$$</blockquote>
            <p>
                si para cualquier entero $n \ge 1$ y para cualquier elección finita de puntos $x_1,\dots,x_n \in \mathcal{X}$,
                el vector aleatorio
            </p>
            <blockquote>
                $$\begin{pmatrix} f(x_1) \\ f(x_2) \\ \vdots \\ f(x_n) \end{pmatrix}$$
            </blockquote>
            <p>
                tiene distribución normal multivariante con vector de medias
            </p>
            <blockquote>
                $$\begin{pmatrix} m(x_1) \\ m(x_2) \\ \vdots \\ m(x_n) \end{pmatrix}$$
            </blockquote>
            <p>
                y matriz de covarianza
            </p>
            <blockquote>
                $$\begin{pmatrix} k(x_1,x_1) & k(x_1,x_2) & \cdots & k(x_1,x_n) \\ k(x_2,x_1) & k(x_2,x_2) & \cdots & k(x_2,x_n) \\ \vdots & \vdots & \ddots & \vdots \\ k(x_n,x_1) & k(x_n,x_2) & \cdots & k(x_n,x_n) \end{pmatrix}.$$
            </blockquote>
            <p>
                En otras palabras, si definimos el vector $f_X = (f(x_1), \dots, f(x_n))^T$, entonces la definición se resume como
            </p>
            <blockquote>$$f_X \sim \mathcal{N}(m_X, K_X),$$</blockquote>
            <p>
                donde $m_X$ es el vector de medias evaluado en $X$ y $(K_X)_{ij} = k(x_i,x_j)$. Esta fórmula es el corazón
                de la teoría. Muestra que un proceso gaussiano no es otra cosa que una familia compatible de distribuciones
                normales multivariantes para todas las posibles elecciones finitas de puntos del dominio.
            </p>
        """
    },
    {
        "id": "sec-3",
        "titulo": "3. La Función Media y la Función de Covarianza",
        "contenido": r"""
            <p>
                La función media $m$ está definida por
            </p>
            <blockquote>$$m(x) = \mathbb{E}[f(x)].$$</blockquote>
            <p>
                Esto significa que, para cada punto $x$, la media del valor aleatorio del proceso en ese punto es $m(x)$.
                Si uno imaginara repetir muchas veces el experimento que genera el proceso y observara el valor en el mismo
                punto $x$, el promedio de esas observaciones tendería a $m(x)$. La función media describe, por tanto, la
                tendencia central del proceso en cada punto del dominio.
            </p>
            <p>
                La función de covarianza $k$ está definida por
            </p>
            <blockquote>$$k(x,x') = \operatorname{Cov}(f(x),f(x')) = \mathbb{E}\left[(f(x)-m(x))(f(x')-m(x'))\right].$$</blockquote>
            <p>
                Esta cantidad mide cómo fluctúan conjuntamente los valores del proceso en dos puntos del dominio. Si $k(x,x')$
                es grande y positivo, entonces los valores $f(x)$ y $f(x')$ tienden a moverse en la misma dirección respecto
                de sus medias. Si $k(x,x')$ es cercano a cero, existe poca relación lineal entre ambos. En particular, la
                varianza puntual del proceso en $x$ es
            </p>
            <blockquote>$$\operatorname{Var}(f(x)) = k(x,x).$$</blockquote>
            <p>
                De aquí se desprende que la diagonal de la matriz de covarianza contiene las varianzas de las evaluaciones puntuales,
                y los términos fuera de la diagonal contienen las covarianzas cruzadas entre distintos puntos.
            </p>
        """
    },
    {
        "id": "sec-4",
        "titulo": "4. Propiedades de una covarianza válida",
        "contenido": r"""
            <p>
                Una propiedad esencial de los procesos gaussianos es que, al igual que en la normal multivariante, la media y
                la covarianza determinan completamente la ley del proceso, en el sentido de sus distribuciones finito-dimensionales.
                Esto no ocurre para un proceso estocástico arbitrario. Lo que hace especial al caso gaussiano es precisamente que
                la información de primer y segundo orden basta para describir por completo todas las distribuciones conjuntas finitas.
            </p>
            <p>
                Conviene explicar con cuidado qué significa que $k$ sea una covarianza válida. No cualquier función
                $k: \mathcal{X} \times \mathcal{X} \to \mathbb{R}$ puede actuar como función de covarianza. Debe cumplir dos
                propiedades fundamentales. La primera es la <strong>simetría</strong>:
            </p>
            <blockquote>$$k(x,x') = k(x',x).$$</blockquote>
            <p>
                Esta propiedad es natural, porque la covarianza entre $f(x)$ y $f(x')$ debe coincidir con la covarianza entre $f(x')$ y $f(x)$.
                La segunda propiedad es la <strong>semidefinición positiva</strong>. Esto significa que, para cualquier elección finita
                de puntos $x_1,\dots,x_n$ y cualquier vector $a=(a_1,\dots,a_n)^T \in \mathbb{R}^n$, se debe cumplir
            </p>
            <blockquote>$$\sum_{i=1}^n \sum_{j=1}^n a_i a_j k(x_i,x_j) \ge 0.$$</blockquote>
            <p>
                Equivalente y matricialmente, esto quiere decir que la matriz $K_X = \left(k(x_i,x_j)\right)_{i,j=1}^n$ debe ser
                semidefinida positiva para toda elección finita de puntos. La razón de esta condición se entiende si se considera la
                combinación lineal $Z = \sum_{i=1}^n a_i f(x_i)$. La varianza de $Z$ es
            </p>
            <blockquote>$$\operatorname{Var}(Z) = \operatorname{Var}\left(\sum_{i=1}^n a_i f(x_i)\right) = \sum_{i=1}^n \sum_{j=1}^n a_i a_j \operatorname{Cov}(f(x_i),f(x_j)) = \sum_{i=1}^n \sum_{j=1}^n a_i a_j k(x_i,x_j),$$</blockquote>
            <p>
                y como toda varianza debe ser no negativa, la desigualdad anterior es obligatoria. Esta observación muestra que la
                condición de semidefinición positiva no es un detalle técnico: expresa simplemente el hecho elemental de que las varianzas
                no pueden ser negativas.
            </p>
            <p>
                Existe además un resultado fundamental de existencia, basado en el <em>teorema de extensión de Kolmogorov</em>, que afirma,
                en esencia, que si se elige una función media $m$ y una función $k$ que sea simétrica y semidefinida positiva, entonces
                existe un proceso gaussiano $f \sim \mathcal{GP}(m,k)$ cuyas distribuciones finito-dimensionales tienen precisamente esa
                media y esa covarianza.
            </p>
        """
    },
    {
        "id": "sec-5",
        "titulo": "5. Equivalencias y centrado del proceso",
        "contenido": r"""
            <p>
                Otra forma equivalente de definir un proceso gaussiano es decir que, para cualquier elección finita de puntos
                $x_1,\dots,x_n \in \mathcal{X}$ y cualquier conjunto de coeficientes reales $a_1,\dots,a_n$, la combinación lineal
            </p>
            <blockquote>$$\sum_{i=1}^n a_i f(x_i)$$</blockquote>
            <p>
                tiene distribución normal univariada. Esta caracterización es enteramente paralela a una propiedad clásica de la
                normal multivariante: un vector es gaussiano si y solo si toda combinación lineal de sus componentes es normal.
            </p>
            <p>
                Vale la pena detenerse en la relación entre la función media y la función de covarianza. La función media $m(x)$
                controla el nivel esperado del proceso. Sin embargo, la función media por sí sola no dice nada sobre la dispersión ni
                sobre la regularidad del proceso. Toda esa información está contenida en la covarianza.
            </p>
            <p>
                La función de covarianza $k(x,x')$, en cambio, cumple varios papeles simultáneos. En primer lugar, determina la varianza
                puntual. En segundo lugar, determina la dependencia lineal entre dos puntos. En tercer lugar, codifica propiedades
                geométricas globales del proceso, tales como suavidad, oscilación, periodicidad y escala de variación. La media fija
                la tendencia central; la covarianza organiza la estructura de fluctuación alrededor de esa tendencia.
            </p>
            <p>
                Esta relación se ve muy claramente en la descomposición
            </p>
            <blockquote>$$f(x) = m(x) + g(x),$$</blockquote>
            <p>
                donde $g$ es un proceso gaussiano centrado, es decir, $\mathbb{E}[g(x)]=0$, con la misma covarianza $k$. En efecto,
                si definimos $g(x) = f(x) - m(x)$, entonces $\mathbb{E}[g(x)] = 0$, y $\operatorname{Cov}(g(x),g(x')) = k(x,x')$.
                Esto muestra que toda la complejidad de segundo orden del proceso queda preservada al centrarlo. En muchas aplicaciones
                prácticas, se trabaja con media nula, $m(x) \equiv 0$, concentrando toda la información sustantiva del modelo en $k$.
            </p>
        """
    },
    {
        "id": "sec-6",
        "titulo": "6. Ejemplo concreto: Kernel RBF en 1D",
        "contenido": r"""
            <p>
                Para entender mejor la definición, conviene considerar un ejemplo concreto. Sea $\mathcal{X} = \mathbb{R}$, y definamos
                la función media nula $m(x)=0$ y la función de covarianza
            </p>
            <blockquote>$$k(x,x') = \sigma^2 \exp\left(-\frac{(x-x')^2}{2\ell^2}\right),$$</blockquote>
            <p>
                donde $\sigma^2>0$ y $\ell>0$ son parámetros. Esta función se conoce como kernel gaussiano o de base radial. Si
                $f \sim \mathcal{GP}(m,k)$, entonces para cualquier par de puntos $x_1, x_2$, el vector
            </p>
            <blockquote>
                $$\begin{pmatrix} f(x_1) \\ f(x_2) \end{pmatrix}$$
            </blockquote>
            <p>
                sigue una normal bivariada con media nula y matriz de covarianza
            </p>
            <blockquote>
                $$\begin{pmatrix} \sigma^2 & \sigma^2\exp\left(-\frac{(x_1-x_2)^2}{2\ell^2}\right) \\ \sigma^2\exp\left(-\frac{(x_1-x_2)^2}{2\ell^2}\right) & \sigma^2 \end{pmatrix}.$$
            </blockquote>
            <p>
                Aquí puede verse de manera precisa el papel de la media y la covarianza. La varianza $\sigma^2$ determina la escala
                típica de las fluctuaciones. La covarianza depende de la distancia $|x_1-x_2|$: si los puntos están muy cerca, la
                covarianza es casi $\sigma^2$; si están muy lejos, cae hacia cero. Esto traduce matemáticamente la idea de suavidad.
            </p>
            <p>
                Podemos incluso calcular el coeficiente de correlación entre $f(x)$ y $f(x')$. Dado que
            </p>
            <blockquote>$$\operatorname{Corr}(f(x),f(x')) = \frac{k(x,x')}{\sqrt{k(x,x)k(x',x')}},$$</blockquote>
            <p>
                en este ejemplo se obtiene
            </p>
            <blockquote>$$\operatorname{Corr}(f(x),f(x')) = \exp\left(-\frac{(x-x')^2}{2\ell^2}\right).$$</blockquote>
            <p>
                La correlación depende únicamente de la distancia entre los puntos. Esta expresión muestra que el parámetro $\ell$
                controla cuán rápido se pierde la dependencia al separarse los puntos del dominio.
            </p>
        """
    },
    {
        "id": "sec-7",
        "titulo": "7. Conclusión: La esencia del Proceso Gaussiano",
        "contenido": r"""
            <p>
                La definición formal de proceso gaussiano también permite entender qué ocurre con distribuciones marginales y condicionales.
                Como toda colección finita de evaluaciones tiene distribución normal multivariante, cualquier subconjunto de esas evaluaciones
                tiene distribución normal marginal, y la distribución condicional de unas evaluaciones dadas otras también es normal. Esta
                estabilidad algebraica es una de las razones por las cuales los procesos gaussianos resultan tan poderosos en regresión y predicción.
            </p>
            <p>
                Hay una observación adicional que ayuda a cerrar la definición desde un punto de vista conceptual. Cuando se dice que un proceso
                gaussiano es una "gaussiana sobre funciones", esta frase debe entenderse con precisión. No significa simplemente que para cada $x$,
                la variable $f(x)$ sea normal. Eso sería una condición mucho más débil. La verdadera esencia del GP es que <strong>para cualquier
                conjunto finito de puntos, el vector conjunto de evaluaciones sea multivariadamente gaussiano</strong>. La información crucial no está
                solo en las marginales puntuales, sino en la forma en que estas se enlazan a través de la covarianza.
            </p>
            <p>
                En consecuencia, la definición formal de proceso gaussiano puede resumirse así. Un proceso gaussiano sobre un dominio $\mathcal{X}$
                es un proceso estocástico $\{f(x):x\in\mathcal{X}\}$ tal que toda colección finita de variables $(f(x_1),\dots,f(x_n))$ tiene
                distribución normal multivariante. Este proceso queda especificado por una función media
            </p>
            <blockquote>$$m(x) = \mathbb{E}[f(x)]$$</blockquote>
            <p>
                y una función de covarianza
            </p>
            <blockquote>$$k(x,x') = \operatorname{Cov}(f(x),f(x')),$$</blockquote>
            <p>
                y la existencia del proceso está garantizada siempre que $k$ sea simétrica y semidefinida positiva. La media describe la tendencia
                esperada del proceso, mientras que la covarianza describe la dispersión puntual, la dependencia entre puntos y la geometría probabilística
                de las trayectorias. En el caso gaussiano, estas dos funciones no son meros resúmenes parciales: constituyen la descripción completa de
                todas las distribuciones finito-dimensionales del proceso.
            </p>
        """
    }
]

# =========================================================
# 3. GENERACIÓN HTML
# =========================================================

def generar_tarjetas_acordeon(datos):
    bloques = []
    for seccion in datos:
        bloques.append(f"""
        <div class="topic-card" id="{html.escape(seccion['id'])}">
            <div class="topic-header">
                <span class="topic-title">{seccion['titulo']}</span>
                <i class="fas fa-chevron-down expand-icon"></i>
            </div>
            <div class="topic-content">
                {seccion['contenido']}
            </div>
        </div>
        """)
    return "\n".join(bloques)

def generar_navegacion(datos):
    chips = []
    for seccion in datos:
        chips.append(
            f'<button class="nav-chip" type="button" data-target="{html.escape(seccion["id"])}">{seccion["titulo"]}</button>'
        )
    return "\n".join(chips)

contenido_dinamico_html = generar_tarjetas_acordeon(secciones_principales_data)
navegacion_html = generar_navegacion(secciones_principales_data)

# =========================================================
# 4. PLANTILLA HTML
# =========================================================

plantilla_profesional = r"""
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{{MAIN_TITLE}}</title>

  <script type="text/x-mathjax-config">
    MathJax.Hub.Config({
      tex2jax: {
        inlineMath: [['$','$']],
        displayMath: [['$$','$$']],
        processEscapes: true
      }
    });
  </script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-MML-AM_CHTML" async></script>

  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
  <link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css" rel="stylesheet">

  <style>
    :root {
      --bg-primary: linear-gradient(135deg, #f0f9ff 0%, #e0f2fe 100%);
      --bg-secondary: rgba(255, 255, 255, 0.85);
      --bg-tertiary: rgba(248, 250, 252, 0.8);
      --text-primary: #0c4a6e;
      --text-secondary: #075985;
      --accent-primary: #0ea5e9;
      --accent-secondary: #38bdf8;
      --accent-gradient: linear-gradient(135deg, var(--accent-primary) 0%, var(--accent-secondary) 100%);
      --border-color: rgba(226, 232, 240, 0.8);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.08);
      --border-radius: 20px;
      --transition: all 0.4s cubic-bezier(0.25, 0.8, 0.25, 1);
      --nav-chip-bg: rgba(255,255,255,0.8);
      --progress-bg: rgba(255,255,255,0.45);
    }

    [data-theme="dark"] {
      --bg-primary: linear-gradient(135deg, #0c4a6e 0%, #1e3a8a 100%);
      --bg-secondary: rgba(30, 58, 138, 0.85);
      --bg-tertiary: rgba(30, 64, 175, 0.7);
      --text-primary: #f0f9ff;
      --text-secondary: #dbeafe;
      --accent-primary: #38bdf8;
      --accent-secondary: #7dd3fc;
      --border-color: rgba(255, 255, 255, 0.15);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.20);
      --nav-chip-bg: rgba(30, 58, 138, 0.65);
      --progress-bg: rgba(255,255,255,0.15);
    }

    * { margin: 0; padding: 0; box-sizing: border-box; }
    html { scroll-behavior: smooth; }
    body { font-family: 'Inter', sans-serif; line-height: 1.8; background: var(--bg-primary); color: var(--text-primary); transition: var(--transition); min-height: 100vh; position: relative; overflow-x: hidden; }

    .particles { position: fixed; top: 0; left: 0; width: 100%; height: 100%; pointer-events: none; z-index: -1; }
    .particle { position: absolute; border-radius: 50%; animation: float 25s infinite linear; opacity: 0; background: rgba(255, 255, 255, 0.55); box-shadow: 0 0 15px rgba(255,255,255,0.25); }
    @keyframes float { 0% { transform: translateY(100vh) rotate(0deg); opacity: 0; } 10%, 90% { opacity: 0.55; } 100% { transform: translateY(-10vh) rotate(360deg); opacity: 0; } }

    .progress-wrapper { position: fixed; top: 0; left: 0; width: 100%; height: 6px; background: var(--progress-bg); z-index: 1200; backdrop-filter: blur(10px); }
    .progress-bar { height: 100%; width: 0%; background: var(--accent-gradient); transition: width 0.15s ease; }

    .theme-toggle { position: fixed; top: 1.2rem; right: 1.2rem; width: 60px; height: 60px; border: 1px solid var(--border-color); border-radius: 50%; background: var(--bg-secondary); backdrop-filter: blur(15px); box-shadow: var(--shadow-card); cursor: pointer; display: flex; align-items: center; justify-content: center; font-size: 1.35rem; color: var(--text-primary); transition: var(--transition); z-index: 1100; }
    .theme-toggle:hover { transform: scale(1.15) rotate(180deg); }

    .container { max-width: 1100px; margin: 0 auto; padding: 2rem; z-index: 1; position: relative; }
    .header { text-align: center; margin-bottom: 2rem; position: relative; padding-top: 1.2rem; }
    .header-badge { display: inline-flex; align-items: center; gap: 0.6rem; background: var(--bg-secondary); border: 1px solid var(--border-color); border-radius: 999px; padding: 0.7rem 1rem; backdrop-filter: blur(15px); box-shadow: var(--shadow-card); margin-bottom: 1rem; color: var(--text-secondary); font-size: 0.95rem; }

    .main-title { font-size: clamp(2.2rem, 5vw, 3.8rem); font-weight: 800; background: var(--accent-gradient); -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text; margin-bottom: 1rem; line-height: 1.15; }
    .subtitle { color: var(--text-secondary); max-width: 900px; margin: 0 auto; font-size: 1.03rem; }

    .content-block, .quick-nav, .control-bar, .topic-card { background: var(--bg-secondary); backdrop-filter: blur(20px); border-radius: var(--border-radius); box-shadow: var(--shadow-card); border: 2px solid var(--border-color); }
    .content-block { padding: 2rem; margin-top: 2rem; }
    .content-block h2 { color: var(--text-primary); margin-bottom: 1rem; font-size: 1.55rem; }
    .content-block p, .content-block li, .subtitle { color: var(--text-secondary); }

    .quick-nav { margin-top: 2rem; padding: 1.4rem; }
    .quick-nav h3 { margin-bottom: 1rem; color: var(--text-primary); font-size: 1.2rem; }
    .nav-grid { display: flex; flex-wrap: wrap; gap: 0.75rem; }
    .nav-chip { padding: 0.65rem 0.95rem; border-radius: 999px; background: var(--nav-chip-bg); color: var(--text-primary); border: 1px solid var(--border-color); font-size: 0.92rem; transition: var(--transition); box-shadow: 0 8px 18px rgba(0,0,0,0.06); cursor: pointer; font-family: inherit; text-align: left; }
    .nav-chip:hover { transform: translateY(-2px); background: var(--bg-tertiary); }

    .control-bar { margin-top: 1.5rem; padding: 1rem 1.2rem; display: flex; gap: 0.8rem; flex-wrap: wrap; justify-content: center; }
    .control-btn { border: 1px solid var(--border-color); background: var(--nav-chip-bg); color: var(--text-primary); border-radius: 999px; padding: 0.75rem 1rem; cursor: pointer; font-family: inherit; font-size: 0.95rem; transition: var(--transition); }
    .control-btn:hover { transform: translateY(-2px); background: var(--bg-tertiary); }

    .lesson-container { display: flex; flex-direction: column; gap: 1.3rem; margin-top: 1.6rem; }
    .topic-card { overflow: hidden; transition: var(--transition); scroll-margin-top: 90px; }
    .topic-card:hover { transform: translateY(-2px); }
    .topic-header { cursor: pointer; padding: 1.5rem 2rem; display: flex; justify-content: space-between; align-items: center; gap: 1rem; user-select: none; }
    .topic-title { font-size: 1.22rem; font-weight: 700; color: var(--text-primary); line-height: 1.4; }
    .expand-icon { font-size: 1.15rem; color: var(--text-secondary); transition: var(--transition); flex-shrink: 0; }
    .topic-card.open .expand-icon { transform: rotate(180deg); }

    .topic-content { max-height: 0; overflow: hidden; transition: max-height 1.2s ease, padding 1.2s ease; background: var(--bg-tertiary); }
    .topic-card.open .topic-content { max-height: 6500px; padding: 1.6rem 2rem 1.8rem 2rem; border-top: 1px solid var(--border-color); }
    .topic-content h4 { color: var(--text-primary); margin-top: 1.5rem; margin-bottom: 0.55rem; font-size: 1.08rem; border-left: 4px solid var(--accent-primary); padding-left: 12px; }
    .topic-content h4:first-child { margin-top: 0; }
    .topic-content p, .topic-content li { color: var(--text-secondary); line-height: 1.8; margin-bottom: 0.95rem; }
    .topic-content strong { color: var(--text-primary); font-weight: 700; }

    .topic-content blockquote { text-align: center; border-left: 4px solid var(--accent-primary); margin: 1.4rem 0; font-style: normal; color: var(--text-secondary); background: rgba(127, 140, 141, 0.05); border-radius: 0 10px 10px 0; padding: 1rem 1.5rem; overflow-x: auto; }

    footer { text-align: center; margin-top: 4rem; padding-top: 2rem; border-top: 1px solid var(--border-color); }
    footer p { color: var(--text-secondary); font-size: 0.95rem; opacity: 0.9; }

    @media (max-width: 768px) {
      .container { padding: 1rem; }
      .topic-header { padding: 1.1rem 1.3rem; }
      .topic-card.open .topic-content { padding: 1.1rem 1.3rem 1.3rem 1.3rem; }
      .theme-toggle { width: 54px; height: 54px; top: 1rem; right: 1rem; }
    }
  </style>
</head>

<body data-theme="dark">
  <div class="progress-wrapper">
    <div class="progress-bar" id="progressBar"></div>
  </div>

  <div class="particles" id="particles-container"></div>

  <div class="theme-toggle" id="themeToggleButton" title="Cambiar tema">
    <i class="fas fa-moon" id="theme-icon"></i>
  </div>

  <div class="container">
    <header class="header">
      <div class="header-badge">
        <i class="fas fa-project-diagram"></i>
        <span>Estadística Aplicada a Data Science · Repositorio</span>
      </div>
      <h1 class="main-title">{{MAIN_TITLE}}</h1>
      <p class="subtitle">
        El fundamento matemático riguroso: desde el espacio de probabilidad hasta las propiedades de la función media y la matriz de covarianza.
      </p>
    </header>

    {{INTRO_HTML}}

    <section class="quick-nav">
      <h3>Navegación rápida</h3>
      <div class="nav-grid">
        {{NAV_HTML}}
      </div>
    </section>

    <section class="control-bar">
      <button class="control-btn" id="expandAllBtn" type="button"><i class="fas fa-plus-circle"></i> Expandir todo</button>
      <button class="control-btn" id="collapseAllBtn" type="button"><i class="fas fa-minus-circle"></i> Contraer todo</button>
      <button class="control-btn" id="topBtn" type="button"><i class="fas fa-arrow-up"></i> Volver arriba</button>
    </section>

    <div class="lesson-container">
      {{DYNAMIC_HTML}}
    </div>

    <footer>
      <p>{{FOOTER_TEXT}}</p>
    </footer>
  </div>

  <script>
    (function() {
      const themeToggleButton = document.getElementById('themeToggleButton');
      const themeIcon = document.getElementById('theme-icon');
      const bodyEl = document.body;
      const expandAllBtn = document.getElementById('expandAllBtn');
      const collapseAllBtn = document.getElementById('collapseAllBtn');
      const topBtn = document.getElementById('topBtn');
      const progressBar = document.getElementById('progressBar');

      function setTheme(theme) {
        bodyEl.setAttribute('data-theme', theme);
        try { localStorage.setItem('gp_theme', theme); } catch (e) {}
        if (themeIcon) { themeIcon.className = theme === 'dark' ? 'fas fa-sun' : 'fas fa-moon'; }
      }

      if (themeToggleButton) {
        themeToggleButton.addEventListener('click', () => {
          const newTheme = (bodyEl.getAttribute('data-theme') || 'dark') === 'dark' ? 'light' : 'dark';
          setTheme(newTheme);
        });
      }

      let preferredTheme = 'dark';
      try { preferredTheme = localStorage.getItem('gp_theme') || 'dark'; } catch (e) {}
      setTheme(preferredTheme);

      const cards = Array.from(document.querySelectorAll('.topic-card'));

      cards.forEach(card => {
        const header = card.querySelector('.topic-header');
        if (header) {
          header.addEventListener('click', () => {
            card.classList.toggle('open');
          });
        }
      });

      if (expandAllBtn) { expandAllBtn.addEventListener('click', () => { cards.forEach(card => card.classList.add('open')); }); }
      if (collapseAllBtn) { collapseAllBtn.addEventListener('click', () => { cards.forEach(card => card.classList.remove('open')); }); }
      if (topBtn) { topBtn.addEventListener('click', () => { window.scrollTo({ top: 0, behavior: 'smooth' }); }); }

      const navButtons = Array.from(document.querySelectorAll('.nav-chip'));
      navButtons.forEach(btn => {
        btn.addEventListener('click', () => {
          const targetId = btn.getAttribute('data-target');
          const target = document.getElementById(targetId);
          if (target) {
            target.classList.add('open');
            setTimeout(() => { target.scrollIntoView({ behavior: 'smooth', block: 'start' }); }, 120);
          }
        });
      });

      const firstTopic = document.querySelector('.topic-card');
      if (firstTopic) { firstTopic.classList.add('open'); }

      const container = document.getElementById('particles-container');
      if (container) {
        for (let i = 0; i < 34; i++) {
          const p = document.createElement('div');
          p.className = 'particle';
          p.style.left = Math.random() * 100 + 'vw';
          const size = (Math.random() * 6 + 3);
          p.style.width = size + 'px';
          p.style.height = size + 'px';
          p.style.animationDelay = Math.random() * -20 + 's';
          p.style.animationDuration = (15 + Math.random() * 12) + 's';
          container.appendChild(p);
        }
      }

      function updateProgress() {
        const scrollTop = document.documentElement.scrollTop || document.body.scrollTop;
        const scrollHeight = document.documentElement.scrollHeight - document.documentElement.clientHeight;
        const progress = scrollHeight > 0 ? (scrollTop / scrollHeight) * 100 : 0;
        progressBar.style.width = progress + '%';
      }

      window.addEventListener('scroll', updateProgress);
      updateProgress();

      if (window.MathJax) {
          MathJax.Hub.Queue(["Typeset", MathJax.Hub]);
      }
    })();
  </script>
</body>
</html>
"""

# =========================================================
# 5. ENSAMBLADO
# =========================================================

final_html = (
    plantilla_profesional
    .replace("{{MAIN_TITLE}}", "2. Definición formal de Proceso Gaussiano")
    .replace("{{INTRO_HTML}}", intro_html_content)
    .replace("{{NAV_HTML}}", navegacion_html)
    .replace("{{DYNAMIC_HTML}}", contenido_dinamico_html)
    .replace("{{FOOTER_TEXT}}", "Material elaborado por el profesor Sergio Gevatschnaider.")
)

display(HTML(final_html))

In [ ]:
# @title Índice Interactivo: 3. Distribuciones finito-dimensionales
from IPython.display import display, HTML
import html

# =========================================================
# 1. INTRODUCCIÓN
# =========================================================

intro_html_content = r"""
<div class="content-block" style="margin-top:0;">
    <h2>Resumen de la Sección</h2>
    <p>
        La noción de <strong>distribución finito-dimensional</strong> es el puente exacto entre la teoría de la normal
        multivariante y la teoría de los Procesos Gaussianos. Cuando se trabaja con un vector aleatorio finito, toda la
        información probabilística está contenida en la ley conjunta de sus componentes. En cambio, un proceso gaussiano
        no es un vector de dimensión fija, sino una familia de variables aleatorias indexadas por un conjunto continuo.
    </p>
    <p>
        En esta sección exploraremos cómo, al "mirar" el proceso a través de un número finito de evaluaciones,
        obtenemos una distribución multivariada ordinaria. Estudiaremos la matriz de Gram, las propiedades de simetría
        y consistencia, y comprenderemos por qué el Proceso Gaussiano es, en esencia, una generalización de
        dimensión infinita de la distribución normal multivariante clásica.
    </p>
</div>
"""

# =========================================================
# 2. SECCIONES (Limpias y con LaTeX corregido)
# =========================================================

secciones_principales_data = [
    {
        "id": "sec-1",
        "titulo": "1. El puente hacia lo finito",
        "contenido": r"""
            <h4>De variables aisladas a la ley conjunta</h4>
            <p>
                Para describir un Proceso Gaussiano rigurosamente, no basta con mirar cada variable $f(x)$ por separado;
                hay que estudiar la ley conjunta de cualquier colección finita de evaluaciones del proceso.
            </p>
            <p>
                Sea entonces un proceso gaussiano $f \sim \mathcal{GP}(m,k)$, definido sobre un dominio $\mathcal{X}$.
                Si se eligen arbitrariamente $n$ puntos del dominio,
            </p>
            <blockquote>$$x_1,\dots,x_n \in \mathcal{X},$$</blockquote>
            <p>
                entonces el vector de evaluaciones del proceso en esos puntos es
            </p>
            <blockquote>$$\begin{pmatrix} f(x_1) \\ f(x_2) \\ \vdots \\ f(x_n) \end{pmatrix}.$$</blockquote>
            <p>
                La propiedad fundamental que define al proceso gaussiano es que este vector tiene distribución normal multivariante. Es decir,
            </p>
            <blockquote>$$\begin{pmatrix} f(x_1) \\ f(x_2) \\ \vdots \\ f(x_n) \end{pmatrix} \sim \mathcal{N}(\mu,K),$$</blockquote>
            <p>
                donde el vector de medias $\mu$ y la matriz de covarianza $K$ vienen dados por la función media $m$ y la función de covarianza $k$.
            </p>
        """
    },
    {
        "id": "sec-2",
        "titulo": "2. La Matriz de Gram y la densidad conjunta",
        "contenido": r"""
            <h4>Construcción explícita de $\mu$ y $K$</h4>
            <p>En forma explícita, el vector de medias es:</p>
            <blockquote>$$\mu = \begin{pmatrix} m(x_1) \\ m(x_2) \\ \vdots \\ m(x_n) \end{pmatrix},$$</blockquote>
            <p>mientras que la matriz $K$ es:</p>
            <blockquote>$$K = \begin{pmatrix} k(x_1,x_1) & k(x_1,x_2) & \cdots & k(x_1,x_n) \\ k(x_2,x_1) & k(x_2,x_2) & \cdots & k(x_2,x_n) \\ \vdots & \vdots & \ddots & \vdots \\ k(x_n,x_1) & k(x_n,x_2) & \cdots & k(x_n,x_n) \end{pmatrix}.$$</blockquote>
            <p>
                A esta matriz se la llama con frecuencia <strong>matriz de Gram</strong> o matriz de covarianza inducida por el kernel.
                Su entrada $(i,j)$ está dada por:
            </p>
            <blockquote>$$K_{ij} = k(x_i,x_j).$$</blockquote>
            <p>
                Esta fórmula merece ser interpretada con cuidado. Cada entrada de la matriz representa la covarianza entre los valores
                del proceso en dos puntos concretos del dominio:
            </p>
            <blockquote>$$K_{ij} = \operatorname{Cov}(f(x_i), f(x_j)).$$</blockquote>
            <p>En particular, las entradas diagonales son:</p>
            <blockquote>$$K_{ii} = k(x_i,x_i) = \operatorname{Var}(f(x_i)),$$</blockquote>
            <p>
                de modo que la diagonal de $K$ contiene las varianzas puntuales del proceso en los puntos elegidos. Las entradas fuera
                de la diagonal miden el grado de dependencia lineal.
            </p>
            <h4>Densidad explícita</h4>
            <p>
                Si la matriz $K$ es invertible, la densidad conjunta de las evaluaciones puede escribirse de manera explícita como:
            </p>
            <blockquote>$$p(z) = \frac{1}{(2\pi)^{n/2}|K|^{1/2}} \exp\left( -\frac{1}{2}(z-\mu)^T K^{-1}(z-\mu) \right),$$</blockquote>
            <p>donde el vector $z$ representa un posible valor del vector aleatorio de evaluaciones.</p>
        """
    },
    {
        "id": "sec-3",
        "titulo": "3. Ejemplo elemental con Kernel Radial",
        "contenido": r"""
            <h4>Aplicación práctica sobre $\mathbb{R}$</h4>
            <p>
                Conviene ver un ejemplo elemental para fijar estas ideas. Supongamos que el dominio es $\mathcal{X}=\mathbb{R}$,
                que la función media es nula, $m(x)=0$, y que el kernel es el kernel gaussiano o radial:
            </p>
            <blockquote>$$k(x,x') = \sigma^2 \exp\left(-\frac{(x-x')^2}{2\ell^2}\right),$$</blockquote>
            <p>
                donde $\sigma^2>0$ y $\ell>0$. Si elegimos tres puntos $x_1=0$, $x_2=1$, $x_3=2$, entonces el vector tiene distribución
                normal trivariada con media nula y matriz de Gram:
            </p>
            <blockquote>$$K = \begin{pmatrix} \sigma^2 & \sigma^2 \exp\left(-\frac{1}{2\ell^2}\right) & \sigma^2 \exp\left(-\frac{4}{2\ell^2}\right) \\ \sigma^2 \exp\left(-\frac{1}{2\ell^2}\right) & \sigma^2 & \sigma^2 \exp\left(-\frac{1}{2\ell^2}\right) \\ \sigma^2 \exp\left(-\frac{4}{2\ell^2}\right) & \sigma^2 \exp\left(-\frac{1}{2\ell^2}\right) & \sigma^2 \end{pmatrix}.$$</blockquote>
            <p>
                Este ejemplo deja ver una propiedad importante. La covarianza entre $f(0)$ y $f(1)$ es más grande que la covarianza
                entre $f(0)$ y $f(2)$, porque los puntos $0$ y $1$ están más cerca que $0$ y $2$. La regularidad geométrica de las
                trayectorias queda ya codificada dentro de la matriz de Gram.
            </p>
        """
    },
    {
        "id": "sec-4",
        "titulo": "4. Propiedades algebraicas del Kernel",
        "contenido": r"""
            <h4>Simetría y Semidefinición Positiva</h4>
            <p>
                La matriz de Gram no es una construcción meramente notacional. Tiene una propiedad algebraica indispensable:
                debe ser simétrica y semidefinida positiva. La simetría proviene de que
            </p>
            <blockquote>$$k(x_i,x_j) = k(x_j,x_i),$$</blockquote>
            <p>
                mientras que la semidefinición positiva significa que, para cualquier vector $a=(a_1,\dots,a_n)^T \in \mathbb{R}^n$,
            </p>
            <blockquote>$$a^T K a \ge 0.$$</blockquote>
            <p>Si se escribe esta expresión desarrollada, se obtiene:</p>
            <blockquote>$$a^T K a = \sum_{i=1}^n \sum_{j=1}^n a_i a_j k(x_i,x_j).$$</blockquote>
            <p>
                La razón probabilística de esta propiedad es simple y profunda. Si definimos la combinación lineal $Z = \sum_{i=1}^n a_i f(x_i)$,
                entonces su varianza es:
            </p>
            <blockquote>$$\operatorname{Var}(Z) = \operatorname{Var}\left(\sum_{i=1}^n a_i f(x_i)\right) = \sum_{i=1}^n \sum_{j=1}^n a_i a_j \operatorname{Cov}(f(x_i),f(x_j)) = a^T K a.$$</blockquote>
            <p>
                Como toda varianza es no negativa, debe cumplirse necesariamente que $a^T K a \ge 0$. Este argumento muestra que la
                positividad del kernel no es un requisito artificial, sino una consecuencia inevitable de que $K$ represente una covarianza genuina.
            </p>
        """
    },
    {
        "id": "sec-5",
        "titulo": "5. Consistencia y generalización infinita",
        "contenido": r"""
            <h4>Estabilidad bajo marginalización</h4>
            <p>
                Las distribuciones finito-dimensionales cumplen además una propiedad de consistencia. Si se eligen $n$ puntos y se considera
                el vector conjunto, al ignorar algunas coordenadas y quedarse solo con $f(x_1),\dots,f(x_r)$ donde $r<n$, la distribución
                de ese subvector debe coincidir con la marginal. Si:
            </p>
            <blockquote>$$\begin{pmatrix} f(x_1) \\ \vdots \\ f(x_n) \end{pmatrix} \sim \mathcal{N}(\mu,K),$$</blockquote>
            <p>
                entonces cualquier subvector sigue siendo normal multivariante. Esta estabilidad hace posible que la familia de distribuciones
                se ensamble en un único objeto global.
            </p>
            <h4>Una extensión de la MVN</h4>
            <p>
                En este punto se vuelve transparente por qué un GP es una generalización infinita de la normal multivariante. Una MVN clásica
                tiene dimensión fija $n$:
            </p>
            <blockquote>$$X = \begin{pmatrix} X_1 \\ \vdots \\ X_n \end{pmatrix} \sim \mathcal{N}(\mu,\Sigma).$$</blockquote>
            <p>
                Un proceso gaussiano no fija una dimensión. Si tomas dos puntos, obtienes una normal bivariada; si tomas mil puntos, una de dimensión mil.
                Contiene simultáneamente todas estas distribuciones. El índice deja de ser una lista finita $1,\dots,n$ y pasa a ser un punto $x$
                de un dominio $\mathcal{X}$.
            </p>
            <h4>Geometría de la nube gaussiana</h4>
            <p>
                En términos geométricos, cada distribución finito-dimensional viene acompañada por un elipsoide de densidad en $\mathbb{R}^n$.
                Si los puntos están muy correlacionados, la nube gaussiana será alargada; si están casi desacoplados, será más esférica.
            </p>
            <p>
                En resumen, para cada conjunto finito de puntos, el proceso produce un vector aleatorio gaussiano con media $\mu_i = m(x_i)$ y
                covarianza $K_{ij} = k(x_i,x_j)$. La matriz de Gram reúne todas las dependencias, consolidando al GP como un objeto matemático robusto.
            </p>
        """
    }
]

# =========================================================
# 3. GENERACIÓN HTML
# =========================================================

def generar_tarjetas_acordeon(datos):
    bloques = []
    for seccion in datos:
        bloques.append(f"""
        <div class="topic-card" id="{html.escape(seccion['id'])}">
            <div class="topic-header">
                <span class="topic-title">{seccion['titulo']}</span>
                <i class="fas fa-chevron-down expand-icon"></i>
            </div>
            <div class="topic-content">
                {seccion['contenido']}
            </div>
        </div>
        """)
    return "\n".join(bloques)

def generar_navegacion(datos):
    chips = []
    for seccion in datos:
        chips.append(
            f'<button class="nav-chip" type="button" data-target="{html.escape(seccion["id"])}">{seccion["titulo"]}</button>'
        )
    return "\n".join(chips)

contenido_dinamico_html = generar_tarjetas_acordeon(secciones_principales_data)
navegacion_html = generar_navegacion(secciones_principales_data)

# =========================================================
# 4. PLANTILLA HTML (Con inyección de MathJax)
# =========================================================

plantilla_profesional = r"""
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{{MAIN_TITLE}}</title>

  <script type="text/x-mathjax-config">
    MathJax.Hub.Config({
      tex2jax: {
        inlineMath: [['$','$']],
        displayMath: [['$$','$$']],
        processEscapes: true
      }
    });
  </script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-MML-AM_CHTML" async></script>

  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
  <link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css" rel="stylesheet">

  <style>
    :root {
      --bg-primary: linear-gradient(135deg, #f0f9ff 0%, #e0f2fe 100%);
      --bg-secondary: rgba(255, 255, 255, 0.85);
      --bg-tertiary: rgba(248, 250, 252, 0.8);
      --text-primary: #0c4a6e;
      --text-secondary: #075985;
      --accent-primary: #0ea5e9;
      --accent-secondary: #38bdf8;
      --accent-gradient: linear-gradient(135deg, var(--accent-primary) 0%, var(--accent-secondary) 100%);
      --border-color: rgba(226, 232, 240, 0.8);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.08);
      --border-radius: 20px;
      --transition: all 0.4s cubic-bezier(0.25, 0.8, 0.25, 1);
      --nav-chip-bg: rgba(255,255,255,0.8);
      --progress-bg: rgba(255,255,255,0.45);
    }

    [data-theme="dark"] {
      --bg-primary: linear-gradient(135deg, #0c4a6e 0%, #1e3a8a 100%);
      --bg-secondary: rgba(30, 58, 138, 0.85);
      --bg-tertiary: rgba(30, 64, 175, 0.7);
      --text-primary: #f0f9ff;
      --text-secondary: #dbeafe;
      --accent-primary: #38bdf8;
      --accent-secondary: #7dd3fc;
      --border-color: rgba(255, 255, 255, 0.15);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.20);
      --nav-chip-bg: rgba(30, 58, 138, 0.65);
      --progress-bg: rgba(255,255,255,0.15);
    }

    * { margin: 0; padding: 0; box-sizing: border-box; }
    html { scroll-behavior: smooth; }
    body { font-family: 'Inter', sans-serif; line-height: 1.8; background: var(--bg-primary); color: var(--text-primary); transition: var(--transition); min-height: 100vh; position: relative; overflow-x: hidden; }

    .particles { position: fixed; top: 0; left: 0; width: 100%; height: 100%; pointer-events: none; z-index: -1; }
    .particle { position: absolute; border-radius: 50%; animation: float 25s infinite linear; opacity: 0; background: rgba(255, 255, 255, 0.55); box-shadow: 0 0 15px rgba(255,255,255,0.25); }
    @keyframes float { 0% { transform: translateY(100vh) rotate(0deg); opacity: 0; } 10%, 90% { opacity: 0.55; } 100% { transform: translateY(-10vh) rotate(360deg); opacity: 0; } }

    .progress-wrapper { position: fixed; top: 0; left: 0; width: 100%; height: 6px; background: var(--progress-bg); z-index: 1200; backdrop-filter: blur(10px); }
    .progress-bar { height: 100%; width: 0%; background: var(--accent-gradient); transition: width 0.15s ease; }

    .theme-toggle { position: fixed; top: 1.2rem; right: 1.2rem; width: 60px; height: 60px; border: 1px solid var(--border-color); border-radius: 50%; background: var(--bg-secondary); backdrop-filter: blur(15px); box-shadow: var(--shadow-card); cursor: pointer; display: flex; align-items: center; justify-content: center; font-size: 1.35rem; color: var(--text-primary); transition: var(--transition); z-index: 1100; }
    .theme-toggle:hover { transform: scale(1.15) rotate(180deg); }

    .container { max-width: 1100px; margin: 0 auto; padding: 2rem; z-index: 1; position: relative; }
    .header { text-align: center; margin-bottom: 2rem; position: relative; padding-top: 1.2rem; }
    .header-badge { display: inline-flex; align-items: center; gap: 0.6rem; background: var(--bg-secondary); border: 1px solid var(--border-color); border-radius: 999px; padding: 0.7rem 1rem; backdrop-filter: blur(15px); box-shadow: var(--shadow-card); margin-bottom: 1rem; color: var(--text-secondary); font-size: 0.95rem; }

    .main-title { font-size: clamp(2.2rem, 5vw, 3.8rem); font-weight: 800; background: var(--accent-gradient); -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text; margin-bottom: 1rem; line-height: 1.15; }
    .subtitle { color: var(--text-secondary); max-width: 900px; margin: 0 auto; font-size: 1.03rem; }

    .content-block, .quick-nav, .control-bar, .topic-card { background: var(--bg-secondary); backdrop-filter: blur(20px); border-radius: var(--border-radius); box-shadow: var(--shadow-card); border: 2px solid var(--border-color); }
    .content-block { padding: 2rem; margin-top: 2rem; }
    .content-block h2 { color: var(--text-primary); margin-bottom: 1rem; font-size: 1.55rem; }
    .content-block p, .content-block li, .subtitle { color: var(--text-secondary); }

    .quick-nav { margin-top: 2rem; padding: 1.4rem; }
    .quick-nav h3 { margin-bottom: 1rem; color: var(--text-primary); font-size: 1.2rem; }
    .nav-grid { display: flex; flex-wrap: wrap; gap: 0.75rem; }
    .nav-chip { padding: 0.65rem 0.95rem; border-radius: 999px; background: var(--nav-chip-bg); color: var(--text-primary); border: 1px solid var(--border-color); font-size: 0.92rem; transition: var(--transition); box-shadow: 0 8px 18px rgba(0,0,0,0.06); cursor: pointer; font-family: inherit; text-align: left; }
    .nav-chip:hover { transform: translateY(-2px); background: var(--bg-tertiary); }

    .control-bar { margin-top: 1.5rem; padding: 1rem 1.2rem; display: flex; gap: 0.8rem; flex-wrap: wrap; justify-content: center; }
    .control-btn { border: 1px solid var(--border-color); background: var(--nav-chip-bg); color: var(--text-primary); border-radius: 999px; padding: 0.75rem 1rem; cursor: pointer; font-family: inherit; font-size: 0.95rem; transition: var(--transition); }
    .control-btn:hover { transform: translateY(-2px); background: var(--bg-tertiary); }

    .lesson-container { display: flex; flex-direction: column; gap: 1.3rem; margin-top: 1.6rem; }
    .topic-card { overflow: hidden; transition: var(--transition); scroll-margin-top: 90px; }
    .topic-card:hover { transform: translateY(-2px); }
    .topic-header { cursor: pointer; padding: 1.5rem 2rem; display: flex; justify-content: space-between; align-items: center; gap: 1rem; user-select: none; }
    .topic-title { font-size: 1.22rem; font-weight: 700; color: var(--text-primary); line-height: 1.4; }
    .expand-icon { font-size: 1.15rem; color: var(--text-secondary); transition: var(--transition); flex-shrink: 0; }
    .topic-card.open .expand-icon { transform: rotate(180deg); }

    .topic-content { max-height: 0; overflow: hidden; transition: max-height 1.2s ease, padding 1.2s ease; background: var(--bg-tertiary); }
    .topic-card.open .topic-content { max-height: 6500px; padding: 1.6rem 2rem 1.8rem 2rem; border-top: 1px solid var(--border-color); }
    .topic-content h4 { color: var(--text-primary); margin-top: 1.5rem; margin-bottom: 0.55rem; font-size: 1.08rem; border-left: 4px solid var(--accent-primary); padding-left: 12px; }
    .topic-content h4:first-child { margin-top: 0; }
    .topic-content p, .topic-content li { color: var(--text-secondary); line-height: 1.8; margin-bottom: 0.95rem; }
    .topic-content strong { color: var(--text-primary); font-weight: 700; }

    .topic-content blockquote { text-align: center; border-left: 4px solid var(--accent-primary); margin: 1.4rem 0; font-style: normal; color: var(--text-secondary); background: rgba(127, 140, 141, 0.05); border-radius: 0 10px 10px 0; padding: 1rem 1.5rem; overflow-x: auto; }

    footer { text-align: center; margin-top: 4rem; padding-top: 2rem; border-top: 1px solid var(--border-color); }
    footer p { color: var(--text-secondary); font-size: 0.95rem; opacity: 0.9; }

    @media (max-width: 768px) {
      .container { padding: 1rem; }
      .topic-header { padding: 1.1rem 1.3rem; }
      .topic-card.open .topic-content { padding: 1.1rem 1.3rem 1.3rem 1.3rem; }
      .theme-toggle { width: 54px; height: 54px; top: 1rem; right: 1rem; }
    }
  </style>
</head>

<body data-theme="dark">
  <div class="progress-wrapper">
    <div class="progress-bar" id="progressBar"></div>
  </div>

  <div class="particles" id="particles-container"></div>

  <div class="theme-toggle" id="themeToggleButton" title="Cambiar tema">
    <i class="fas fa-moon" id="theme-icon"></i>
  </div>

  <div class="container">
    <header class="header">
      <div class="header-badge">
        <i class="fas fa-project-diagram"></i>
        <span>Aplicaciones en Data Science · Módulo Matemático</span>
      </div>
      <h1 class="main-title">{{MAIN_TITLE}}</h1>
      <p class="subtitle">
        El puente matemático desde la teoría de la normal multivariante hacia el espacio infinito-dimensional de los Procesos Gaussianos.
      </p>
    </header>

    {{INTRO_HTML}}

    <section class="quick-nav">
      <h3>Navegación rápida</h3>
      <div class="nav-grid">
        {{NAV_HTML}}
      </div>
    </section>

    <section class="control-bar">
      <button class="control-btn" id="expandAllBtn" type="button"><i class="fas fa-plus-circle"></i> Expandir todo</button>
      <button class="control-btn" id="collapseAllBtn" type="button"><i class="fas fa-minus-circle"></i> Contraer todo</button>
      <button class="control-btn" id="topBtn" type="button"><i class="fas fa-arrow-up"></i> Volver arriba</button>
    </section>

    <div class="lesson-container">
      {{DYNAMIC_HTML}}
    </div>

    <footer>
      <p>{{FOOTER_TEXT}}</p>
    </footer>
  </div>

  <script>
    (function() {
      const themeToggleButton = document.getElementById('themeToggleButton');
      const themeIcon = document.getElementById('theme-icon');
      const bodyEl = document.body;
      const expandAllBtn = document.getElementById('expandAllBtn');
      const collapseAllBtn = document.getElementById('collapseAllBtn');
      const topBtn = document.getElementById('topBtn');
      const progressBar = document.getElementById('progressBar');

      function setTheme(theme) {
        bodyEl.setAttribute('data-theme', theme);
        try { localStorage.setItem('gp_theme', theme); } catch (e) {}
        if (themeIcon) { themeIcon.className = theme === 'dark' ? 'fas fa-sun' : 'fas fa-moon'; }
      }

      if (themeToggleButton) {
        themeToggleButton.addEventListener('click', () => {
          const newTheme = (bodyEl.getAttribute('data-theme') || 'dark') === 'dark' ? 'light' : 'dark';
          setTheme(newTheme);
        });
      }

      let preferredTheme = 'dark';
      try { preferredTheme = localStorage.getItem('gp_theme') || 'dark'; } catch (e) {}
      setTheme(preferredTheme);

      const cards = Array.from(document.querySelectorAll('.topic-card'));

      cards.forEach(card => {
        const header = card.querySelector('.topic-header');
        if (header) {
          header.addEventListener('click', () => {
            card.classList.toggle('open');
          });
        }
      });

      if (expandAllBtn) { expandAllBtn.addEventListener('click', () => { cards.forEach(card => card.classList.add('open')); }); }
      if (collapseAllBtn) { collapseAllBtn.addEventListener('click', () => { cards.forEach(card => card.classList.remove('open')); }); }
      if (topBtn) { topBtn.addEventListener('click', () => { window.scrollTo({ top: 0, behavior: 'smooth' }); }); }

      const navButtons = Array.from(document.querySelectorAll('.nav-chip'));
      navButtons.forEach(btn => {
        btn.addEventListener('click', () => {
          const targetId = btn.getAttribute('data-target');
          const target = document.getElementById(targetId);
          if (target) {
            target.classList.add('open');
            setTimeout(() => { target.scrollIntoView({ behavior: 'smooth', block: 'start' }); }, 120);
          }
        });
      });

      const firstTopic = document.querySelector('.topic-card');
      if (firstTopic) { firstTopic.classList.add('open'); }

      const container = document.getElementById('particles-container');
      if (container) {
        for (let i = 0; i < 34; i++) {
          const p = document.createElement('div');
          p.className = 'particle';
          p.style.left = Math.random() * 100 + 'vw';
          const size = (Math.random() * 6 + 3);
          p.style.width = size + 'px';
          p.style.height = size + 'px';
          p.style.animationDelay = Math.random() * -20 + 's';
          p.style.animationDuration = (15 + Math.random() * 12) + 's';
          container.appendChild(p);
        }
      }

      function updateProgress() {
        const scrollTop = document.documentElement.scrollTop || document.body.scrollTop;
        const scrollHeight = document.documentElement.scrollHeight - document.documentElement.clientHeight;
        const progress = scrollHeight > 0 ? (scrollTop / scrollHeight) * 100 : 0;
        progressBar.style.width = progress + '%';
      }

      window.addEventListener('scroll', updateProgress);
      updateProgress();

      // Asegurar que MathJax procese el DOM recién insertado si es necesario
      if (window.MathJax) {
          MathJax.Hub.Queue(["Typeset", MathJax.Hub]);
      }
    })();
  </script>
</body>
</html>
"""

# =========================================================
# 5. ENSAMBLADO
# =========================================================

final_html = (
    plantilla_profesional
    .replace("{{MAIN_TITLE}}", "3. Distribuciones finito-dimensionales")
    .replace("{{INTRO_HTML}}", intro_html_content)
    .replace("{{NAV_HTML}}", navegacion_html)
    .replace("{{DYNAMIC_HTML}}", contenido_dinamico_html)
    .replace("{{FOOTER_TEXT}}", "Material elaborado por el profesor Sergio Gevatschnaider.")
)

display(HTML(final_html))

In [ ]:
# @title Índice Interactivo: 4. Kernels y estructura de covarianza
from IPython.display import display, HTML
import html

# =========================================================
# 1. INTRODUCCIÓN
# =========================================================

intro_html_content = r"""
<div class="content-block" style="margin-top:0;">
    <h2>Resumen de la Sección</h2>
    <p>
        En la teoría de Procesos Gaussianos, el objeto matemático más importante no es solamente la función media,
        sino sobre todo la función de covarianza, también llamada <strong>kernel</strong>. El kernel no es un
        accesorio técnico: es el mecanismo que organiza toda la geometría de las trayectorias posibles.
    </p>
    <p>
        En esta sección estudiaremos qué significa que un kernel sea válido (simetría y semidefinición positiva),
        cómo traduce la estructura de la matriz de covarianza clásica al espacio funcional, y analizaremos en detalle
        las familias de kernels más importantes (RBF, Matérn, Lineal, Periódico), comprendiendo qué hipótesis
        estadísticas y geométricas codifica cada uno sobre la función desconocida.
    </p>
</div>
"""

# =========================================================
# 2. SECCIONES (Texto estructurado y LaTeX corregido)
# =========================================================

secciones_principales_data = [
    {
        "id": "sec-1",
        "titulo": "1. El rol fundamental del Kernel",
        "contenido": r"""
            <h4>La función de covarianza</h4>
            <p>
                Si un proceso gaussiano se escribe como $f \sim \mathcal{GP}(m,k)$, entonces la función media está dada por $m(x)=\mathbb{E}[f(x)]$,
                mientras que el kernel está dado por:
            </p>
            <blockquote>$$k(x,x') = \operatorname{Cov}(f(x),f(x')).$$</blockquote>
            <p>
                Esto significa que el kernel asigna, a cada par de puntos $x, x'$ del dominio $\mathcal{X}$, la covarianza entre los valores aleatorios
                del proceso en esos puntos. En términos intuitivos, el kernel mide cuánto se parecen, desde el punto de vista probabilístico,
                los valores de la función en dos ubicaciones del espacio de entrada.
            </p>
            <p>
                Si $k(x,x')$ es grande, el proceso tiende a producir valores parecidos en $x$ y en $x'$. Si $k(x,x')$ es pequeño,
                el valor de la función en un punto aporta poca información lineal sobre el valor en el otro.
            </p>
            <h4>La matriz de Gram</h4>
            <p>
                Para ver esto con precisión, recordemos que si se eligen puntos $x_1,\dots,x_n$, el vector de evaluaciones tiene distribución normal
                multivariante con matriz de covarianza:
            </p>
            <blockquote>$$K = \begin{pmatrix} k(x_1,x_1) & \cdots & k(x_1,x_n) \\ \vdots & \ddots & \vdots \\ k(x_n,x_1) & \cdots & k(x_n,x_n) \end{pmatrix}.$$</blockquote>
            <p>
                Esta matriz $K$ es la matriz de Gram. Cuando uno elige un kernel, en realidad está eligiendo una familia entera de matrices
                de covarianza para todos los posibles subconjuntos finitos del dominio, fijando así la forma probabilística global del proceso.
            </p>
        """
    },
    {
        "id": "sec-2",
        "titulo": "2. Propiedades matemáticas de un kernel válido",
        "contenido": r"""
            <h4>Simetría y Semidefinición Positiva</h4>
            <p>
                No cualquier función $k(x,x')$ puede actuar como función de covarianza. Para que eso sea posible, debe cumplir dos propiedades fundamentales.
                La primera es la <strong>simetría</strong>, que exige que para todo par $x,x' \in \mathcal{X}$:
            </p>
            <blockquote>$$k(x,x') = k(x',x).$$</blockquote>
            <p>
                Esta condición es inevitable, porque la covarianza siempre satisface $\operatorname{Cov}(f(x),f(x')) = \operatorname{Cov}(f(x'),f(x))$.
            </p>
            <p>
                La segunda propiedad es la <strong>semidefinición positiva</strong>. Esto significa que para cualquier colección finita de puntos
                y cualquier vector de coeficientes $a = (a_1,\dots,a_n)^T \in \mathbb{R}^n$, se debe verificar que:
            </p>
            <blockquote>$$\sum_{i=1}^n \sum_{j=1}^n a_i a_j k(x_i,x_j) \ge 0.$$</blockquote>
            <p>
                De manera equivalente, si $K$ es la matriz de Gram construida a partir de esos puntos, debe cumplirse $a^T K a \ge 0$.
                La razón probabilística es inmediata si se considera la combinación lineal $Z = \sum_{i=1}^n a_i f(x_i)$. Su varianza es:
            </p>
            <blockquote>$$\operatorname{Var}(Z) = \sum_{i=1}^n \sum_{j=1}^n a_i a_j \operatorname{Cov}(f(x_i),f(x_j)) = a^T K a.$$</blockquote>
            <p>
                Como toda varianza es no negativa, debe cumplirse necesariamente que $\operatorname{Var}(Z) \ge 0$. Un kernel es válido porque produce matrices
                de covarianza matemáticamente posibles, actuando como la traducción funcional de la matriz $\Sigma$ en la normal multivariante.
            </p>
        """
    },
    {
        "id": "sec-3",
        "titulo": "3. Kernels clásicos: RBF y Matérn",
        "contenido": r"""
            <h4>Kernel RBF (Gaussiano Radial)</h4>
            <p>
                El primero y más conocido es el kernel RBF o <em>squared exponential</em>. Su expresión es:
            </p>
            <blockquote>$$k_{\text{RBF}}(x,x') = \sigma_f^2 \exp\left( -\frac{\|x-x'\|^2}{2\ell^2} \right).$$</blockquote>
            <p>
                Aquí, $\sigma_f^2 > 0$ es la varianza marginal (escala vertical) y $\ell > 0$ es la longitud de escala. La consecuencia geométrica de este kernel
                es que las funciones muestreadas tienden a ser extremadamente suaves (infinitamente diferenciables).
            </p>
            <h4>Familia de Kernels Matérn</h4>
            <p>
                Para permitir funciones con quiebres o mayor rugosidad, se utilizan los kernels Matérn. Su forma general depende de un parámetro de
                regularidad $\nu > 0$:
            </p>
            <blockquote>$$k_{\text{Matérn}}(x,x') = \sigma_f^2 \frac{2^{1-\nu}}{\Gamma(\nu)} \left( \frac{\sqrt{2\nu}\|x-x'\|}{\ell} \right)^\nu K_\nu\left( \frac{\sqrt{2\nu}\|x-x'\|}{\ell} \right).$$</blockquote>
            <p>
                Cuanto mayor es $\nu$, más suaves son las funciones (convergiendo al RBF cuando $\nu \to \infty$). Los casos particulares más utilizados son:
            </p>
            <ul>
                <li><strong>Para $\nu=1/2$:</strong> Genera trayectorias continuas pero no diferenciables.<br><br> $$k_{1/2}(x,x') = \sigma_f^2 \exp\left( -\frac{\|x-x'\|}{\ell} \right)$$<br></li>
                <li><strong>Para $\nu=3/2$:</strong> Trayectorias una vez diferenciables.<br><br> $$k_{3/2}(x,x') = \sigma_f^2 \left( 1+\frac{\sqrt{3}\|x-x'\|}{\ell} \right) \exp\left( -\frac{\sqrt{3}\|x-x'\|}{\ell} \right)$$<br></li>
                <li><strong>Para $\nu=5/2$:</strong> Trayectorias dos veces diferenciables.<br><br> $$k_{5/2}(x,x') = \sigma_f^2 \left( 1+\frac{\sqrt{5}\|x-x'\|}{\ell} + \frac{5\|x-x'\|^2}{3\ell^2} \right) \exp\left( -\frac{\sqrt{5}\|x-x'\|}{\ell} \right)$$</li>
            </ul>
        """
    },
    {
        "id": "sec-4",
        "titulo": "4. Kernels Lineal, Periódico y de Ruido",
        "contenido": r"""
            <h4>Kernel Lineal</h4>
            <p>
                El kernel lineal codifica la idea de que la covarianza entre dos puntos depende linealmente de su alineación en el espacio de entrada:
            </p>
            <blockquote>$$k_{\text{lin}}(x,x') = \sigma_b^2 + \sigma_v^2 \, x^T x'.$$</blockquote>
            <p>
                Produce funciones que corresponden a modelos lineales bayesianos, imponiendo una estructura global de tipo afín en lugar de una
                suavidad local geométrica.
            </p>
            <h4>Kernel Periódico</h4>
            <p>
                Diseñado para modelar funciones que exhiben oscilaciones regulares (estacionalidad), su forma habitual es:
            </p>
            <blockquote>$$k_{\text{per}}(x,x') = \sigma_f^2 \exp\left( -\frac{2\sin^2\left(\pi\frac{|x-x'|}{p}\right)}{\ell^2} \right).$$</blockquote>
            <p>
                Donde $p > 0$ es el período. La covarianza se repite cada $p$ unidades, favoreciendo funciones cuyo comportamiento se repite cíclicamente.
            </p>
            <h4>Kernel de Ruido Blanco</h4>
            <p>
                Para representar ruido puro o error independiente en las mediciones, se utiliza:
            </p>
            <blockquote>$$k_{\text{noise}}(x,x') = \sigma_n^2 \, \delta_{x,x'},$$</blockquote>
            <p>
                donde $\delta_{x,x'}$ vale $1$ si $x=x'$ y $0$ en caso contrario. Indica que no hay covarianza entre puntos distintos.
            </p>
        """
    },
    {
        "id": "sec-5",
        "titulo": "5. Estacionariedad y combinación de Kernels",
        "contenido": r"""
            <h4>Kernels Estacionarios vs. No Estacionarios</h4>
            <p>
                Un kernel es <strong>estacionario</strong> si depende solo de la diferencia $x-x'$, o de la distancia $\|x-x'\|$:
            </p>
            <blockquote>$$k(x,x') = \kappa(\|x-x'\|).$$</blockquote>
            <p>
                Esto significa que la estructura de covarianza es homogénea en todo el dominio (ej. RBF, Matérn, Periódico). En cambio, un kernel
                no estacionario (como el lineal) depende explícitamente de $x$ y $x'$ por separado, cambiando su estructura según la región del espacio.
            </p>
            <h4>Composición de Kernels</h4>
            <p>
                Los kernels válidos pueden combinarse mediante suma o multiplicación para construir modelos más expresivos:
            </p>
            <blockquote>$$k(x,x') = k_1(x,x') + k_2(x,x') \qquad \text{y} \qquad k(x,x') = k_1(x,x') \, k_2(x,x').$$</blockquote>
            <p>
                Por ejemplo, en la práctica suele sumarse un kernel suave con un kernel de ruido blanco para modelar observaciones ruidosas:
            </p>
            <blockquote>$$k_{\text{total}}(x,x') = k_{\text{smooth}}(x,x') + \sigma_n^2 \, \delta_{x,x'}.$$</blockquote>
            <p>
                En conclusión, elegir un kernel equivale a responder: <em>¿qué clase de funciones considero plausibles antes de ver los datos?</em>
                La elección del kernel es el paso donde la intuición sobre el problema se transforma en estructura probabilística formal.
            </p>
        """
    }
]

# =========================================================
# 3. GENERACIÓN HTML
# =========================================================

def generar_tarjetas_acordeon(datos):
    bloques = []
    for seccion in datos:
        bloques.append(f"""
        <div class="topic-card" id="{html.escape(seccion['id'])}">
            <div class="topic-header">
                <span class="topic-title">{seccion['titulo']}</span>
                <i class="fas fa-chevron-down expand-icon"></i>
            </div>
            <div class="topic-content">
                {seccion['contenido']}
            </div>
        </div>
        """)
    return "\n".join(bloques)

def generar_navegacion(datos):
    chips = []
    for seccion in datos:
        chips.append(
            f'<button class="nav-chip" type="button" data-target="{html.escape(seccion["id"])}">{seccion["titulo"]}</button>'
        )
    return "\n".join(chips)

contenido_dinamico_html = generar_tarjetas_acordeon(secciones_principales_data)
navegacion_html = generar_navegacion(secciones_principales_data)

# =========================================================
# 4. PLANTILLA HTML
# =========================================================

plantilla_profesional = r"""
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{{MAIN_TITLE}}</title>

  <script type="text/x-mathjax-config">
    MathJax.Hub.Config({
      tex2jax: {
        inlineMath: [['$','$']],
        displayMath: [['$$','$$']],
        processEscapes: true
      }
    });
  </script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-MML-AM_CHTML" async></script>

  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
  <link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css" rel="stylesheet">

  <style>
    :root {
      --bg-primary: linear-gradient(135deg, #f0f9ff 0%, #e0f2fe 100%);
      --bg-secondary: rgba(255, 255, 255, 0.85);
      --bg-tertiary: rgba(248, 250, 252, 0.8);
      --text-primary: #0c4a6e;
      --text-secondary: #075985;
      --accent-primary: #0ea5e9;
      --accent-secondary: #38bdf8;
      --accent-gradient: linear-gradient(135deg, var(--accent-primary) 0%, var(--accent-secondary) 100%);
      --border-color: rgba(226, 232, 240, 0.8);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.08);
      --border-radius: 20px;
      --transition: all 0.4s cubic-bezier(0.25, 0.8, 0.25, 1);
      --nav-chip-bg: rgba(255,255,255,0.8);
      --progress-bg: rgba(255,255,255,0.45);
    }

    [data-theme="dark"] {
      --bg-primary: linear-gradient(135deg, #0c4a6e 0%, #1e3a8a 100%);
      --bg-secondary: rgba(30, 58, 138, 0.85);
      --bg-tertiary: rgba(30, 64, 175, 0.7);
      --text-primary: #f0f9ff;
      --text-secondary: #dbeafe;
      --accent-primary: #38bdf8;
      --accent-secondary: #7dd3fc;
      --border-color: rgba(255, 255, 255, 0.15);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.20);
      --nav-chip-bg: rgba(30, 58, 138, 0.65);
      --progress-bg: rgba(255,255,255,0.15);
    }

    * { margin: 0; padding: 0; box-sizing: border-box; }
    html { scroll-behavior: smooth; }
    body { font-family: 'Inter', sans-serif; line-height: 1.8; background: var(--bg-primary); color: var(--text-primary); transition: var(--transition); min-height: 100vh; position: relative; overflow-x: hidden; }

    .particles { position: fixed; top: 0; left: 0; width: 100%; height: 100%; pointer-events: none; z-index: -1; }
    .particle { position: absolute; border-radius: 50%; animation: float 25s infinite linear; opacity: 0; background: rgba(255, 255, 255, 0.55); box-shadow: 0 0 15px rgba(255,255,255,0.25); }
    @keyframes float { 0% { transform: translateY(100vh) rotate(0deg); opacity: 0; } 10%, 90% { opacity: 0.55; } 100% { transform: translateY(-10vh) rotate(360deg); opacity: 0; } }

    .progress-wrapper { position: fixed; top: 0; left: 0; width: 100%; height: 6px; background: var(--progress-bg); z-index: 1200; backdrop-filter: blur(10px); }
    .progress-bar { height: 100%; width: 0%; background: var(--accent-gradient); transition: width 0.15s ease; }

    .theme-toggle { position: fixed; top: 1.2rem; right: 1.2rem; width: 60px; height: 60px; border: 1px solid var(--border-color); border-radius: 50%; background: var(--bg-secondary); backdrop-filter: blur(15px); box-shadow: var(--shadow-card); cursor: pointer; display: flex; align-items: center; justify-content: center; font-size: 1.35rem; color: var(--text-primary); transition: var(--transition); z-index: 1100; }
    .theme-toggle:hover { transform: scale(1.15) rotate(180deg); }

    .container { max-width: 1100px; margin: 0 auto; padding: 2rem; z-index: 1; position: relative; }
    .header { text-align: center; margin-bottom: 2rem; position: relative; padding-top: 1.2rem; }
    .header-badge { display: inline-flex; align-items: center; gap: 0.6rem; background: var(--bg-secondary); border: 1px solid var(--border-color); border-radius: 999px; padding: 0.7rem 1rem; backdrop-filter: blur(15px); box-shadow: var(--shadow-card); margin-bottom: 1rem; color: var(--text-secondary); font-size: 0.95rem; }

    .main-title { font-size: clamp(2.2rem, 5vw, 3.8rem); font-weight: 800; background: var(--accent-gradient); -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text; margin-bottom: 1rem; line-height: 1.15; }
    .subtitle { color: var(--text-secondary); max-width: 900px; margin: 0 auto; font-size: 1.03rem; }

    .content-block, .quick-nav, .control-bar, .topic-card { background: var(--bg-secondary); backdrop-filter: blur(20px); border-radius: var(--border-radius); box-shadow: var(--shadow-card); border: 2px solid var(--border-color); }
    .content-block { padding: 2rem; margin-top: 2rem; }
    .content-block h2 { color: var(--text-primary); margin-bottom: 1rem; font-size: 1.55rem; }
    .content-block p, .content-block li, .subtitle { color: var(--text-secondary); }

    .quick-nav { margin-top: 2rem; padding: 1.4rem; }
    .quick-nav h3 { margin-bottom: 1rem; color: var(--text-primary); font-size: 1.2rem; }
    .nav-grid { display: flex; flex-wrap: wrap; gap: 0.75rem; }
    .nav-chip { padding: 0.65rem 0.95rem; border-radius: 999px; background: var(--nav-chip-bg); color: var(--text-primary); border: 1px solid var(--border-color); font-size: 0.92rem; transition: var(--transition); box-shadow: 0 8px 18px rgba(0,0,0,0.06); cursor: pointer; font-family: inherit; text-align: left; }
    .nav-chip:hover { transform: translateY(-2px); background: var(--bg-tertiary); }

    .control-bar { margin-top: 1.5rem; padding: 1rem 1.2rem; display: flex; gap: 0.8rem; flex-wrap: wrap; justify-content: center; }
    .control-btn { border: 1px solid var(--border-color); background: var(--nav-chip-bg); color: var(--text-primary); border-radius: 999px; padding: 0.75rem 1rem; cursor: pointer; font-family: inherit; font-size: 0.95rem; transition: var(--transition); }
    .control-btn:hover { transform: translateY(-2px); background: var(--bg-tertiary); }

    .lesson-container { display: flex; flex-direction: column; gap: 1.3rem; margin-top: 1.6rem; }
    .topic-card { overflow: hidden; transition: var(--transition); scroll-margin-top: 90px; }
    .topic-card:hover { transform: translateY(-2px); }
    .topic-header { cursor: pointer; padding: 1.5rem 2rem; display: flex; justify-content: space-between; align-items: center; gap: 1rem; user-select: none; }
    .topic-title { font-size: 1.22rem; font-weight: 700; color: var(--text-primary); line-height: 1.4; }
    .expand-icon { font-size: 1.15rem; color: var(--text-secondary); transition: var(--transition); flex-shrink: 0; }
    .topic-card.open .expand-icon { transform: rotate(180deg); }

    .topic-content { max-height: 0; overflow: hidden; transition: max-height 1.2s ease, padding 1.2s ease; background: var(--bg-tertiary); }
    .topic-card.open .topic-content { max-height: 6500px; padding: 1.6rem 2rem 1.8rem 2rem; border-top: 1px solid var(--border-color); }
    .topic-content h4 { color: var(--text-primary); margin-top: 1.5rem; margin-bottom: 0.55rem; font-size: 1.08rem; border-left: 4px solid var(--accent-primary); padding-left: 12px; }
    .topic-content h4:first-child { margin-top: 0; }
    .topic-content p, .topic-content li { color: var(--text-secondary); line-height: 1.8; margin-bottom: 0.95rem; }
    .topic-content strong { color: var(--text-primary); font-weight: 700; }

    .topic-content blockquote { text-align: center; border-left: 4px solid var(--accent-primary); margin: 1.4rem 0; font-style: normal; color: var(--text-secondary); background: rgba(127, 140, 141, 0.05); border-radius: 0 10px 10px 0; padding: 1rem 1.5rem; overflow-x: auto; }

    footer { text-align: center; margin-top: 4rem; padding-top: 2rem; border-top: 1px solid var(--border-color); }
    footer p { color: var(--text-secondary); font-size: 0.95rem; opacity: 0.9; }

    @media (max-width: 768px) {
      .container { padding: 1rem; }
      .topic-header { padding: 1.1rem 1.3rem; }
      .topic-card.open .topic-content { padding: 1.1rem 1.3rem 1.3rem 1.3rem; }
      .theme-toggle { width: 54px; height: 54px; top: 1rem; right: 1rem; }
    }
  </style>
</head>

<body data-theme="dark">
  <div class="progress-wrapper">
    <div class="progress-bar" id="progressBar"></div>
  </div>

  <div class="particles" id="particles-container"></div>

  <div class="theme-toggle" id="themeToggleButton" title="Cambiar tema">
    <i class="fas fa-moon" id="theme-icon"></i>
  </div>

  <div class="container">
    <header class="header">
      <div class="header-badge">
        <i class="fas fa-project-diagram"></i>
        <span>Aplicaciones en Data Science · Módulo Matemático</span>
      </div>
      <h1 class="main-title">{{MAIN_TITLE}}</h1>
      <p class="subtitle">
        Estudio de la función de covarianza, sus propiedades matemáticas y las principales familias de kernels utilizadas en el modelado espacial y temporal.
      </p>
    </header>

    {{INTRO_HTML}}

    <section class="quick-nav">
      <h3>Navegación rápida</h3>
      <div class="nav-grid">
        {{NAV_HTML}}
      </div>
    </section>

    <section class="control-bar">
      <button class="control-btn" id="expandAllBtn" type="button"><i class="fas fa-plus-circle"></i> Expandir todo</button>
      <button class="control-btn" id="collapseAllBtn" type="button"><i class="fas fa-minus-circle"></i> Contraer todo</button>
      <button class="control-btn" id="topBtn" type="button"><i class="fas fa-arrow-up"></i> Volver arriba</button>
    </section>

    <div class="lesson-container">
      {{DYNAMIC_HTML}}
    </div>

    <footer>
      <p>{{FOOTER_TEXT}}</p>
    </footer>
  </div>

  <script>
    (function() {
      const themeToggleButton = document.getElementById('themeToggleButton');
      const themeIcon = document.getElementById('theme-icon');
      const bodyEl = document.body;
      const expandAllBtn = document.getElementById('expandAllBtn');
      const collapseAllBtn = document.getElementById('collapseAllBtn');
      const topBtn = document.getElementById('topBtn');
      const progressBar = document.getElementById('progressBar');

      function setTheme(theme) {
        bodyEl.setAttribute('data-theme', theme);
        try { localStorage.setItem('gp_theme', theme); } catch (e) {}
        if (themeIcon) { themeIcon.className = theme === 'dark' ? 'fas fa-sun' : 'fas fa-moon'; }
      }

      if (themeToggleButton) {
        themeToggleButton.addEventListener('click', () => {
          const newTheme = (bodyEl.getAttribute('data-theme') || 'dark') === 'dark' ? 'light' : 'dark';
          setTheme(newTheme);
        });
      }

      let preferredTheme = 'dark';
      try { preferredTheme = localStorage.getItem('gp_theme') || 'dark'; } catch (e) {}
      setTheme(preferredTheme);

      const cards = Array.from(document.querySelectorAll('.topic-card'));

      cards.forEach(card => {
        const header = card.querySelector('.topic-header');
        if (header) {
          header.addEventListener('click', () => {
            card.classList.toggle('open');
          });
        }
      });

      if (expandAllBtn) { expandAllBtn.addEventListener('click', () => { cards.forEach(card => card.classList.add('open')); }); }
      if (collapseAllBtn) { collapseAllBtn.addEventListener('click', () => { cards.forEach(card => card.classList.remove('open')); }); }
      if (topBtn) { topBtn.addEventListener('click', () => { window.scrollTo({ top: 0, behavior: 'smooth' }); }); }

      const navButtons = Array.from(document.querySelectorAll('.nav-chip'));
      navButtons.forEach(btn => {
        btn.addEventListener('click', () => {
          const targetId = btn.getAttribute('data-target');
          const target = document.getElementById(targetId);
          if (target) {
            target.classList.add('open');
            setTimeout(() => { target.scrollIntoView({ behavior: 'smooth', block: 'start' }); }, 120);
          }
        });
      });

      const firstTopic = document.querySelector('.topic-card');
      if (firstTopic) { firstTopic.classList.add('open'); }

      const container = document.getElementById('particles-container');
      if (container) {
        for (let i = 0; i < 34; i++) {
          const p = document.createElement('div');
          p.className = 'particle';
          p.style.left = Math.random() * 100 + 'vw';
          const size = (Math.random() * 6 + 3);
          p.style.width = size + 'px';
          p.style.height = size + 'px';
          p.style.animationDelay = Math.random() * -20 + 's';
          p.style.animationDuration = (15 + Math.random() * 12) + 's';
          container.appendChild(p);
        }
      }

      function updateProgress() {
        const scrollTop = document.documentElement.scrollTop || document.body.scrollTop;
        const scrollHeight = document.documentElement.scrollHeight - document.documentElement.clientHeight;
        const progress = scrollHeight > 0 ? (scrollTop / scrollHeight) * 100 : 0;
        progressBar.style.width = progress + '%';
      }

      window.addEventListener('scroll', updateProgress);
      updateProgress();

      if (window.MathJax) {
          MathJax.Hub.Queue(["Typeset", MathJax.Hub]);
      }
    })();
  </script>
</body>
</html>
"""

# =========================================================
# 5. ENSAMBLADO
# =========================================================

final_html = (
    plantilla_profesional
    .replace("{{MAIN_TITLE}}", "4. Kernels y estructura de covarianza")
    .replace("{{INTRO_HTML}}", intro_html_content)
    .replace("{{NAV_HTML}}", navegacion_html)
    .replace("{{DYNAMIC_HTML}}", contenido_dinamico_html)
    .replace("{{FOOTER_TEXT}}", "Material elaborado por el profesor Sergio Gevatschnaider.")
)

display(HTML(final_html))

In [ ]:
# @title Índice Interactivo: 5. Geometría e interpretación probabilística (Versión Completa)
from IPython.display import display, HTML
import html

# =========================================================
# 1. INTRODUCCIÓN
# =========================================================

intro_html_content = r"""
<div class="content-block" style="margin-top:0;">
    <h2>Resumen de la Sección</h2>
    <p>
        Una vez introducidos la definición formal del proceso gaussiano y el papel del kernel como función de covarianza,
        el paso siguiente consiste en comprender su interpretación geométrica y probabilística. Esta interpretación es importante
        porque permite dejar de ver al proceso gaussiano como una definición abstracta y empezar a entenderlo como un modelo
        sobre formas posibles de funciones, sobre incertidumbre y sobre relaciones de semejanza entre entradas.
    </p>
    <p>
        En efecto, un proceso gaussiano no es solamente una herramienta algebraica para construir matrices de covarianza;
        es, ante todo, una manera de describir un conjunto de funciones plausibles y de asignarles una estructura de variación y dependencia.
    </p>
</div>
"""

# =========================================================
# 2. SECCIONES (Texto 100% completo y LaTeX corregido)
# =========================================================

secciones_principales_data = [
    {
        "id": "sec-1",
        "titulo": "1. Interacción entre Centro y Covarianza",
        "contenido": r"""
            <p>
                Recordemos que un proceso gaussiano se escribe como
            </p>
            <blockquote>$$f \sim \mathcal{GP}(m,k),$$</blockquote>
            <p>
                donde la función media está dada por
            </p>
            <blockquote>$$m(x) = \mathbb{E}[f(x)]$$</blockquote>
            <p>
                y la función de covarianza o kernel está dada por
            </p>
            <blockquote>$$k(x,x') = \operatorname{Cov}(f(x),f(x')).$$</blockquote>
            <p>
                Estas dos funciones cumplen papeles conceptualmente distintos. La función media describe la tendencia central del proceso, mientras que la función de covarianza describe cómo fluctúan conjuntamente los valores de la función en distintos puntos del dominio. Toda la geometría probabilística del modelo nace precisamente de esta interacción entre centro y covarianza.
            </p>
            <p>
                Conviene empezar por la función media. Si se fija un punto $x$ del dominio, la variable aleatoria $f(x)$ tiene esperanza $m(x)$. Esto quiere decir que, si el proceso se repitiera muchas veces y se observara siempre el valor en ese mismo punto, el promedio de esas observaciones tendería a $m(x)$. Por ello, la media debe interpretarse como la tendencia central de la función. No es una media de parámetros, ni una media global de toda la trayectoria, sino una media puntual, definida para cada valor del dominio.
            </p>
            <p>
                Si se dibujan muchas funciones generadas por un mismo proceso gaussiano, la curva $m(x)$ actúa como la línea alrededor de la cual esas funciones tienden a oscilar. Por ejemplo, si
            </p>
            <blockquote>$$m(x) = 0 \quad \text{para todo } x,$$</blockquote>
            <p>
                entonces el proceso está centrado alrededor de la función nula. En ese caso, antes de ver datos, no se favorecen sistemáticamente valores positivos ni negativos. Si en cambio la media es
            </p>
            <blockquote>$$m(x) = ax+b,$$</blockquote>
            <p>
                entonces el proceso queda centrado alrededor de una tendencia lineal. Las realizaciones del proceso no coincidirán exactamente con esa recta, pero tenderán a fluctuar alrededor de ella. Desde este punto de vista, la función media cumple un papel análogo al del vector $\mu$ en la normal multivariante. En una distribución normal multivariante,
            </p>
            <blockquote>$$X \sim \mathcal{N}(\mu,\Sigma),$$</blockquote>
            <p>
                el vector $\mu$ indica el centro de la nube de probabilidad. En un proceso gaussiano, la función $m(x)$ indica el centro funcional alrededor del cual se distribuyen las trayectorias.
            </p>
        """
    },
    {
        "id": "sec-2",
        "titulo": "2. La covarianza como geometría de semejanza",
        "contenido": r"""
            <p>
                Sin embargo, la media por sí sola no dice casi nada sobre la forma de las funciones. Dos procesos gaussianos pueden tener exactamente la misma media y, aun así, producir realizaciones radicalmente distintas. Toda esa diferencia reside en la covarianza. La función de covarianza
            </p>
            <blockquote>$$k(x,x') = \operatorname{Cov}(f(x),f(x'))$$</blockquote>
            <p>
                describe la dependencia entre los valores del proceso en distintos puntos del dominio. Si $k(x,x')$ es grande y positivo, entonces cuando $f(x)$ toma un valor mayor que su media, es probable que $f(x')$ también tienda a estar por encima de su media. Si $k(x,x')$ es pequeño o cercano a cero, el valor de la función en $x$ aporta poca información lineal sobre el valor en $x'$. Si $k(x,x')$ es negativo, el modelo expresa una relación de compensación: cuando el valor en un punto está por encima de la media, el del otro tiende a estar por debajo.
            </p>
            <p>
                Esta observación tiene una interpretación geométrica muy profunda. En un proceso gaussiano, la covarianza no solo mide dispersión; también actúa como una geometría de semejanza entre entradas. El valor de $k(x,x')$ puede leerse como una medida de proximidad estadística entre los puntos $x$ y $x'$. Si dos entradas tienen covarianza alta, el proceso las considera "similares" en el sentido de que sus salidas tenderán a comportarse de manera parecida. Si la covarianza es baja, el proceso las considera poco relacionadas. De este modo, el kernel induce una noción de geometría sobre el dominio de entrada que no necesariamente coincide con la geometría euclídea ordinaria, aunque muchas veces la incorpora.
            </p>
            <p>
                Para ver esto con claridad, consideremos un kernel radial como el kernel RBF,
            </p>
            <blockquote>$$k(x,x') = \sigma_f^2 \exp\left( -\frac{\|x-x'\|^2}{2\ell^2} \right).$$</blockquote>
            <p>
                En este caso, la covarianza depende solamente de la distancia entre $x$ y $x'$. Si los puntos están muy cerca, la covarianza es próxima a $\sigma_f^2$; si están lejos, la covarianza decrece rápidamente hacia cero. El proceso, por lo tanto, traduce la cercanía geométrica en el espacio de entrada en semejanza probabilística en el espacio de salida. Dos entradas cercanas producen valores de la función fuertemente correlacionados; dos entradas lejanas se vuelven casi independientes en términos de correlación lineal. Así, la covarianza implementa una noción de vecindad funcional.
            </p>
            <p>
                Esta idea puede hacerse todavía más precisa mediante la cantidad
            </p>
            <blockquote>$$d_k(x,x')^2 = k(x,x) + k(x',x') - 2k(x,x').$$</blockquote>
            <p>
                Esta expresión tiene la forma de una distancia cuadrática inducida por el kernel. En efecto, si $k$ es un kernel semidefinido positivo, esta cantidad define una pseudodistancia sobre el dominio. Cuanto mayor es $k(x,x')$, menor es $d_k(x,x')$, y viceversa. Esto muestra que el kernel no es simplemente una receta para fabricar matrices de covarianza: es un objeto geométrico que organiza el dominio en términos de semejanza estadística. En un sentido profundo, un proceso gaussiano ve el espacio de entrada a través de la lente del kernel.
            </p>
        """
    },
    {
        "id": "sec-3",
        "titulo": "3. Bandas de incertidumbre e interpretación visual",
        "contenido": r"""
            <p>
                Esta geometría se manifiesta con fuerza cuando se estudian las bandas de incertidumbre. Fijemos un punto $x$. Como $f(x)$ es una variable aleatoria normal con media $m(x)$ y varianza $k(x,x)$, se tiene
            </p>
            <blockquote>$$f(x) \sim \mathcal{N}(m(x), k(x,x)).$$</blockquote>
            <p>
                Por tanto, una banda puntual natural de incertidumbre alrededor de la media viene dada por
            </p>
            <blockquote>$$m(x) \pm c\sqrt{k(x,x)},$$</blockquote>
            <p>
                donde $c$ es una constante elegida según el nivel de probabilidad que se quiera representar. Por ejemplo, si $c=1.96$, se obtiene la banda puntual asociada aproximadamente al <strong>95%</strong> de probabilidad en el caso gaussiano univariado:
            </p>
            <blockquote>$$m(x) \pm 1.96\sqrt{k(x,x)}.$$</blockquote>
            <p>
                Esta fórmula debe interpretarse con cuidado. No significa que la trayectoria completa del proceso quede dentro de esa banda con probabilidad <strong>95%</strong>; significa, más modestamente, que para cada punto fijo $x$, el valor aleatorio $f(x)$ cae dentro de esa banda con probabilidad aproximada <strong>95%</strong>. Es, por lo tanto, una banda puntual, no simultánea.
            </p>
            <p>
                Aun así, esta representación es extraordinariamente útil. La curva $m(x)$ actúa como la predicción central del proceso, mientras que la cantidad
            </p>
            <blockquote>$$\sqrt{k(x,x)}$$</blockquote>
            <p>
                mide el tamaño local de la incertidumbre. En regiones donde la varianza es grande, las bandas se ensanchan; en regiones donde la varianza es pequeña, se estrechan. En un proceso gaussiano previo, antes de ver datos, esta incertidumbre está determinada enteramente por el kernel. Después de condicionar respecto de observaciones, la varianza posterior disminuye cerca de los puntos observados y suele aumentar en zonas alejadas. Esta es una de las razones por las cuales los procesos gaussianos resultan tan valiosos en predicción e interpolación: no solo producen una estimación central, sino también una medida explícita y local de la ignorancia restante.
            </p>
            <div style="text-align: center; margin: 2rem 0;">

            </div>
            <p>
                Las bandas de incertidumbre permiten visualizar probabilísticamente el modelo. Si uno grafica $m(x)$ y añade las curvas
            </p>
            <blockquote>$$m(x) + 2\sqrt{k(x,x)} \qquad \text{y} \qquad m(x) - 2\sqrt{k(x,x)},$$</blockquote>
            <p>
                obtiene una región en torno a la media dentro de la cual se concentran muchas realizaciones del proceso. Esta imagen es el análogo funcional de los elipsoides de densidad de la normal multivariante. En dimensión finita, una normal multivariante con media $\mu$ y covarianza $\Sigma$ organiza su masa de probabilidad alrededor de elipsoides determinados por la forma cuadrática
            </p>
            <blockquote>$$(x-\mu)^T \Sigma^{-1} (x-\mu).$$</blockquote>
            <p>
                En el caso de un proceso gaussiano, no se trabaja directamente con un único elipsoide en un espacio funcional infinito-dimensional, pero la misma lógica reaparece cuando se proyecta el proceso sobre un conjunto finito de puntos del dominio. Allí vuelven a aparecer gaussianas multivariantes, con sus elipsoides de nivel determinados por la matriz de Gram.
            </p>
        """
    },
    {
        "id": "sec-4",
        "titulo": "4. Muestras de funciones desde un GP",
        "contenido": r"""
            <p>
                Esto conduce naturalmente a la noción de muestras de funciones desde un proceso gaussiano. Decir que
            </p>
            <blockquote>$$f \sim \mathcal{GP}(m,k)$$</blockquote>
            <p>
                significa que una realización concreta del proceso es una función $x \mapsto f(x)$. Para generar o imaginar una muestra del proceso, puede elegirse primero un conjunto finito de puntos $x_1,\dots,x_n$. Entonces el vector
            </p>
            <blockquote>
                $$\begin{pmatrix} f(x_1) \\ f(x_2) \\ \vdots \\ f(x_n) \end{pmatrix} \sim \mathcal{N}(\mu,K),$$
            </blockquote>
            <p>
                con
            </p>
            <blockquote>
                $$\mu = \begin{pmatrix} m(x_1) \\ m(x_2) \\ \vdots \\ m(x_n) \end{pmatrix} \qquad \text{y} \qquad K_{ij} = k(x_i,x_j).$$
            </blockquote>
            <p>
                Una muestra de esta distribución multivariante da un conjunto de valores $(z_1,\dots,z_n)$. Si se repite este procedimiento sobre una malla suficientemente fina del dominio y luego se unen los puntos de manera adecuada, se obtiene una representación de una trayectoria del proceso. Cada muestra corresponde a una función posible, y el conjunto de todas esas funciones refleja la distribución inducida por el GP.
            </p>
            <div style="text-align: center; margin: 2rem 0;">

            </div>
            <p>
                Esto tiene un valor interpretativo enorme. Si el kernel es RBF con longitud de escala grande, las muestras tenderán a ser muy suaves y a variar lentamente. Si el kernel es Matérn con parámetro de regularidad bajo, las trayectorias serán más rugosas. Si el kernel es periódico, las funciones muestreadas mostrarán oscilaciones cíclicas. Si el kernel es lineal, las realizaciones se parecerán a funciones afines. En otras palabras, dibujar muestras del GP es una manera directa de visualizar qué cree el modelo antes de ver datos. Cada kernel induce un "estilo" particular de trayectorias.
            </p>
            <p>
                Consideremos, por ejemplo, un proceso gaussiano de media nula y kernel RBF:
            </p>
            <blockquote>$$m(x) = 0, \qquad k(x,x') = \sigma_f^2 \exp\left( -\frac{(x-x')^2}{2\ell^2} \right).$$</blockquote>
            <p>
                Si $\ell$ es muy grande, los puntos alejados del dominio siguen estando bastante correlacionados, y las funciones resultantes tienden a ser globalmente suaves y lentas. Si $\ell$ es pequeño, la correlación cae rápido y las trayectorias pueden cambiar más bruscamente. Si $\sigma_f^2$ aumenta, la amplitud vertical de las oscilaciones se vuelve mayor. Esto muestra que el aspecto de las funciones muestreadas está controlado directamente por la covarianza. La media fija el eje alrededor del cual oscilan las trayectorias; el kernel fija cómo oscilan.
            </p>
        """
    },
    {
        "id": "sec-5",
        "titulo": "5. El kernel como análogo funcional de la matriz Σ",
        "contenido": r"""
            <p>
                Desde este punto de vista, el kernel debe ser entendido como el análogo funcional de la matriz $\Sigma$ en la normal multivariante. Esta analogía es exacta y central. En una normal multivariante,
            </p>
            <blockquote>$$X \sim \mathcal{N}(\mu,\Sigma),$$</blockquote>
            <p>
                el vector $\mu$ fija el centro de la distribución y la matriz $\Sigma$ fija la forma de la nube de probabilidad. En un proceso gaussiano,
            </p>
            <blockquote>$$f \sim \mathcal{GP}(m,k),$$</blockquote>
            <p>
                la función media $m$ fija el centro funcional del proceso y el kernel $k$ fija la forma de la distribución sobre funciones. Si uno restringe el proceso a un conjunto finito de puntos $x_1,\dots,x_n$, la analogía se vuelve completamente explícita. El vector de medias es
            </p>
            <blockquote>
                $$\mu = \begin{pmatrix} m(x_1) \\ \vdots \\ m(x_n) \end{pmatrix},$$
            </blockquote>
            <p>
                y la matriz de covarianza es
            </p>
            <blockquote>
                $$K = \begin{pmatrix} k(x_1,x_1) & k(x_1,x_2) & \cdots & k(x_1,x_n) \\ k(x_2,x_1) & k(x_2,x_2) & \cdots & k(x_2,x_n) \\ \vdots & \vdots & \ddots & \vdots \\ k(x_n,x_1) & k(x_n,x_2) & \cdots & k(x_n,x_n) \end{pmatrix}.$$
            </blockquote>
            <p>
                Es decir, la matriz $\Sigma$ de la MVN es reemplazada, en el contexto funcional, por una regla que genera matrices de covarianza para cualquier colección finita de puntos. Esa regla es precisamente el kernel. Podría decirse que $\Sigma$ es una covarianza finita y fija, mientras que $k$ es una covarianza móvil, capaz de adaptarse a cualquier tamaño y ubicación del conjunto de evaluación.
            </p>
            <p>
                Esta analogía también ayuda a entender el contenido geométrico del proceso. En la MVN, $\Sigma$ determina las direcciones principales de dispersión, la distancia de Mahalanobis y la forma de las regiones de alta densidad. En el GP, el kernel determina qué funciones están más concentradas alrededor de la media, cómo se relacionan entre sí los valores funcionales y cómo se propaga la información desde unas entradas hacia otras. El paso de $\Sigma$ a $k$ es, por lo tanto, el paso de una geometría sobre vectores a una geometría sobre funciones.
            </p>
        """
    },
    {
        "id": "sec-6",
        "titulo": "6. Síntesis y Conclusión",
        "contenido": r"""
            <p>
                Se puede condensar todo esto en una idea fundamental. Un proceso gaussiano no debe imaginarse como una sola curva acompañada por un error. Debe imaginarse como una nube de funciones posibles, centrada alrededor de una media y estructurada por una covarianza. Esa covarianza transforma semejanza entre entradas en correlación entre salidas, fija la suavidad típica de las trayectorias, determina la amplitud de las bandas de incertidumbre y organiza la familia completa de funciones plausibles. La interpretación probabilística y la interpretación geométrica no son dos lecturas separadas del modelo: son dos maneras de mirar la misma estructura.
            </p>
            <p>
                En resumen, la media de un proceso gaussiano representa la tendencia central de la función, mientras que la covarianza representa la geometría de semejanza entre puntos del dominio. Las bandas de incertidumbre describen, punto por punto, cuánto puede apartarse la función de su media. Las muestras del proceso permiten visualizar trayectorias completas coherentes con esa estructura probabilística. Y el kernel cumple exactamente el papel que juega la matriz $\Sigma$ en la normal multivariante, aunque ahora trasladado al contexto funcional. Gracias a esta analogía, el proceso gaussiano puede entenderse como una distribución gaussiana en el espacio de funciones, con centro funcional dado por $m$ y geometría funcional dada por $k$.
            </p>
        """
    }
]

# =========================================================
# 3. GENERACIÓN HTML
# =========================================================

def generar_tarjetas_acordeon(datos):
    bloques = []
    for seccion in datos:
        bloques.append(f"""
        <div class="topic-card" id="{html.escape(seccion['id'])}">
            <div class="topic-header">
                <span class="topic-title">{seccion['titulo']}</span>
                <i class="fas fa-chevron-down expand-icon"></i>
            </div>
            <div class="topic-content">
                {seccion['contenido']}
            </div>
        </div>
        """)
    return "\n".join(bloques)

def generar_navegacion(datos):
    chips = []
    for seccion in datos:
        chips.append(
            f'<button class="nav-chip" type="button" data-target="{html.escape(seccion["id"])}">{seccion["titulo"]}</button>'
        )
    return "\n".join(chips)

contenido_dinamico_html = generar_tarjetas_acordeon(secciones_principales_data)
navegacion_html = generar_navegacion(secciones_principales_data)

# =========================================================
# 4. PLANTILLA HTML
# =========================================================

plantilla_profesional = r"""
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{{MAIN_TITLE}}</title>

  <script type="text/x-mathjax-config">
    MathJax.Hub.Config({
      tex2jax: {
        inlineMath: [['$','$']],
        displayMath: [['$$','$$']],
        processEscapes: true
      }
    });
  </script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-MML-AM_CHTML" async></script>

  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
  <link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css" rel="stylesheet">

  <style>
    :root {
      --bg-primary: linear-gradient(135deg, #f0f9ff 0%, #e0f2fe 100%);
      --bg-secondary: rgba(255, 255, 255, 0.85);
      --bg-tertiary: rgba(248, 250, 252, 0.8);
      --text-primary: #0c4a6e;
      --text-secondary: #075985;
      --accent-primary: #0ea5e9;
      --accent-secondary: #38bdf8;
      --accent-gradient: linear-gradient(135deg, var(--accent-primary) 0%, var(--accent-secondary) 100%);
      --border-color: rgba(226, 232, 240, 0.8);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.08);
      --border-radius: 20px;
      --transition: all 0.4s cubic-bezier(0.25, 0.8, 0.25, 1);
      --nav-chip-bg: rgba(255,255,255,0.8);
      --progress-bg: rgba(255,255,255,0.45);
    }

    [data-theme="dark"] {
      --bg-primary: linear-gradient(135deg, #0c4a6e 0%, #1e3a8a 100%);
      --bg-secondary: rgba(30, 58, 138, 0.85);
      --bg-tertiary: rgba(30, 64, 175, 0.7);
      --text-primary: #f0f9ff;
      --text-secondary: #dbeafe;
      --accent-primary: #38bdf8;
      --accent-secondary: #7dd3fc;
      --border-color: rgba(255, 255, 255, 0.15);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.20);
      --nav-chip-bg: rgba(30, 58, 138, 0.65);
      --progress-bg: rgba(255,255,255,0.15);
    }

    * { margin: 0; padding: 0; box-sizing: border-box; }
    html { scroll-behavior: smooth; }
    body { font-family: 'Inter', sans-serif; line-height: 1.8; background: var(--bg-primary); color: var(--text-primary); transition: var(--transition); min-height: 100vh; position: relative; overflow-x: hidden; }

    .particles { position: fixed; top: 0; left: 0; width: 100%; height: 100%; pointer-events: none; z-index: -1; }
    .particle { position: absolute; border-radius: 50%; animation: float 25s infinite linear; opacity: 0; background: rgba(255, 255, 255, 0.55); box-shadow: 0 0 15px rgba(255,255,255,0.25); }
    @keyframes float { 0% { transform: translateY(100vh) rotate(0deg); opacity: 0; } 10%, 90% { opacity: 0.55; } 100% { transform: translateY(-10vh) rotate(360deg); opacity: 0; } }

    .progress-wrapper { position: fixed; top: 0; left: 0; width: 100%; height: 6px; background: var(--progress-bg); z-index: 1200; backdrop-filter: blur(10px); }
    .progress-bar { height: 100%; width: 0%; background: var(--accent-gradient); transition: width 0.15s ease; }

    .theme-toggle { position: fixed; top: 1.2rem; right: 1.2rem; width: 60px; height: 60px; border: 1px solid var(--border-color); border-radius: 50%; background: var(--bg-secondary); backdrop-filter: blur(15px); box-shadow: var(--shadow-card); cursor: pointer; display: flex; align-items: center; justify-content: center; font-size: 1.35rem; color: var(--text-primary); transition: var(--transition); z-index: 1100; }
    .theme-toggle:hover { transform: scale(1.15) rotate(180deg); }

    .container { max-width: 1100px; margin: 0 auto; padding: 2rem; z-index: 1; position: relative; }
    .header { text-align: center; margin-bottom: 2rem; position: relative; padding-top: 1.2rem; }
    .header-badge { display: inline-flex; align-items: center; gap: 0.6rem; background: var(--bg-secondary); border: 1px solid var(--border-color); border-radius: 999px; padding: 0.7rem 1rem; backdrop-filter: blur(15px); box-shadow: var(--shadow-card); margin-bottom: 1rem; color: var(--text-secondary); font-size: 0.95rem; }

    .main-title { font-size: clamp(2.2rem, 5vw, 3.8rem); font-weight: 800; background: var(--accent-gradient); -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text; margin-bottom: 1rem; line-height: 1.15; }
    .subtitle { color: var(--text-secondary); max-width: 900px; margin: 0 auto; font-size: 1.03rem; }

    .content-block, .quick-nav, .control-bar, .topic-card { background: var(--bg-secondary); backdrop-filter: blur(20px); border-radius: var(--border-radius); box-shadow: var(--shadow-card); border: 2px solid var(--border-color); }
    .content-block { padding: 2rem; margin-top: 2rem; }
    .content-block h2 { color: var(--text-primary); margin-bottom: 1rem; font-size: 1.55rem; }
    .content-block p, .content-block li, .subtitle { color: var(--text-secondary); }

    .quick-nav { margin-top: 2rem; padding: 1.4rem; }
    .quick-nav h3 { margin-bottom: 1rem; color: var(--text-primary); font-size: 1.2rem; }
    .nav-grid { display: flex; flex-wrap: wrap; gap: 0.75rem; }
    .nav-chip { padding: 0.65rem 0.95rem; border-radius: 999px; background: var(--nav-chip-bg); color: var(--text-primary); border: 1px solid var(--border-color); font-size: 0.92rem; transition: var(--transition); box-shadow: 0 8px 18px rgba(0,0,0,0.06); cursor: pointer; font-family: inherit; text-align: left; }
    .nav-chip:hover { transform: translateY(-2px); background: var(--bg-tertiary); }

    .control-bar { margin-top: 1.5rem; padding: 1rem 1.2rem; display: flex; gap: 0.8rem; flex-wrap: wrap; justify-content: center; }
    .control-btn { border: 1px solid var(--border-color); background: var(--nav-chip-bg); color: var(--text-primary); border-radius: 999px; padding: 0.75rem 1rem; cursor: pointer; font-family: inherit; font-size: 0.95rem; transition: var(--transition); }
    .control-btn:hover { transform: translateY(-2px); background: var(--bg-tertiary); }

    .lesson-container { display: flex; flex-direction: column; gap: 1.3rem; margin-top: 1.6rem; }
    .topic-card { overflow: hidden; transition: var(--transition); scroll-margin-top: 90px; }
    .topic-card:hover { transform: translateY(-2px); }
    .topic-header { cursor: pointer; padding: 1.5rem 2rem; display: flex; justify-content: space-between; align-items: center; gap: 1rem; user-select: none; }
    .topic-title { font-size: 1.22rem; font-weight: 700; color: var(--text-primary); line-height: 1.4; }
    .expand-icon { font-size: 1.15rem; color: var(--text-secondary); transition: var(--transition); flex-shrink: 0; }
    .topic-card.open .expand-icon { transform: rotate(180deg); }

    .topic-content { max-height: 0; overflow: hidden; transition: max-height 1.2s ease, padding 1.2s ease; background: var(--bg-tertiary); }
    .topic-card.open .topic-content { max-height: 6500px; padding: 1.6rem 2rem 1.8rem 2rem; border-top: 1px solid var(--border-color); }
    .topic-content h4 { color: var(--text-primary); margin-top: 1.5rem; margin-bottom: 0.55rem; font-size: 1.08rem; border-left: 4px solid var(--accent-primary); padding-left: 12px; }
    .topic-content h4:first-child { margin-top: 0; }
    .topic-content p, .topic-content li { color: var(--text-secondary); line-height: 1.8; margin-bottom: 0.95rem; }
    .topic-content strong { color: var(--text-primary); font-weight: 700; }

    .topic-content blockquote { text-align: center; border-left: 4px solid var(--accent-primary); margin: 1.4rem 0; font-style: normal; color: var(--text-secondary); background: rgba(127, 140, 141, 0.05); border-radius: 0 10px 10px 0; padding: 1rem 1.5rem; overflow-x: auto; }

    footer { text-align: center; margin-top: 4rem; padding-top: 2rem; border-top: 1px solid var(--border-color); }
    footer p { color: var(--text-secondary); font-size: 0.95rem; opacity: 0.9; }

    @media (max-width: 768px) {
      .container { padding: 1rem; }
      .topic-header { padding: 1.1rem 1.3rem; }
      .topic-card.open .topic-content { padding: 1.1rem 1.3rem 1.3rem 1.3rem; }
      .theme-toggle { width: 54px; height: 54px; top: 1rem; right: 1rem; }
    }
  </style>
</head>

<body data-theme="dark">
  <div class="progress-wrapper">
    <div class="progress-bar" id="progressBar"></div>
  </div>

  <div class="particles" id="particles-container"></div>

  <div class="theme-toggle" id="themeToggleButton" title="Cambiar tema">
    <i class="fas fa-moon" id="theme-icon"></i>
  </div>

  <div class="container">
    <header class="header">
      <div class="header-badge">
        <i class="fas fa-project-diagram"></i>
        <span>Estadística Aplicada a Data Science · Repositorio</span>
      </div>
      <h1 class="main-title">{{MAIN_TITLE}}</h1>
      <p class="subtitle">
        Visualizando la abstracción probabilística: distancias funcionales, intervalos de confianza locales y la matriz de Gram como lente geométrica.
      </p>
    </header>

    {{INTRO_HTML}}

    <section class="quick-nav">
      <h3>Navegación rápida</h3>
      <div class="nav-grid">
        {{NAV_HTML}}
      </div>
    </section>

    <section class="control-bar">
      <button class="control-btn" id="expandAllBtn" type="button"><i class="fas fa-plus-circle"></i> Expandir todo</button>
      <button class="control-btn" id="collapseAllBtn" type="button"><i class="fas fa-minus-circle"></i> Contraer todo</button>
      <button class="control-btn" id="topBtn" type="button"><i class="fas fa-arrow-up"></i> Volver arriba</button>
    </section>

    <div class="lesson-container">
      {{DYNAMIC_HTML}}
    </div>

    <footer>
      <p>{{FOOTER_TEXT}}</p>
    </footer>
  </div>

  <script>
    (function() {
      const themeToggleButton = document.getElementById('themeToggleButton');
      const themeIcon = document.getElementById('theme-icon');
      const bodyEl = document.body;
      const expandAllBtn = document.getElementById('expandAllBtn');
      const collapseAllBtn = document.getElementById('collapseAllBtn');
      const topBtn = document.getElementById('topBtn');
      const progressBar = document.getElementById('progressBar');

      function setTheme(theme) {
        bodyEl.setAttribute('data-theme', theme);
        try { localStorage.setItem('gp_theme', theme); } catch (e) {}
        if (themeIcon) { themeIcon.className = theme === 'dark' ? 'fas fa-sun' : 'fas fa-moon'; }
      }

      if (themeToggleButton) {
        themeToggleButton.addEventListener('click', () => {
          const newTheme = (bodyEl.getAttribute('data-theme') || 'dark') === 'dark' ? 'light' : 'dark';
          setTheme(newTheme);
        });
      }

      let preferredTheme = 'dark';
      try { preferredTheme = localStorage.getItem('gp_theme') || 'dark'; } catch (e) {}
      setTheme(preferredTheme);

      const cards = Array.from(document.querySelectorAll('.topic-card'));

      cards.forEach(card => {
        const header = card.querySelector('.topic-header');
        if (header) {
          header.addEventListener('click', () => {
            card.classList.toggle('open');
          });
        }
      });

      if (expandAllBtn) { expandAllBtn.addEventListener('click', () => { cards.forEach(card => card.classList.add('open')); }); }
      if (collapseAllBtn) { collapseAllBtn.addEventListener('click', () => { cards.forEach(card => card.classList.remove('open')); }); }
      if (topBtn) { topBtn.addEventListener('click', () => { window.scrollTo({ top: 0, behavior: 'smooth' }); }); }

      const navButtons = Array.from(document.querySelectorAll('.nav-chip'));
      navButtons.forEach(btn => {
        btn.addEventListener('click', () => {
          const targetId = btn.getAttribute('data-target');
          const target = document.getElementById(targetId);
          if (target) {
            target.classList.add('open');
            setTimeout(() => { target.scrollIntoView({ behavior: 'smooth', block: 'start' }); }, 120);
          }
        });
      });

      const firstTopic = document.querySelector('.topic-card');
      if (firstTopic) { firstTopic.classList.add('open'); }

      const container = document.getElementById('particles-container');
      if (container) {
        for (let i = 0; i < 34; i++) {
          const p = document.createElement('div');
          p.className = 'particle';
          p.style.left = Math.random() * 100 + 'vw';
          const size = (Math.random() * 6 + 3);
          p.style.width = size + 'px';
          p.style.height = size + 'px';
          p.style.animationDelay = Math.random() * -20 + 's';
          p.style.animationDuration = (15 + Math.random() * 12) + 's';
          container.appendChild(p);
        }
      }

      function updateProgress() {
        const scrollTop = document.documentElement.scrollTop || document.body.scrollTop;
        const scrollHeight = document.documentElement.scrollHeight - document.documentElement.clientHeight;
        const progress = scrollHeight > 0 ? (scrollTop / scrollHeight) * 100 : 0;
        progressBar.style.width = progress + '%';
      }

      window.addEventListener('scroll', updateProgress);
      updateProgress();

      if (window.MathJax) {
          MathJax.Hub.Queue(["Typeset", MathJax.Hub]);
      }
    })();
  </script>
</body>
</html>
"""

# =========================================================
# 5. ENSAMBLADO
# =========================================================

final_html = (
    plantilla_profesional
    .replace("{{MAIN_TITLE}}", "5. Geometría e interpretación probabilística")
    .replace("{{INTRO_HTML}}", intro_html_content)
    .replace("{{NAV_HTML}}", navegacion_html)
    .replace("{{DYNAMIC_HTML}}", contenido_dinamico_html)
    .replace("{{FOOTER_TEXT}}", "Material elaborado por el profesor Sergio Gevatschnaider.")
)

display(HTML(final_html))

In [ ]:
# @title Índice Interactivo: 6. Condicionamiento y regresión con GP
from IPython.display import display, HTML
import html

# =========================================================
# 1. INTRODUCCIÓN
# =========================================================

intro_html_content = r"""
<div class="content-block" style="margin-top:0;">
    <h2>Resumen de la Sección</h2>
    <p>
        La utilidad más importante de los Procesos Gaussianos aparece cuando se los usa para <strong>aprender una función a partir de datos</strong>.
        Hasta ahora, hemos visto al GP como una distribución previa sobre funciones posibles. Sin embargo, el objetivo real de la regresión
        es actualizar esa distribución cuando se observan pares de datos empíricos.
    </p>
    <p>
        En esta sección estudiaremos el paso central de esta teoría: <strong>el condicionamiento</strong>. Veremos cómo se formula el modelo
        observacional con ruido, cómo se construye la distribución conjunta entre puntos observados y no observados, y derivaremos las
        fórmulas exactas para la media y la covarianza posterior, que constituyen el corazón de la regresión probabilística.
    </p>
</div>
"""

# =========================================================
# 2. SECCIONES (Texto estructurado y LaTeX corregido)
# =========================================================

secciones_principales_data = [
    {
        "id": "sec-1",
        "titulo": "1. El modelo observacional y la función latente",
        "contenido": r"""
            <h4>Datos ruidosos y función desconocida</h4>
            <p>
                Supongamos que existe una función desconocida $f: \mathcal{X} \to \mathbb{R}$ y que se observan datos de la forma:
            </p>
            <blockquote>$$y_i = f(x_i) + \varepsilon_i, \qquad i=1,\dots,n,$$</blockquote>
            <p>
                donde $x_1,\dots,x_n$ son entradas observadas y $\varepsilon_1,\dots,\varepsilon_n$ representan ruido de medición o error aleatorio.
                La hipótesis estándar es que estos errores satisfacen:
            </p>
            <blockquote>$$\varepsilon_i \sim \mathcal{N}(0,\sigma_n^2),$$</blockquote>
            <p>
                son independientes entre sí y además independientes del proceso $f$. El parámetro $\sigma_n^2$ representa la varianza del ruido observacional.
                Esta suposición distingue entre la <strong>función latente</strong> $f$, que queremos aprender, y los datos observados $y_i$, que son versiones perturbadas.
            </p>
            <h4>La distribución previa</h4>
            <p>
                El modelo completo se construye imponiendo un proceso gaussiano como distribución previa sobre la función latente:
            </p>
            <blockquote>$$f \sim \mathcal{GP}(m,k).$$</blockquote>
            <p>
                El problema de la regresión consiste en pasar de esta distribución previa a una distribución posterior sobre $f$, una vez que se conocen los datos empíricos.
            </p>
        """
    },
    {
        "id": "sec-2",
        "titulo": "2. Formulación matricial y distribución marginal",
        "contenido": r"""
            <h4>Vectores y Matrices del Modelo</h4>
            <p>
                Introducimos una notación matricial compacta. Definimos el conjunto de entradas observadas como $X = (x_1,\dots,x_n)$, el vector
                de observaciones $y$, y el vector de valores latentes $f(X)$:
            </p>
            <blockquote>$$y = \begin{pmatrix} y_1 \\ y_2 \\ \vdots \\ y_n \end{pmatrix}, \qquad f(X) = \begin{pmatrix} f(x_1) \\ f(x_2) \\ \vdots \\ f(x_n) \end{pmatrix}.$$</blockquote>
            <p>
                La función media evaluada en los puntos observados es $m(X)$, y la matriz de Gram sobre el conjunto $X$ será:
            </p>
            <blockquote>$$K(X,X) = \begin{pmatrix} k(x_1,x_1) & \cdots & k(x_1,x_n) \\ \vdots & \ddots & \vdots \\ k(x_n,x_1) & \cdots & k(x_n,x_n) \end{pmatrix}.$$</blockquote>
            <p>
                Como el proceso es gaussiano, el vector latente tiene distribución $f(X) \sim \mathcal{N}(m(X), K(X,X))$.
            </p>
            <h4>Distribución marginal de los datos</h4>
            <p>
                El modelo de observación puede escribirse en forma vectorial como $y = f(X) + \varepsilon$, donde $\varepsilon \sim \mathcal{N}(0,\sigma_n^2 I_n)$.
                Dado que la suma de variables gaussianas independientes sigue siendo gaussiana, se deduce que el vector de observaciones $y$ tiene distribución marginal:
            </p>
            <blockquote>$$y \sim \mathcal{N}\left(m(X), K(X,X) + \sigma_n^2 I_n\right).$$</blockquote>
            <p>
                Esta fórmula muestra que el ruido observacional no cambia la media, pero incrementa la covarianza en la diagonal, reflejando la variabilidad adicional introducida por la medición.
            </p>
        """
    },
    {
        "id": "sec-3",
        "titulo": "3. Distribución conjunta entre observados y no observados",
        "contenido": r"""
            <h4>Puntos de prueba (Test points)</h4>
            <p>
                El objetivo de la regresión es predecir el valor de la función en nuevos puntos no observados. Introducimos un conjunto de puntos de prueba
                $X_* = (x_1^*,\dots,x_m^*)$ y definimos el vector de valores latentes correspondientes $f_*$.
            </p>
            <p>
                También definimos sus respectivas medias $m(X_*)$, su matriz de covarianza $K(X_*,X_*)$, y la matriz de covarianza cruzada entre
                puntos de prueba y puntos observados:
            </p>
            <blockquote>$$K(X_*,X) = \begin{pmatrix} k(x_1^*,x_1) & \cdots & k(x_1^*,x_n) \\ \vdots & \ddots & \vdots \\ k(x_m^*,x_1) & \cdots & k(x_m^*,x_n) \end{pmatrix}.$$</blockquote>
            <p>Por simetría del kernel, se tiene $K(X,X_*) = K(X_*,X)^T$.</p>
            <h4>La Ley Conjunta</h4>
            <p>
                La pieza central de la teoría es la distribución conjunta entre observados y no observados. El vector conjunto formado por las observaciones $y$
                y los valores latentes futuros $f_*$ es también normal multivariante:
            </p>
            <blockquote>
                $$\begin{pmatrix} y \\ f_* \end{pmatrix} \sim \mathcal{N} \left( \begin{pmatrix} m(X) \\ m(X_*) \end{pmatrix}, \begin{pmatrix} K(X,X)+\sigma_n^2 I_n & K(X,X_*) \\ K(X_*,X) & K(X_*,X_*) \end{pmatrix} \right).$$
            </blockquote>
            <p>
                Esta expresión condensa toda la regresión. El bloque superior describe las observaciones ruidosas, el bloque inferior la covarianza previa de los valores de prueba,
                y los bloques cruzados cómo se relacionan los puntos observados con los no observados.
            </p>
        """
    },
    {
        "id": "sec-4",
        "titulo": "4. Condicionamiento: Media y Covarianza Posterior",
        "contenido": r"""
            <h4>La fórmula del condicionamiento gaussiano</h4>
            <p>
                Entra en juego una propiedad clásica: la distribución condicional de una parte de un vector gaussiano, dada la otra, sigue siendo normal.
                Si $\begin{pmatrix} u \\ v \end{pmatrix} \sim \mathcal{N} \left( \begin{pmatrix} \mu_u \\ \mu_v \end{pmatrix}, \begin{pmatrix} \Sigma_{uu} & \Sigma_{uv} \\ \Sigma_{vu} & \Sigma_{vv} \end{pmatrix} \right)$, entonces:
            </p>
            <blockquote>$$v \mid u \sim \mathcal{N}\left( \mu_v + \Sigma_{vu}\Sigma_{uu}^{-1}(u-\mu_u), \, \Sigma_{vv}-\Sigma_{vu}\Sigma_{uu}^{-1}\Sigma_{uv} \right).$$</blockquote>
            <h4>El corazón de la regresión con GP</h4>
            <p>
                Aplicando esta fórmula ($u=y$ y $v=f_*$), obtenemos la distribución posterior de los valores latentes en los puntos de prueba:
            </p>
            <blockquote>$$f_* \mid X,y,X_* \sim \mathcal{N}\left(m_{\mathrm{post}}(X_*), \, \Sigma_{\mathrm{post}}(X_*)\right),$$</blockquote>
            <p>donde la <strong>media posterior</strong> es:</p>
            <blockquote>$$m_{\mathrm{post}}(X_*) = m(X_*) + K(X_*,X)\left(K(X,X)+\sigma_n^2 I_n\right)^{-1}\left(y-m(X)\right),$$</blockquote>
            <p>y la <strong>covarianza posterior</strong> es:</p>
            <blockquote>$$\Sigma_{\mathrm{post}}(X_*) = K(X_*,X_*) - K(X_*,X)\left(K(X,X)+\sigma_n^2 I_n\right)^{-1}K(X,X_*).$$</blockquote>
            <p>
                La media posterior corrige la media previa propagando los residuos $(y-m(X))$ mediante el kernel. La covarianza posterior resta
                la información aportada por los datos observados a la incertidumbre previa, expresando una idea fundamental: <em>observar datos reduce incertidumbre</em>.
            </p>
        """
    },
    {
        "id": "sec-5",
        "titulo": "5. Interpolación probabilística y el límite sin ruido",
        "contenido": r"""
            <h4>Predicción puntual y observaciones futuras</h4>
            <p>
                Para un único punto de prueba $x^*$, el valor latente $f(x^*)$ tiene distribución normal univariada
                $f(x^*) \mid X,y,x^* \sim \mathcal{N}\left(m_{\mathrm{post}}(x^*), s_{\mathrm{post}}^2(x^*)\right)$.
            </p>
            <p>
                Si lo que se quiere predecir es una <em>nueva observación ruidosa</em> $y^* = f(x^*) + \varepsilon^*$, la distribución predictiva
                tiene la misma media, pero una varianza que incluye el ruido de medición:
            </p>
            <blockquote>$$y^* \mid X,y,x^* \sim \mathcal{N}\left(m_{\mathrm{post}}(x^*), \, s_{\mathrm{post}}^2(x^*) + \sigma_n^2\right).$$</blockquote>
            <h4>El límite sin ruido ($\sigma_n^2 \to 0$)</h4>
            <p>
                Un caso instructivo es cuando las observaciones no tienen ruido. En ese caso, la media posterior tiende a interpolar exactamente
                los datos empíricos ($y_i \approx f(x_i)$), y la varianza posterior en los puntos observados tiende a cero.
                Cuando $\sigma_n^2 > 0$, el modelo ya no interpola exactamente, sino que suaviza las observaciones.
            </p>
            <h4>Aprender una distribución sobre funciones</h4>
            <p>
                El término $\left(K(X,X)+\sigma_n^2 I_n\right)^{-1}\left(y-m(X)\right)$ actúa como un conjunto de pesos. La predicción es una combinación
                lineal de las observaciones, ponderada por cuán similares son los puntos nuevos a los antiguos según el kernel.
            </p>
            <p>
                En síntesis, este aprendizaje no ajusta una curva única. La media posterior resume la función más plausible en promedio; la covarianza
                posterior resume la incertidumbre residual. El proceso no responde "la función es esta", sino "estas son las funciones compatibles con la evidencia".
            </p>
        """
    }
]

# =========================================================
# 3. GENERACIÓN HTML
# =========================================================

def generar_tarjetas_acordeon(datos):
    bloques = []
    for seccion in datos:
        bloques.append(f"""
        <div class="topic-card" id="{html.escape(seccion['id'])}">
            <div class="topic-header">
                <span class="topic-title">{seccion['titulo']}</span>
                <i class="fas fa-chevron-down expand-icon"></i>
            </div>
            <div class="topic-content">
                {seccion['contenido']}
            </div>
        </div>
        """)
    return "\n".join(bloques)

def generar_navegacion(datos):
    chips = []
    for seccion in datos:
        chips.append(
            f'<button class="nav-chip" type="button" data-target="{html.escape(seccion["id"])}">{seccion["titulo"]}</button>'
        )
    return "\n".join(chips)

contenido_dinamico_html = generar_tarjetas_acordeon(secciones_principales_data)
navegacion_html = generar_navegacion(secciones_principales_data)

# =========================================================
# 4. PLANTILLA HTML
# =========================================================

plantilla_profesional = r"""
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{{MAIN_TITLE}}</title>

  <script type="text/x-mathjax-config">
    MathJax.Hub.Config({
      tex2jax: {
        inlineMath: [['$','$']],
        displayMath: [['$$','$$']],
        processEscapes: true
      }
    });
  </script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-MML-AM_CHTML" async></script>

  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
  <link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css" rel="stylesheet">

  <style>
    :root {
      --bg-primary: linear-gradient(135deg, #f0f9ff 0%, #e0f2fe 100%);
      --bg-secondary: rgba(255, 255, 255, 0.85);
      --bg-tertiary: rgba(248, 250, 252, 0.8);
      --text-primary: #0c4a6e;
      --text-secondary: #075985;
      --accent-primary: #0ea5e9;
      --accent-secondary: #38bdf8;
      --accent-gradient: linear-gradient(135deg, var(--accent-primary) 0%, var(--accent-secondary) 100%);
      --border-color: rgba(226, 232, 240, 0.8);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.08);
      --border-radius: 20px;
      --transition: all 0.4s cubic-bezier(0.25, 0.8, 0.25, 1);
      --nav-chip-bg: rgba(255,255,255,0.8);
      --progress-bg: rgba(255,255,255,0.45);
    }

    [data-theme="dark"] {
      --bg-primary: linear-gradient(135deg, #0c4a6e 0%, #1e3a8a 100%);
      --bg-secondary: rgba(30, 58, 138, 0.85);
      --bg-tertiary: rgba(30, 64, 175, 0.7);
      --text-primary: #f0f9ff;
      --text-secondary: #dbeafe;
      --accent-primary: #38bdf8;
      --accent-secondary: #7dd3fc;
      --border-color: rgba(255, 255, 255, 0.15);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.20);
      --nav-chip-bg: rgba(30, 58, 138, 0.65);
      --progress-bg: rgba(255,255,255,0.15);
    }

    * { margin: 0; padding: 0; box-sizing: border-box; }
    html { scroll-behavior: smooth; }
    body { font-family: 'Inter', sans-serif; line-height: 1.8; background: var(--bg-primary); color: var(--text-primary); transition: var(--transition); min-height: 100vh; position: relative; overflow-x: hidden; }

    .particles { position: fixed; top: 0; left: 0; width: 100%; height: 100%; pointer-events: none; z-index: -1; }
    .particle { position: absolute; border-radius: 50%; animation: float 25s infinite linear; opacity: 0; background: rgba(255, 255, 255, 0.55); box-shadow: 0 0 15px rgba(255,255,255,0.25); }
    @keyframes float { 0% { transform: translateY(100vh) rotate(0deg); opacity: 0; } 10%, 90% { opacity: 0.55; } 100% { transform: translateY(-10vh) rotate(360deg); opacity: 0; } }

    .progress-wrapper { position: fixed; top: 0; left: 0; width: 100%; height: 6px; background: var(--progress-bg); z-index: 1200; backdrop-filter: blur(10px); }
    .progress-bar { height: 100%; width: 0%; background: var(--accent-gradient); transition: width 0.15s ease; }

    .theme-toggle { position: fixed; top: 1.2rem; right: 1.2rem; width: 60px; height: 60px; border: 1px solid var(--border-color); border-radius: 50%; background: var(--bg-secondary); backdrop-filter: blur(15px); box-shadow: var(--shadow-card); cursor: pointer; display: flex; align-items: center; justify-content: center; font-size: 1.35rem; color: var(--text-primary); transition: var(--transition); z-index: 1100; }
    .theme-toggle:hover { transform: scale(1.15) rotate(180deg); }

    .container { max-width: 1100px; margin: 0 auto; padding: 2rem; z-index: 1; position: relative; }
    .header { text-align: center; margin-bottom: 2rem; position: relative; padding-top: 1.2rem; }
    .header-badge { display: inline-flex; align-items: center; gap: 0.6rem; background: var(--bg-secondary); border: 1px solid var(--border-color); border-radius: 999px; padding: 0.7rem 1rem; backdrop-filter: blur(15px); box-shadow: var(--shadow-card); margin-bottom: 1rem; color: var(--text-secondary); font-size: 0.95rem; }

    .main-title { font-size: clamp(2.2rem, 5vw, 3.8rem); font-weight: 800; background: var(--accent-gradient); -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text; margin-bottom: 1rem; line-height: 1.15; }
    .subtitle { color: var(--text-secondary); max-width: 900px; margin: 0 auto; font-size: 1.03rem; }

    .content-block, .quick-nav, .control-bar, .topic-card { background: var(--bg-secondary); backdrop-filter: blur(20px); border-radius: var(--border-radius); box-shadow: var(--shadow-card); border: 2px solid var(--border-color); }
    .content-block { padding: 2rem; margin-top: 2rem; }
    .content-block h2 { color: var(--text-primary); margin-bottom: 1rem; font-size: 1.55rem; }
    .content-block p, .content-block li, .subtitle { color: var(--text-secondary); }

    .quick-nav { margin-top: 2rem; padding: 1.4rem; }
    .quick-nav h3 { margin-bottom: 1rem; color: var(--text-primary); font-size: 1.2rem; }
    .nav-grid { display: flex; flex-wrap: wrap; gap: 0.75rem; }
    .nav-chip { padding: 0.65rem 0.95rem; border-radius: 999px; background: var(--nav-chip-bg); color: var(--text-primary); border: 1px solid var(--border-color); font-size: 0.92rem; transition: var(--transition); box-shadow: 0 8px 18px rgba(0,0,0,0.06); cursor: pointer; font-family: inherit; text-align: left; }
    .nav-chip:hover { transform: translateY(-2px); background: var(--bg-tertiary); }

    .control-bar { margin-top: 1.5rem; padding: 1rem 1.2rem; display: flex; gap: 0.8rem; flex-wrap: wrap; justify-content: center; }
    .control-btn { border: 1px solid var(--border-color); background: var(--nav-chip-bg); color: var(--text-primary); border-radius: 999px; padding: 0.75rem 1rem; cursor: pointer; font-family: inherit; font-size: 0.95rem; transition: var(--transition); }
    .control-btn:hover { transform: translateY(-2px); background: var(--bg-tertiary); }

    .lesson-container { display: flex; flex-direction: column; gap: 1.3rem; margin-top: 1.6rem; }
    .topic-card { overflow: hidden; transition: var(--transition); scroll-margin-top: 90px; }
    .topic-card:hover { transform: translateY(-2px); }
    .topic-header { cursor: pointer; padding: 1.5rem 2rem; display: flex; justify-content: space-between; align-items: center; gap: 1rem; user-select: none; }
    .topic-title { font-size: 1.22rem; font-weight: 700; color: var(--text-primary); line-height: 1.4; }
    .expand-icon { font-size: 1.15rem; color: var(--text-secondary); transition: var(--transition); flex-shrink: 0; }
    .topic-card.open .expand-icon { transform: rotate(180deg); }

    .topic-content { max-height: 0; overflow: hidden; transition: max-height 1.2s ease, padding 1.2s ease; background: var(--bg-tertiary); }
    .topic-card.open .topic-content { max-height: 6500px; padding: 1.6rem 2rem 1.8rem 2rem; border-top: 1px solid var(--border-color); }
    .topic-content h4 { color: var(--text-primary); margin-top: 1.5rem; margin-bottom: 0.55rem; font-size: 1.08rem; border-left: 4px solid var(--accent-primary); padding-left: 12px; }
    .topic-content h4:first-child { margin-top: 0; }
    .topic-content p, .topic-content li { color: var(--text-secondary); line-height: 1.8; margin-bottom: 0.95rem; }
    .topic-content strong { color: var(--text-primary); font-weight: 700; }

    .topic-content blockquote { text-align: center; border-left: 4px solid var(--accent-primary); margin: 1.4rem 0; font-style: normal; color: var(--text-secondary); background: rgba(127, 140, 141, 0.05); border-radius: 0 10px 10px 0; padding: 1rem 1.5rem; overflow-x: auto; }

    footer { text-align: center; margin-top: 4rem; padding-top: 2rem; border-top: 1px solid var(--border-color); }
    footer p { color: var(--text-secondary); font-size: 0.95rem; opacity: 0.9; }

    @media (max-width: 768px) {
      .container { padding: 1rem; }
      .topic-header { padding: 1.1rem 1.3rem; }
      .topic-card.open .topic-content { padding: 1.1rem 1.3rem 1.3rem 1.3rem; }
      .theme-toggle { width: 54px; height: 54px; top: 1rem; right: 1rem; }
    }
  </style>
</head>

<body data-theme="dark">
  <div class="progress-wrapper">
    <div class="progress-bar" id="progressBar"></div>
  </div>

  <div class="particles" id="particles-container"></div>

  <div class="theme-toggle" id="themeToggleButton" title="Cambiar tema">
    <i class="fas fa-moon" id="theme-icon"></i>
  </div>

  <div class="container">
    <header class="header">
      <div class="header-badge">
        <i class="fas fa-project-diagram"></i>
        <span>Estadística Aplicada a Data Science · Repositorio</span>
      </div>
      <h1 class="main-title">{{MAIN_TITLE}}</h1>
      <p class="subtitle">
        Derivación exacta de la distribución conjunta, el condicionamiento multivariante y la actualización probabilística frente a la observación empírica.
      </p>
    </header>

    {{INTRO_HTML}}

    <section class="quick-nav">
      <h3>Navegación rápida</h3>
      <div class="nav-grid">
        {{NAV_HTML}}
      </div>
    </section>

    <section class="control-bar">
      <button class="control-btn" id="expandAllBtn" type="button"><i class="fas fa-plus-circle"></i> Expandir todo</button>
      <button class="control-btn" id="collapseAllBtn" type="button"><i class="fas fa-minus-circle"></i> Contraer todo</button>
      <button class="control-btn" id="topBtn" type="button"><i class="fas fa-arrow-up"></i> Volver arriba</button>
    </section>

    <div class="lesson-container">
      {{DYNAMIC_HTML}}
    </div>

    <footer>
      <p>{{FOOTER_TEXT}}</p>
    </footer>
  </div>

  <script>
    (function() {
      const themeToggleButton = document.getElementById('themeToggleButton');
      const themeIcon = document.getElementById('theme-icon');
      const bodyEl = document.body;
      const expandAllBtn = document.getElementById('expandAllBtn');
      const collapseAllBtn = document.getElementById('collapseAllBtn');
      const topBtn = document.getElementById('topBtn');
      const progressBar = document.getElementById('progressBar');

      function setTheme(theme) {
        bodyEl.setAttribute('data-theme', theme);
        try { localStorage.setItem('gp_theme', theme); } catch (e) {}
        if (themeIcon) { themeIcon.className = theme === 'dark' ? 'fas fa-sun' : 'fas fa-moon'; }
      }

      if (themeToggleButton) {
        themeToggleButton.addEventListener('click', () => {
          const newTheme = (bodyEl.getAttribute('data-theme') || 'dark') === 'dark' ? 'light' : 'dark';
          setTheme(newTheme);
        });
      }

      let preferredTheme = 'dark';
      try { preferredTheme = localStorage.getItem('gp_theme') || 'dark'; } catch (e) {}
      setTheme(preferredTheme);

      const cards = Array.from(document.querySelectorAll('.topic-card'));

      cards.forEach(card => {
        const header = card.querySelector('.topic-header');
        if (header) {
          header.addEventListener('click', () => {
            card.classList.toggle('open');
          });
        }
      });

      if (expandAllBtn) { expandAllBtn.addEventListener('click', () => { cards.forEach(card => card.classList.add('open')); }); }
      if (collapseAllBtn) { collapseAllBtn.addEventListener('click', () => { cards.forEach(card => card.classList.remove('open')); }); }
      if (topBtn) { topBtn.addEventListener('click', () => { window.scrollTo({ top: 0, behavior: 'smooth' }); }); }

      const navButtons = Array.from(document.querySelectorAll('.nav-chip'));
      navButtons.forEach(btn => {
        btn.addEventListener('click', () => {
          const targetId = btn.getAttribute('data-target');
          const target = document.getElementById(targetId);
          if (target) {
            target.classList.add('open');
            setTimeout(() => { target.scrollIntoView({ behavior: 'smooth', block: 'start' }); }, 120);
          }
        });
      });

      const firstTopic = document.querySelector('.topic-card');
      if (firstTopic) { firstTopic.classList.add('open'); }

      const container = document.getElementById('particles-container');
      if (container) {
        for (let i = 0; i < 34; i++) {
          const p = document.createElement('div');
          p.className = 'particle';
          p.style.left = Math.random() * 100 + 'vw';
          const size = (Math.random() * 6 + 3);
          p.style.width = size + 'px';
          p.style.height = size + 'px';
          p.style.animationDelay = Math.random() * -20 + 's';
          p.style.animationDuration = (15 + Math.random() * 12) + 's';
          container.appendChild(p);
        }
      }

      function updateProgress() {
        const scrollTop = document.documentElement.scrollTop || document.body.scrollTop;
        const scrollHeight = document.documentElement.scrollHeight - document.documentElement.clientHeight;
        const progress = scrollHeight > 0 ? (scrollTop / scrollHeight) * 100 : 0;
        progressBar.style.width = progress + '%';
      }

      window.addEventListener('scroll', updateProgress);
      updateProgress();

      if (window.MathJax) {
          MathJax.Hub.Queue(["Typeset", MathJax.Hub]);
      }
    })();
  </script>
</body>
</html>
"""

# =========================================================
# 5. ENSAMBLADO
# =========================================================

final_html = (
    plantilla_profesional
    .replace("{{MAIN_TITLE}}", "6. Condicionamiento y regresión con GP")
    .replace("{{INTRO_HTML}}", intro_html_content)
    .replace("{{NAV_HTML}}", navegacion_html)
    .replace("{{DYNAMIC_HTML}}", contenido_dinamico_html)
    .replace("{{FOOTER_TEXT}}", "Material elaborado por el profesor Sergio Gevatschnaider.")
)

display(HTML(final_html))

In [ ]:
# @title Índice Interactivo: 7. Selección de hiperparámetros
from IPython.display import display, HTML
import html

# =========================================================
# 1. INTRODUCCIÓN
# =========================================================

intro_html_content = r"""
<div class="content-block" style="margin-top:0;">
    <h2>Resumen de la Sección</h2>
    <p>
        En la regresión con procesos gaussianos, el kernel determina la estructura de covarianza del modelo y, con ello, la forma de las
        funciones que el proceso considera plausibles antes de observar datos. Sin embargo, un kernel no suele venir completamente fijado:
        contiene parámetros que regulan la escala, la suavidad, la amplitud y el ruido del problema.
    </p>
    <p>
        A esos parámetros se los llama <strong>hiperparámetros</strong>. La selección de hiperparámetros es una etapa central del modelado
        con procesos gaussianos, porque de ella depende si el modelo aprenderá una función razonablemente suave, si interpretará correctamente
        la variabilidad observada y si distinguirá adecuadamente entre estructura real y ruido.
    </p>
</div>
"""

# =========================================================
# 2. SECCIONES (Texto 100% completo y LaTeX corregido)
# =========================================================

secciones_principales_data = [
    {
        "id": "sec-1",
        "titulo": "1. Los hiperparámetros y el modelo observacional",
        "contenido": r"""
            <p>
                Para fijar ideas, conviene partir de un ejemplo concreto. Si se utiliza un kernel RBF o <em>squared exponential</em>, la función de covarianza suele escribirse como
            </p>
            <blockquote>$$k_\theta(x,x') = \sigma_f^2 \exp\left( -\frac{\|x-x'\|^2}{2\ell^2} \right),$$</blockquote>
            <p>
                donde $\theta=(\ell,\sigma_f^2)$ representa el conjunto de hiperparámetros del kernel. Si además las observaciones están contaminadas por ruido gaussiano independiente, el modelo observacional es
            </p>
            <blockquote>$$y_i = f(x_i) + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0,\sigma_n^2),$$</blockquote>
            <p>
                de modo que aparece un tercer hiperparámetro, la varianza del ruido $\sigma_n^2$. En consecuencia, incluso en el caso más simple, el modelo depende ya de tres cantidades fundamentales:
            </p>
            <blockquote>$$\theta=(\ell,\sigma_f^2,\sigma_n^2).$$</blockquote>
            <p>
                Estas cantidades no son parámetros de una trayectoria particular de la función, sino parámetros que controlan la distribución previa y la forma de la inferencia. Por eso se los llama hiperparámetros: están en un nivel superior respecto de los valores funcionales que el proceso quiere aprender.
            </p>
        """
    },
    {
        "id": "sec-2",
        "titulo": "2. La longitud de escala (\u2113)",
        "contenido": r"""
            <p>
                La primera cantidad clave es la longitud de escala, normalmente denotada por $\ell$. Este hiperparámetro controla cuánto se parecen los valores de la función en puntos cercanos del dominio. En el kernel RBF, la covarianza entre dos puntos $x$ y $x'$ es
            </p>
            <blockquote>$$k(x,x') = \sigma_f^2 \exp\left( -\frac{\|x-x'\|^2}{2\ell^2} \right).$$</blockquote>
            <p>
                Si $\ell$ es grande, el término $\frac{\|x-x'\|^2}{2\ell^2}$ crece lentamente, y por tanto la covarianza decrece despacio al aumentar la distancia entre los puntos. Esto significa que el proceso considera que valores bastante alejados del dominio siguen siendo semejantes. Como consecuencia, las funciones plausibles serán suaves, variarán lentamente y presentarán cambios graduales. En cambio, si $\ell$ es pequeño, la covarianza cae rápidamente incluso para separaciones modestas. El modelo considera entonces que la influencia de un punto observado se desvanece pronto y, por lo tanto, permite funciones que oscilan más, cambian más bruscamente y se adaptan con mayor flexibilidad a variaciones locales.
            </p>
            <p>
                La longitud de escala es, pues, un parámetro de suavidad horizontal. Determina cuán rápido cambia la función cuando uno se mueve en el espacio de entrada. Puede verse como la distancia característica a partir de la cual dos puntos dejan de ser fuertemente correlacionados. Si el dominio está en varias dimensiones, es frecuente incluso introducir una longitud de escala distinta por coordenada:
            </p>
            <blockquote>$$k(x,x') = \sigma_f^2 \exp\left( -\sum_{j=1}^{p}\frac{(x_j-x_j')^2}{2\ell_j^2} \right).$$</blockquote>
            <p>
                En este caso, cada $\ell_j$ mide cuán relevante es la coordenada $j$. Si $\ell_j$ es muy grande, la función cambia poco al moverse en esa dirección, lo que sugiere que esa variable influye débilmente. Si $\ell_j$ es pequeño, la función es sensible a variaciones en esa coordenada. Esta variante se conoce como <em>automatic relevance determination</em> y muestra que los hiperparámetros no solo regulan suavidad, sino que también pueden codificar relevancia de variables.
            </p>
        """
    },
    {
        "id": "sec-3",
        "titulo": "3. La varianza del proceso (\u03C3\u00B2_f)",
        "contenido": r"""
            <p>
                La segunda cantidad fundamental es la varianza del proceso, habitualmente denotada por $\sigma_f^2$. En el kernel RBF, así como en muchos otros kernels, esta cantidad aparece multiplicando toda la covarianza:
            </p>
            <blockquote>$$k(x,x') = \sigma_f^2 \,\kappa(x,x'),$$</blockquote>
            <p>
                donde $\kappa(x,x')$ contiene la forma geométrica y $\sigma_f^2$ fija la escala vertical. En particular, como
            </p>
            <blockquote>$$k(x,x) = \sigma_f^2,$$</blockquote>
            <p>
                cuando el kernel está normalizado, la varianza marginal del proceso en cada punto es $\sigma_f^2$. Esto significa que $\sigma_f^2$ controla la amplitud típica de las fluctuaciones de la función alrededor de su media. Si $\sigma_f^2$ es grande, el modelo permite funciones con oscilaciones de gran magnitud. Si $\sigma_f^2$ es pequeña, el proceso se concentra alrededor de la media previa y las trayectorias plausibles presentan poca variabilidad.
            </p>
            <p>
                La interpretación geométrica de $\sigma_f^2$ es clara. Mientras que $\ell$ regula la escala horizontal de variación, $\sigma_f^2$ regula la escala vertical. Una longitud de escala grande con una varianza del proceso grande produce funciones suaves pero de gran amplitud. Una longitud de escala grande con una varianza pequeña produce funciones suaves y cercanas a la media. Una longitud de escala pequeña con una varianza grande permite funciones muy flexibles y con oscilaciones pronunciadas. La interacción entre estos dos hiperparámetros define, en gran medida, la clase de trayectorias que el modelo considera plausibles.
            </p>
        """
    },
    {
        "id": "sec-4",
        "titulo": "4. La varianza del ruido (\u03C3\u00B2_n) y la covarianza marginal",
        "contenido": r"""
            <p>
                La tercera cantidad esencial es la varianza del ruido, denotada por $\sigma_n^2$. Esta aparece en el modelo de observación
            </p>
            <blockquote>$$y_i = f(x_i) + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0,\sigma_n^2).$$</blockquote>
            <p>
                La función latente $f$ representa la señal estructural que se desea aprender, mientras que $\varepsilon_i$ representa perturbaciones de medición, errores experimentales o fluctuaciones no explicadas por la estructura funcional principal. Si $\sigma_n^2$ es pequeño, el modelo interpreta que las observaciones son confiables y que la mayor parte de la variabilidad observada debe explicarse por la función latente. En ese caso, la regresión tenderá a seguir los datos de forma bastante cercana. Si $\sigma_n^2$ es grande, el modelo interpreta que los datos están muy contaminados por ruido y, por tanto, la función aprendida será más suave y menos sensible a fluctuaciones individuales.
            </p>
            <p>
                Desde el punto de vista matricial, el ruido afecta la covarianza de las observaciones mediante la fórmula
            </p>
            <blockquote>$$y \sim \mathcal{N}\left(m(X),\,K(X,X)+\sigma_n^2 I_n\right),$$</blockquote>
            <p>
                donde $K(X,X)$ es la matriz de Gram del kernel sobre los puntos observados. El término $\sigma_n^2 I_n$ añade una cantidad positiva a la diagonal, lo que refleja que cada observación tiene una fuente adicional de variabilidad independiente. Esta estructura también tiene un papel numérico importante, porque ayuda a estabilizar la inversión de la matriz de covarianza cuando $K(X,X)$ está cerca de ser singular.
            </p>
        """
    },
    {
        "id": "sec-5",
        "titulo": "5. Maximización del Log Marginal Likelihood",
        "contenido": r"""
            <p>
                Una vez identificados los hiperparámetros principales, la cuestión es cómo seleccionarlos. En el enfoque clásico de procesos gaussianos, esto se hace maximizando la verosimilitud marginal de los datos. La idea es la siguiente. Como el proceso previo y el ruido son gaussianos, el vector de observaciones $y$ tiene distribución marginal normal:
            </p>
            <blockquote>$$y \sim \mathcal{N}\left(m(X),\,C_\theta\right),$$</blockquote>
            <p>
                donde
            </p>
            <blockquote>$$C_\theta = K_\theta(X,X) + \sigma_n^2 I_n.$$</blockquote>
            <p>
                Aquí se ha subrayado con el subíndice $\theta$ que la matriz de covarianza depende de los hiperparámetros. La verosimilitud marginal es entonces
            </p>
            <blockquote>$$p(y \mid X,\theta) = \frac{1}{(2\pi)^{n/2}|C_\theta|^{1/2}} \exp\left( -\frac{1}{2}(y-m(X))^T C_\theta^{-1}(y-m(X)) \right).$$</blockquote>
            <p>
                Esta cantidad mide cuán plausibles son los datos observados bajo el modelo definido por los hiperparámetros $\theta$. Elegir buenos hiperparámetros significa, en esencia, elegir aquellos valores que hacen que los datos observados sean más probables según el modelo.
            </p>
            <p>
                En la práctica se trabaja casi siempre con el <em>log marginal likelihood</em>, porque resulta algebraicamente más manejable y numéricamente más estable. Tomando logaritmos se obtiene
            </p>
            <blockquote>$$\log p(y \mid X,\theta) = -\frac{1}{2}(y-m(X))^T C_\theta^{-1}(y-m(X)) -\frac{1}{2}\log |C_\theta| -\frac{n}{2}\log(2\pi).$$</blockquote>
            <p>
                Esta fórmula es una de las más importantes de toda la teoría de GP. Conviene interpretarla término por término. El primer término, $-\frac{1}{2}(y-m(X))^T C_\theta^{-1}(y-m(X))$, mide qué tan bien el modelo ajusta los datos. Si los datos observados se alinean bien con la media y la estructura de covarianza propuesta, este término será relativamente grande en valor negativo moderado; si el ajuste es pobre, será muy penalizante. En cierto sentido, este es el término de fidelidad a los datos.
            </p>
            <p>
                El segundo término, $-\frac{1}{2}\log |C_\theta|$, actúa como una penalización por complejidad. El determinante $|C_\theta|$ refleja el volumen de incertidumbre permitido por el modelo. Modelos que generan una covarianza muy amplia o muy flexible pueden pagar un precio en este término. Por eso suele decirse que aquí aparece una forma automática de control de complejidad: el modelo no es premiado solo por ajustarse bien a los datos, sino también por hacerlo de manera parsimoniosa.
            </p>
            <p>
                El tercer término, $-\frac{n}{2}\log(2\pi)$, es una constante respecto de $\theta$, y no afecta directamente la optimización, aunque sí forma parte de la expresión completa.
            </p>
            <p>
                Esta descomposición es conceptualmente profunda, porque muestra que la selección de hiperparámetros en procesos gaussianos no consiste solo en "ajustar lo mejor posible" en un sentido ingenuo. El log marginal likelihood equilibra automáticamente dos fuerzas: el ajuste a los datos y la complejidad del modelo. Esa es una de las ventajas más elegantes del enfoque bayesiano. En lugar de introducir una penalización ad hoc, la propia estructura probabilística de la verosimilitud marginal incorpora un balance natural entre flexibilidad y parsimonia.
            </p>
        """
    },
    {
        "id": "sec-6",
        "titulo": "6. Optimización y el dilema Sobreajuste vs. Suavidad",
        "contenido": r"""
            <p>
                El criterio habitual consiste entonces en elegir
            </p>
            <blockquote>$$\hat\theta = \arg\max_\theta \log p(y \mid X,\theta),$$</blockquote>
            <p>
                o, equivalentemente,
            </p>
            <blockquote>$$\hat\theta = \arg\min_\theta \left[-\log p(y \mid X,\theta)\right].$$</blockquote>
            <p>
                Dado que las restricciones naturales son $\ell>0$, $\sigma_f^2>0$, $\sigma_n^2>0$, muchas veces la optimización se realiza sobre los logaritmos de estos parámetros, por ejemplo
            </p>
            <blockquote>$$\eta_\ell = \log \ell, \qquad \eta_f = \log \sigma_f^2, \qquad \eta_n = \log \sigma_n^2,$$</blockquote>
            <p>
                para garantizar automáticamente su positividad. Esto también facilita el uso de métodos numéricos de optimización, como gradiente, quasi-Newton o algoritmos similares.
            </p>
            <p>
                Desde el punto de vista interpretativo, la selección de hiperparámetros controla de manera decisiva el compromiso entre sobreajuste y suavidad. Este es uno de los temas más importantes del aprendizaje con procesos gaussianos. Para entenderlo, conviene analizar qué ocurre cuando se modifican los hiperparámetros extremos.
            </p>
            <p>
                Supongamos primero que la longitud de escala $\ell$ es muy pequeña. Entonces el kernel RBF decrece rápidamente, y puntos incluso moderadamente cercanos se vuelven casi incorrelacionados. El proceso puede variar muy bruscamente de un lugar a otro. Si además la varianza del ruido $\sigma_n^2$ es pequeña, el modelo interpretará que casi toda fluctuación en los datos debe ser explicada por la función latente. El resultado será una función posterior que sigue muy de cerca las observaciones y puede adaptarse incluso a irregularidades accidentales debidas al ruido. Este es el escenario típico de sobreajuste: el modelo se vuelve demasiado sensible a detalles locales y pierde capacidad de generalización.
            </p>
            <p>
                En el extremo opuesto, si la longitud de escala $\ell$ es muy grande, la covarianza entre puntos decae muy lentamente, y el modelo fuerza a la función a variar de manera muy suave. Si además la varianza del ruido $\sigma_n^2$ es grande, el GP interpretará que buena parte de la dispersión observada proviene del ruido y no de la señal latente. En este caso, la función posterior tenderá a ser muy lisa, quizá demasiado, y puede no capturar variaciones reales del fenómeno. Este es el riesgo del subajuste o de una suavidad excesiva: el modelo ignora estructura genuina de los datos en nombre de una regularidad demasiado fuerte.
            </p>
            <p>
                La varianza del proceso $\sigma_f^2$ también participa en este balance. Si es muy pequeña, la función se mantiene muy cerca de la media previa, lo que puede producir un ajuste demasiado rígido. Si es muy grande, el modelo permite oscilaciones de alta amplitud, lo que aumenta la flexibilidad y, según la longitud de escala y el ruido, puede favorecer adaptaciones demasiado agresivas. Por eso la selección de hiperparámetros no puede hacerse parámetro por parámetro en aislamiento: importa la combinación conjunta entre escala horizontal, escala vertical y nivel de ruido.
            </p>
            <p>
                El papel del log marginal likelihood se comprende mejor precisamente en esta tensión entre sobreajuste y suavidad. Un modelo demasiado flexible puede ajustarse bien a los datos de entrenamiento, pero paga un precio en el término de complejidad. Un modelo demasiado rígido puede tener un término de complejidad favorable, pero ajustará mal los datos y será penalizado por el término de fidelidad. La optimización de la verosimilitud marginal busca un punto de equilibrio en el que el modelo sea suficientemente flexible para explicar la evidencia, pero no tanto como para perseguir el ruido. Esta es una de las razones por las cuales los procesos gaussianos son tan atractivos: la regularización no se introduce externamente, sino que emerge del propio marco probabilístico.
            </p>
            <p>
                Puede ser útil ilustrar esto con una imagen conceptual simple. Supongamos que se tienen datos de una relación funcional suave, pero con pequeñas perturbaciones aleatorias. Si se elige un $\ell$ extremadamente pequeño y un $\sigma_n^2$ casi nulo, el GP generará una curva posterior muy ondulada que pasa cerca de casi todos los puntos, incluida la variación espuria del ruido. Si, en cambio, se elige un $\ell$ moderado y un $\sigma_n^2$ compatible con el nivel de dispersión observada, la media posterior tenderá a recuperar la tendencia subyacente y a ignorar pequeñas oscilaciones accidentales. Aquí se ve claramente que el problema de hiperparámetros es, en esencia, el problema de decidir qué parte de los datos debe interpretarse como señal y qué parte como ruido.
            </p>
            <div style="text-align: center; margin: 2rem 0;">

            </div>
        """
    },
    {
        "id": "sec-7",
        "titulo": "7. Cuantificación de incertidumbre y Conclusión",
        "contenido": r"""
            <p>
                También es importante notar que la selección de hiperparámetros no solo afecta la media posterior, sino también la incertidumbre posterior. Un modelo con ruido grande y longitud de escala grande tenderá a producir bandas de incertidumbre amplias y predicciones conservadoras. Un modelo con ruido pequeño y correlación muy local puede producir bandas estrechas cerca de los puntos observados, pero quizá con una estructura demasiado optimista o demasiado irregular. Por tanto, la elección de $\theta$ regula simultáneamente ajuste, suavidad y cuantificación de incertidumbre.
            </p>
            <p>
                En contextos prácticos, además de maximizar el log marginal likelihood, a veces se introducen <em>priors</em> sobre los hiperparámetros y se realiza inferencia bayesiana plena sobre $\theta$. En ese caso, en lugar de seleccionar un único valor óptimo, se integra sobre la incertidumbre en los hiperparámetros. Sin embargo, el enfoque más común y computacionalmente eficiente sigue siendo el de maximización de la verosimilitud marginal, a veces llamado <em>evidence maximization</em>. En este marco, los hiperparámetros elegidos son aquellos que mejor explican la evidencia observada.
            </p>
            <p>
                En resumen, la selección de hiperparámetros en procesos gaussianos es la etapa en la que se determina la escala de correlación del fenómeno, la amplitud de sus fluctuaciones y el nivel de ruido presente en las observaciones. La longitud de escala $\ell$ controla la suavidad y la correlación local; la varianza del proceso $\sigma_f^2$ controla la amplitud vertical de las funciones plausibles; la varianza del ruido $\sigma_n^2$ controla cuánto de la variabilidad observada se atribuye a señal y cuánto a ruido. El criterio de selección estándar es la maximización del log marginal likelihood,
            </p>
            <blockquote>$$\log p(y \mid X,\theta) = -\frac{1}{2}(y-m(X))^T C_\theta^{-1}(y-m(X)) -\frac{1}{2}\log |C_\theta| -\frac{n}{2}\log(2\pi),$$</blockquote>
            <p>
                que equilibra ajuste a los datos y complejidad del modelo. Gracias a este equilibrio, los procesos gaussianos ofrecen un mecanismo elegante para navegar entre el sobreajuste y la suavidad excesiva, aprendiendo funciones que sean flexibles pero no arbitrarias.
            </p>
        """
    }
]

# =========================================================
# 3. GENERACIÓN HTML
# =========================================================

def generar_tarjetas_acordeon(datos):
    bloques = []
    for seccion in datos:
        bloques.append(f"""
        <div class="topic-card" id="{html.escape(seccion['id'])}">
            <div class="topic-header">
                <span class="topic-title">{seccion['titulo']}</span>
                <i class="fas fa-chevron-down expand-icon"></i>
            </div>
            <div class="topic-content">
                {seccion['contenido']}
            </div>
        </div>
        """)
    return "\n".join(bloques)

def generar_navegacion(datos):
    chips = []
    for seccion in datos:
        chips.append(
            f'<button class="nav-chip" type="button" data-target="{html.escape(seccion["id"])}">{seccion["titulo"]}</button>'
        )
    return "\n".join(chips)

contenido_dinamico_html = generar_tarjetas_acordeon(secciones_principales_data)
navegacion_html = generar_navegacion(secciones_principales_data)

# =========================================================
# 4. PLANTILLA HTML
# =========================================================

plantilla_profesional = r"""
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{{MAIN_TITLE}}</title>

  <script type="text/x-mathjax-config">
    MathJax.Hub.Config({
      tex2jax: {
        inlineMath: [['$','$']],
        displayMath: [['$$','$$']],
        processEscapes: true
      }
    });
  </script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-MML-AM_CHTML" async></script>

  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
  <link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css" rel="stylesheet">

  <style>
    :root {
      --bg-primary: linear-gradient(135deg, #f0f9ff 0%, #e0f2fe 100%);
      --bg-secondary: rgba(255, 255, 255, 0.85);
      --bg-tertiary: rgba(248, 250, 252, 0.8);
      --text-primary: #0c4a6e;
      --text-secondary: #075985;
      --accent-primary: #0ea5e9;
      --accent-secondary: #38bdf8;
      --accent-gradient: linear-gradient(135deg, var(--accent-primary) 0%, var(--accent-secondary) 100%);
      --border-color: rgba(226, 232, 240, 0.8);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.08);
      --border-radius: 20px;
      --transition: all 0.4s cubic-bezier(0.25, 0.8, 0.25, 1);
      --nav-chip-bg: rgba(255,255,255,0.8);
      --progress-bg: rgba(255,255,255,0.45);
    }

    [data-theme="dark"] {
      --bg-primary: linear-gradient(135deg, #0c4a6e 0%, #1e3a8a 100%);
      --bg-secondary: rgba(30, 58, 138, 0.85);
      --bg-tertiary: rgba(30, 64, 175, 0.7);
      --text-primary: #f0f9ff;
      --text-secondary: #dbeafe;
      --accent-primary: #38bdf8;
      --accent-secondary: #7dd3fc;
      --border-color: rgba(255, 255, 255, 0.15);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.20);
      --nav-chip-bg: rgba(30, 58, 138, 0.65);
      --progress-bg: rgba(255,255,255,0.15);
    }

    * { margin: 0; padding: 0; box-sizing: border-box; }
    html { scroll-behavior: smooth; }
    body { font-family: 'Inter', sans-serif; line-height: 1.8; background: var(--bg-primary); color: var(--text-primary); transition: var(--transition); min-height: 100vh; position: relative; overflow-x: hidden; }

    .particles { position: fixed; top: 0; left: 0; width: 100%; height: 100%; pointer-events: none; z-index: -1; }
    .particle { position: absolute; border-radius: 50%; animation: float 25s infinite linear; opacity: 0; background: rgba(255, 255, 255, 0.55); box-shadow: 0 0 15px rgba(255,255,255,0.25); }
    @keyframes float { 0% { transform: translateY(100vh) rotate(0deg); opacity: 0; } 10%, 90% { opacity: 0.55; } 100% { transform: translateY(-10vh) rotate(360deg); opacity: 0; } }

    .progress-wrapper { position: fixed; top: 0; left: 0; width: 100%; height: 6px; background: var(--progress-bg); z-index: 1200; backdrop-filter: blur(10px); }
    .progress-bar { height: 100%; width: 0%; background: var(--accent-gradient); transition: width 0.15s ease; }

    .theme-toggle { position: fixed; top: 1.2rem; right: 1.2rem; width: 60px; height: 60px; border: 1px solid var(--border-color); border-radius: 50%; background: var(--bg-secondary); backdrop-filter: blur(15px); box-shadow: var(--shadow-card); cursor: pointer; display: flex; align-items: center; justify-content: center; font-size: 1.35rem; color: var(--text-primary); transition: var(--transition); z-index: 1100; }
    .theme-toggle:hover { transform: scale(1.15) rotate(180deg); }

    .container { max-width: 1100px; margin: 0 auto; padding: 2rem; z-index: 1; position: relative; }
    .header { text-align: center; margin-bottom: 2rem; position: relative; padding-top: 1.2rem; }
    .header-badge { display: inline-flex; align-items: center; gap: 0.6rem; background: var(--bg-secondary); border: 1px solid var(--border-color); border-radius: 999px; padding: 0.7rem 1rem; backdrop-filter: blur(15px); box-shadow: var(--shadow-card); margin-bottom: 1rem; color: var(--text-secondary); font-size: 0.95rem; }

    .main-title { font-size: clamp(2.2rem, 5vw, 3.8rem); font-weight: 800; background: var(--accent-gradient); -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text; margin-bottom: 1rem; line-height: 1.15; }
    .subtitle { color: var(--text-secondary); max-width: 900px; margin: 0 auto; font-size: 1.03rem; }

    .content-block, .quick-nav, .control-bar, .topic-card { background: var(--bg-secondary); backdrop-filter: blur(20px); border-radius: var(--border-radius); box-shadow: var(--shadow-card); border: 2px solid var(--border-color); }
    .content-block { padding: 2rem; margin-top: 2rem; }
    .content-block h2 { color: var(--text-primary); margin-bottom: 1rem; font-size: 1.55rem; }
    .content-block p, .content-block li, .subtitle { color: var(--text-secondary); }

    .quick-nav { margin-top: 2rem; padding: 1.4rem; }
    .quick-nav h3 { margin-bottom: 1rem; color: var(--text-primary); font-size: 1.2rem; }
    .nav-grid { display: flex; flex-wrap: wrap; gap: 0.75rem; }
    .nav-chip { padding: 0.65rem 0.95rem; border-radius: 999px; background: var(--nav-chip-bg); color: var(--text-primary); border: 1px solid var(--border-color); font-size: 0.92rem; transition: var(--transition); box-shadow: 0 8px 18px rgba(0,0,0,0.06); cursor: pointer; font-family: inherit; text-align: left; }
    .nav-chip:hover { transform: translateY(-2px); background: var(--bg-tertiary); }

    .control-bar { margin-top: 1.5rem; padding: 1rem 1.2rem; display: flex; gap: 0.8rem; flex-wrap: wrap; justify-content: center; }
    .control-btn { border: 1px solid var(--border-color); background: var(--nav-chip-bg); color: var(--text-primary); border-radius: 999px; padding: 0.75rem 1rem; cursor: pointer; font-family: inherit; font-size: 0.95rem; transition: var(--transition); }
    .control-btn:hover { transform: translateY(-2px); background: var(--bg-tertiary); }

    .lesson-container { display: flex; flex-direction: column; gap: 1.3rem; margin-top: 1.6rem; }
    .topic-card { overflow: hidden; transition: var(--transition); scroll-margin-top: 90px; }
    .topic-card:hover { transform: translateY(-2px); }
    .topic-header { cursor: pointer; padding: 1.5rem 2rem; display: flex; justify-content: space-between; align-items: center; gap: 1rem; user-select: none; }
    .topic-title { font-size: 1.22rem; font-weight: 700; color: var(--text-primary); line-height: 1.4; }
    .expand-icon { font-size: 1.15rem; color: var(--text-secondary); transition: var(--transition); flex-shrink: 0; }
    .topic-card.open .expand-icon { transform: rotate(180deg); }

    .topic-content { max-height: 0; overflow: hidden; transition: max-height 1.2s ease, padding 1.2s ease; background: var(--bg-tertiary); }
    .topic-card.open .topic-content { max-height: 6500px; padding: 1.6rem 2rem 1.8rem 2rem; border-top: 1px solid var(--border-color); }
    .topic-content h4 { color: var(--text-primary); margin-top: 1.5rem; margin-bottom: 0.55rem; font-size: 1.08rem; border-left: 4px solid var(--accent-primary); padding-left: 12px; }
    .topic-content h4:first-child { margin-top: 0; }
    .topic-content p, .topic-content li { color: var(--text-secondary); line-height: 1.8; margin-bottom: 0.95rem; }
    .topic-content strong { color: var(--text-primary); font-weight: 700; }

    .topic-content blockquote { text-align: center; border-left: 4px solid var(--accent-primary); margin: 1.4rem 0; font-style: normal; color: var(--text-secondary); background: rgba(127, 140, 141, 0.05); border-radius: 0 10px 10px 0; padding: 1rem 1.5rem; overflow-x: auto; }

    footer { text-align: center; margin-top: 4rem; padding-top: 2rem; border-top: 1px solid var(--border-color); }
    footer p { color: var(--text-secondary); font-size: 0.95rem; opacity: 0.9; }

    @media (max-width: 768px) {
      .container { padding: 1rem; }
      .topic-header { padding: 1.1rem 1.3rem; }
      .topic-card.open .topic-content { padding: 1.1rem 1.3rem 1.3rem 1.3rem; }
      .theme-toggle { width: 54px; height: 54px; top: 1rem; right: 1rem; }
    }
  </style>
</head>

<body data-theme="dark">
  <div class="progress-wrapper">
    <div class="progress-bar" id="progressBar"></div>
  </div>

  <div class="particles" id="particles-container"></div>

  <div class="theme-toggle" id="themeToggleButton" title="Cambiar tema">
    <i class="fas fa-moon" id="theme-icon"></i>
  </div>

  <div class="container">
    <header class="header">
      <div class="header-badge">
        <i class="fas fa-project-diagram"></i>
        <span>Estadística Aplicada a Data Science · Repositorio</span>
      </div>
      <h1 class="main-title">{{MAIN_TITLE}}</h1>
      <p class="subtitle">
        Optimización de la covarianza: control de la suavidad, la amplitud y el ruido a través de la maximización de la Verosimilitud Marginal.
      </p>
    </header>

    {{INTRO_HTML}}

    <section class="quick-nav">
      <h3>Navegación rápida</h3>
      <div class="nav-grid">
        {{NAV_HTML}}
      </div>
    </section>

    <section class="control-bar">
      <button class="control-btn" id="expandAllBtn" type="button"><i class="fas fa-plus-circle"></i> Expandir todo</button>
      <button class="control-btn" id="collapseAllBtn" type="button"><i class="fas fa-minus-circle"></i> Contraer todo</button>
      <button class="control-btn" id="topBtn" type="button"><i class="fas fa-arrow-up"></i> Volver arriba</button>
    </section>

    <div class="lesson-container">
      {{DYNAMIC_HTML}}
    </div>

    <footer>
      <p>{{FOOTER_TEXT}}</p>
    </footer>
  </div>

  <script>
    (function() {
      const themeToggleButton = document.getElementById('themeToggleButton');
      const themeIcon = document.getElementById('theme-icon');
      const bodyEl = document.body;
      const expandAllBtn = document.getElementById('expandAllBtn');
      const collapseAllBtn = document.getElementById('collapseAllBtn');
      const topBtn = document.getElementById('topBtn');
      const progressBar = document.getElementById('progressBar');

      function setTheme(theme) {
        bodyEl.setAttribute('data-theme', theme);
        try { localStorage.setItem('gp_theme', theme); } catch (e) {}
        if (themeIcon) { themeIcon.className = theme === 'dark' ? 'fas fa-sun' : 'fas fa-moon'; }
      }

      if (themeToggleButton) {
        themeToggleButton.addEventListener('click', () => {
          const newTheme = (bodyEl.getAttribute('data-theme') || 'dark') === 'dark' ? 'light' : 'dark';
          setTheme(newTheme);
        });
      }

      let preferredTheme = 'dark';
      try { preferredTheme = localStorage.getItem('gp_theme') || 'dark'; } catch (e) {}
      setTheme(preferredTheme);

      const cards = Array.from(document.querySelectorAll('.topic-card'));

      cards.forEach(card => {
        const header = card.querySelector('.topic-header');
        if (header) {
          header.addEventListener('click', () => {
            card.classList.toggle('open');
          });
        }
      });

      if (expandAllBtn) { expandAllBtn.addEventListener('click', () => { cards.forEach(card => card.classList.add('open')); }); }
      if (collapseAllBtn) { collapseAllBtn.addEventListener('click', () => { cards.forEach(card => card.classList.remove('open')); }); }
      if (topBtn) { topBtn.addEventListener('click', () => { window.scrollTo({ top: 0, behavior: 'smooth' }); }); }

      const navButtons = Array.from(document.querySelectorAll('.nav-chip'));
      navButtons.forEach(btn => {
        btn.addEventListener('click', () => {
          const targetId = btn.getAttribute('data-target');
          const target = document.getElementById(targetId);
          if (target) {
            target.classList.add('open');
            setTimeout(() => { target.scrollIntoView({ behavior: 'smooth', block: 'start' }); }, 120);
          }
        });
      });

      const firstTopic = document.querySelector('.topic-card');
      if (firstTopic) { firstTopic.classList.add('open'); }

      const container = document.getElementById('particles-container');
      if (container) {
        for (let i = 0; i < 34; i++) {
          const p = document.createElement('div');
          p.className = 'particle';
          p.style.left = Math.random() * 100 + 'vw';
          const size = (Math.random() * 6 + 3);
          p.style.width = size + 'px';
          p.style.height = size + 'px';
          p.style.animationDelay = Math.random() * -20 + 's';
          p.style.animationDuration = (15 + Math.random() * 12) + 's';
          container.appendChild(p);
        }
      }

      function updateProgress() {
        const scrollTop = document.documentElement.scrollTop || document.body.scrollTop;
        const scrollHeight = document.documentElement.scrollHeight - document.documentElement.clientHeight;
        const progress = scrollHeight > 0 ? (scrollTop / scrollHeight) * 100 : 0;
        progressBar.style.width = progress + '%';
      }

      window.addEventListener('scroll', updateProgress);
      updateProgress();

      if (window.MathJax) {
          MathJax.Hub.Queue(["Typeset", MathJax.Hub]);
      }
    })();
  </script>
</body>
</html>
"""

# =========================================================
# 5. ENSAMBLADO
# =========================================================

final_html = (
    plantilla_profesional
    .replace("{{MAIN_TITLE}}", "7. Selección de hiperparámetros")
    .replace("{{INTRO_HTML}}", intro_html_content)
    .replace("{{NAV_HTML}}", navegacion_html)
    .replace("{{DYNAMIC_HTML}}", contenido_dinamico_html)
    .replace("{{FOOTER_TEXT}}", "Material elaborado por el profesor Sergio Gevatschnaider.")
)

display(HTML(final_html))

In [ ]:
# @title Índice Interactivo: 8. Relación con otros métodos
from IPython.display import display, HTML
import html

# =========================================================
# 1. INTRODUCCIÓN
# =========================================================

intro_html_content = r"""
<div class="content-block" style="margin-top:0;">
    <h2>Resumen de la Sección</h2>
    <p>
        Los procesos gaussianos pueden entenderse como un punto de convergencia entre varios métodos clásicos y modernos
        de aprendizaje estadístico. No constituyen una técnica aislada, sino una formulación probabilística que reinterpreta
        modelos lineales bayesianos, métodos de suavizado, algoritmos kernel y ciertos límites de redes neuronales dentro de un mismo lenguaje.
    </p>
    <p>
        En esta sección exploraremos la red de equivalencias teóricas que conecta a los GP con otras familias de algoritmos,
        demostrando cómo el concepto de <em>distribución sobre funciones</em> unifica la inferencia bayesiana, la regularización geométrica
        y el aprendizaje no lineal.
    </p>
</div>
"""

# =========================================================
# 2. SECCIONES (Texto 100% completo y LaTeX corregido)
# =========================================================

secciones_principales_data = [
    {
        "id": "sec-1",
        "titulo": "1. Regresión Lineal Bayesiana: El caso base",
        "contenido": r"""
            <h4>De pesos paramétricos a funciones aleatorias</h4>
            <p>
                La conexión más directa aparece con la regresión lineal bayesiana. Si se parte del modelo
            </p>
            <blockquote>$$y_i = \phi(x_i)^T w + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0,\sigma^2),$$</blockquote>
            <p>
                con prior gaussiano sobre los pesos
            </p>
            <blockquote>$$w \sim \mathcal{N}(0,\Sigma_w),$$</blockquote>
            <p>
                entonces la función latente $f(x) = \phi(x)^T w$ es un objeto aleatorio gaussiano. En efecto, para cualquier colección finita $x_1,\dots,x_n$, el vector $(f(x_1),\dots,f(x_n))^T$ tiene distribución normal multivariante con media cero y matriz de covarianza
            </p>
            <blockquote>$$K_{ij} = k(x_i,x_j) = \phi(x_i)^T \Sigma_w \, \phi(x_j).$$</blockquote>
            <p>
                Por lo tanto, la regresión lineal bayesiana es un caso particular de proceso gaussiano con kernel inducido por un mapa de características finito. Desde esta perspectiva, un GP generaliza la regresión lineal bayesiana al reemplazar una base explícita y finita por una covarianza definida directamente en el espacio de entrada, permitiendo trabajar con espacios de dimensión muy alta o incluso infinita sin construir las <em>features</em> de forma explícita.
            </p>
        """
    },
    {
        "id": "sec-2",
        "titulo": "2. Splines y regularización variacional",
        "contenido": r"""
            <h4>Suavizado equivalente a un prior gaussiano</h4>
            <p>
                La relación con <em>splines</em> surge al interpretar el problema de estimación funcional como un equilibrio entre ajuste a los datos y suavidad. En <em>smoothing splines</em>, la función estimada suele obtenerse minimizando un funcional del tipo
            </p>
            <blockquote>$$\sum_{i=1}^n (y_i-f(x_i))^2 + \lambda \int \left(f^{(m)}(t)\right)^2 \, dt,$$</blockquote>
            <p>
                donde el segundo término penaliza curvatura u oscilación. Esta formulación variacional tiene una lectura bayesiana natural: la penalización de derivadas corresponde a imponer un prior gaussiano sobre la función, con covarianza diseñada para favorecer regularidad.
            </p>
            <p>
                Así, muchos splines pueden verse como estimadores MAP o medias posteriores bajo procesos gaussianos apropiados. En particular, ciertas familias de <em>smoothing splines</em> aparecen al elegir kernels vinculados con espacios de Sobolev o con procesos integrados. La idea central es que "suavizar" y "poner un prior gaussiano sobre funciones regulares" son, en gran medida, dos lenguajes para describir la misma estructura.
            </p>
        """
    },
    {
        "id": "sec-3",
        "titulo": "3. Métodos Kernel (Kernel Ridge Regression)",
        "contenido": r"""
            <h4>Predicción puntual vs. Inferencia completa</h4>
            <p>
                La conexión con <em>kernel methods</em> es todavía más estrecha. Dado un kernel positivo definido $k$, la matriz de Gram $K_{ij}=k(x_i,x_j)$ organiza las similitudes entre observaciones. En <em>kernel ridge regression</em>, la predicción en un nuevo punto $x_*$ toma la forma
            </p>
            <blockquote>$$\hat{f}(x_*) = k_*^T (K+\lambda I)^{-1} y,$$</blockquote>
            <p>
                donde $k_* = (k(x_*,x_1),\dots,k(x_*,x_n))^T$. En regresión con proceso gaussiano, si el ruido es gaussiano con varianza $\sigma^2$, la media posterior es
            </p>
            <blockquote>$$m_*(x_*) = k_*^T (K+\sigma^2 I)^{-1} y.$$</blockquote>
            <p>
                La coincidencia formal no es accidental: la media posterior de un GP y la solución de <em>kernel ridge regression</em> comparten la misma estructura algebraica. La diferencia conceptual es que el método kernel clásico entrega un estimador puntual regularizado, mientras que el GP proporciona además cuantificación explícita de incertidumbre mediante la varianza posterior
            </p>
            <blockquote>$$\operatorname{Var}(f(x_*) \mid y) = k(x_*,x_*) - k_*^T (K+\sigma^2 I)^{-1} k_*.$$</blockquote>
            <p>
                En otras palabras, los <em>kernel methods</em> producen predicción; los procesos gaussianos producen predicción <em>más</em> distribución posterior sobre funciones.
            </p>
        """
    },
    {
        "id": "sec-4",
        "titulo": "4. El espacio RKHS como puente geométrico",
        "contenido": r"""
            <h4>El Teorema del Representador</h4>
            <p>
                El espacio de Hilbert con núcleo reproductor, o RKHS, funciona como puente teórico entre ambas miradas. Todo kernel positivo definido $k$ induce un RKHS $\mathcal{H}_k$, que es un espacio de funciones donde el producto interno satisface la propiedad reproductora
            </p>
            <blockquote>$$f(x) = \langle f, k(\cdot,x) \rangle_{\mathcal{H}_k}.$$</blockquote>
            <p>
                Este hecho convierte al kernel en objeto geométrico y no solo algorítmico. En aprendizaje regularizado, la norma del RKHS controla complejidad funcional, y problemas del tipo
            </p>
            <blockquote>$$\min_{f \in \mathcal{H}_k} \sum_{i=1}^n L(y_i,f(x_i)) + \lambda \|f\|_{\mathcal{H}_k}^2$$</blockquote>
            <p>
                admiten, por el teorema del representador, soluciones finitas de la forma
            </p>
            <blockquote>$$f(x) = \sum_{i=1}^n \alpha_i k(x,x_i).$$</blockquote>
            <p>
                Desde el lado bayesiano, el mismo kernel define la covarianza de un GP. Sin embargo, conviene distinguir ambos niveles: trayectorias típicas de un proceso gaussiano no pertenecen necesariamente al RKHS asociado, pero el RKHS describe de manera precisa las direcciones de variación "de energía finita" y la geometría de regularización inducida por el prior. Así, el RKHS articula la equivalencia entre regularización determinista y modelado probabilístico, mostrando que penalizar norma funcional y asumir covarianza gaussiana son dos expresiones de una misma estructura subyacente.
            </p>
            <div style="text-align: center; margin: 2rem 0;">

            </div>
        """
    },
    {
        "id": "sec-5",
        "titulo": "5. Diferencias estructurales con las Redes Neuronales",
        "contenido": r"""
            <h4>No paramétrico vs Paramétrico</h4>
            <p>
                La diferencia entre procesos gaussianos y redes neuronales puede formularse en varios planos. En primer lugar, un GP es un modelo no paramétrico en el sentido funcional: la complejidad efectiva crece con los datos y el objeto primitivo es una distribución sobre funciones. Una red neuronal estándar es paramétrica: parte de un conjunto finito de pesos y aprende una función mediante optimización en un espacio de parámetros.
            </p>
            <h4>Sesgo inductivo e inferencia</h4>
            <p>
                En segundo lugar, el GP separa de manera nítida la elección de <em>inductive bias</em> a través del kernel. Propiedades como suavidad, periodicidad, estacionariedad o estructura multi-escala se codifican directamente en $k(x,x')$. En redes neuronales, ese sesgo inductivo emerge de la arquitectura, la inicialización, la dinámica de entrenamiento y la regularización implícita.
            </p>
            <p>
                En tercer lugar, la inferencia también difiere. En GP, la actualización posterior es cerrada en el caso gaussiano y produce incertidumbre calibrada de manera explícita. En redes neuronales, el entrenamiento estándar entrega usualmente una sola solución puntual, y la incertidumbre requiere mecanismos adicionales (aproximaciones bayesianas, <em>ensembles</em>, <em>dropout</em>).
            </p>
            <p>
                En cuarto lugar, computacionalmente, los GP exactos requieren invertir matrices $n \times n$, con costo típicamente cúbico en el tamaño muestral, mientras que las redes escalan mejor a grandes volúmenes de datos mediante entrenamiento estocástico.
            </p>
        """
    },
    {
        "id": "sec-6",
        "titulo": "6. Límites infinitos y Conclusión",
        "contenido": r"""
            <h4>Redes de ancho infinito</h4>
            <p>
                Existe además una relación profunda entre ambos mundos: ciertas redes neuronales de ancho infinito convergen, bajo hipótesis apropiadas de inicialización, a procesos gaussianos (<em>Neural Network Gaussian Process</em>, NNGP). En ese régimen, la covarianza inducida por la arquitectura define un kernel límite. Esto muestra que un GP puede interpretarse como el límite infinito de ciertas familias de redes.
            </p>
            <p>
                Sin embargo, una red neuronal entrenada finitamente no es en general equivalente a ese GP límite, porque el aprendizaje modifica los parámetros y, con ello, la representación interna (<em>feature learning</em>). Por eso, aunque comparten vínculos asintóticos, GP y redes neuronales siguen siendo paradigmas distintos: uno prioriza inferencia probabilística exacta sobre funciones; el otro prioriza flexibilidad representacional y escalabilidad mediante optimización paramétrica.
            </p>
            <h4>Síntesis conceptual</h4>
            <p>
                En síntesis, la regresión lineal bayesiana aparece como el caso de base explícita finita, los splines como formulaciones de suavidad equivalentes a ciertos priors gaussianos, los métodos kernel como contrapartes regularizadas cuya solución coincide con la media posterior del GP, y el RKHS como el lenguaje geométrico que unifica covarianza, regularización y representación funcional. Esta red de equivalencias convierte a los procesos gaussianos en una pieza central para comprender cómo se conectan inferencia bayesiana, geometría funcional y aprendizaje no lineal.
            </p>
        """
    }
]

# =========================================================
# 3. GENERACIÓN HTML
# =========================================================

def generar_tarjetas_acordeon(datos):
    bloques = []
    for seccion in datos:
        bloques.append(f"""
        <div class="topic-card" id="{html.escape(seccion['id'])}">
            <div class="topic-header">
                <span class="topic-title">{seccion['titulo']}</span>
                <i class="fas fa-chevron-down expand-icon"></i>
            </div>
            <div class="topic-content">
                {seccion['contenido']}
            </div>
        </div>
        """)
    return "\n".join(bloques)

def generar_navegacion(datos):
    chips = []
    for seccion in datos:
        chips.append(
            f'<button class="nav-chip" type="button" data-target="{html.escape(seccion["id"])}">{seccion["titulo"]}</button>'
        )
    return "\n".join(chips)

contenido_dinamico_html = generar_tarjetas_acordeon(secciones_principales_data)
navegacion_html = generar_navegacion(secciones_principales_data)

# =========================================================
# 4. PLANTILLA HTML
# =========================================================

plantilla_profesional = r"""
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{{MAIN_TITLE}}</title>

  <script type="text/x-mathjax-config">
    MathJax.Hub.Config({
      tex2jax: {
        inlineMath: [['$','$']],
        displayMath: [['$$','$$']],
        processEscapes: true
      }
    });
  </script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-MML-AM_CHTML" async></script>

  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
  <link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css" rel="stylesheet">

  <style>
    :root {
      --bg-primary: linear-gradient(135deg, #f0f9ff 0%, #e0f2fe 100%);
      --bg-secondary: rgba(255, 255, 255, 0.85);
      --bg-tertiary: rgba(248, 250, 252, 0.8);
      --text-primary: #0c4a6e;
      --text-secondary: #075985;
      --accent-primary: #0ea5e9;
      --accent-secondary: #38bdf8;
      --accent-gradient: linear-gradient(135deg, var(--accent-primary) 0%, var(--accent-secondary) 100%);
      --border-color: rgba(226, 232, 240, 0.8);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.08);
      --border-radius: 20px;
      --transition: all 0.4s cubic-bezier(0.25, 0.8, 0.25, 1);
      --nav-chip-bg: rgba(255,255,255,0.8);
      --progress-bg: rgba(255,255,255,0.45);
    }

    [data-theme="dark"] {
      --bg-primary: linear-gradient(135deg, #0c4a6e 0%, #1e3a8a 100%);
      --bg-secondary: rgba(30, 58, 138, 0.85);
      --bg-tertiary: rgba(30, 64, 175, 0.7);
      --text-primary: #f0f9ff;
      --text-secondary: #dbeafe;
      --accent-primary: #38bdf8;
      --accent-secondary: #7dd3fc;
      --border-color: rgba(255, 255, 255, 0.15);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.20);
      --nav-chip-bg: rgba(30, 58, 138, 0.65);
      --progress-bg: rgba(255,255,255,0.15);
    }

    * { margin: 0; padding: 0; box-sizing: border-box; }
    html { scroll-behavior: smooth; }
    body { font-family: 'Inter', sans-serif; line-height: 1.8; background: var(--bg-primary); color: var(--text-primary); transition: var(--transition); min-height: 100vh; position: relative; overflow-x: hidden; }

    .particles { position: fixed; top: 0; left: 0; width: 100%; height: 100%; pointer-events: none; z-index: -1; }
    .particle { position: absolute; border-radius: 50%; animation: float 25s infinite linear; opacity: 0; background: rgba(255, 255, 255, 0.55); box-shadow: 0 0 15px rgba(255,255,255,0.25); }
    @keyframes float { 0% { transform: translateY(100vh) rotate(0deg); opacity: 0; } 10%, 90% { opacity: 0.55; } 100% { transform: translateY(-10vh) rotate(360deg); opacity: 0; } }

    .progress-wrapper { position: fixed; top: 0; left: 0; width: 100%; height: 6px; background: var(--progress-bg); z-index: 1200; backdrop-filter: blur(10px); }
    .progress-bar { height: 100%; width: 0%; background: var(--accent-gradient); transition: width 0.15s ease; }

    .theme-toggle { position: fixed; top: 1.2rem; right: 1.2rem; width: 60px; height: 60px; border: 1px solid var(--border-color); border-radius: 50%; background: var(--bg-secondary); backdrop-filter: blur(15px); box-shadow: var(--shadow-card); cursor: pointer; display: flex; align-items: center; justify-content: center; font-size: 1.35rem; color: var(--text-primary); transition: var(--transition); z-index: 1100; }
    .theme-toggle:hover { transform: scale(1.15) rotate(180deg); }

    .container { max-width: 1100px; margin: 0 auto; padding: 2rem; z-index: 1; position: relative; }
    .header { text-align: center; margin-bottom: 2rem; position: relative; padding-top: 1.2rem; }
    .header-badge { display: inline-flex; align-items: center; gap: 0.6rem; background: var(--bg-secondary); border: 1px solid var(--border-color); border-radius: 999px; padding: 0.7rem 1rem; backdrop-filter: blur(15px); box-shadow: var(--shadow-card); margin-bottom: 1rem; color: var(--text-secondary); font-size: 0.95rem; }

    .main-title { font-size: clamp(2.2rem, 5vw, 3.8rem); font-weight: 800; background: var(--accent-gradient); -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text; margin-bottom: 1rem; line-height: 1.15; }
    .subtitle { color: var(--text-secondary); max-width: 900px; margin: 0 auto; font-size: 1.03rem; }

    .content-block, .quick-nav, .control-bar, .topic-card { background: var(--bg-secondary); backdrop-filter: blur(20px); border-radius: var(--border-radius); box-shadow: var(--shadow-card); border: 2px solid var(--border-color); }
    .content-block { padding: 2rem; margin-top: 2rem; }
    .content-block h2 { color: var(--text-primary); margin-bottom: 1rem; font-size: 1.55rem; }
    .content-block p, .content-block li, .subtitle { color: var(--text-secondary); }

    .quick-nav { margin-top: 2rem; padding: 1.4rem; }
    .quick-nav h3 { margin-bottom: 1rem; color: var(--text-primary); font-size: 1.2rem; }
    .nav-grid { display: flex; flex-wrap: wrap; gap: 0.75rem; }
    .nav-chip { padding: 0.65rem 0.95rem; border-radius: 999px; background: var(--nav-chip-bg); color: var(--text-primary); border: 1px solid var(--border-color); font-size: 0.92rem; transition: var(--transition); box-shadow: 0 8px 18px rgba(0,0,0,0.06); cursor: pointer; font-family: inherit; text-align: left; }
    .nav-chip:hover { transform: translateY(-2px); background: var(--bg-tertiary); }

    .control-bar { margin-top: 1.5rem; padding: 1rem 1.2rem; display: flex; gap: 0.8rem; flex-wrap: wrap; justify-content: center; }
    .control-btn { border: 1px solid var(--border-color); background: var(--nav-chip-bg); color: var(--text-primary); border-radius: 999px; padding: 0.75rem 1rem; cursor: pointer; font-family: inherit; font-size: 0.95rem; transition: var(--transition); }
    .control-btn:hover { transform: translateY(-2px); background: var(--bg-tertiary); }

    .lesson-container { display: flex; flex-direction: column; gap: 1.3rem; margin-top: 1.6rem; }
    .topic-card { overflow: hidden; transition: var(--transition); scroll-margin-top: 90px; }
    .topic-card:hover { transform: translateY(-2px); }
    .topic-header { cursor: pointer; padding: 1.5rem 2rem; display: flex; justify-content: space-between; align-items: center; gap: 1rem; user-select: none; }
    .topic-title { font-size: 1.22rem; font-weight: 700; color: var(--text-primary); line-height: 1.4; }
    .expand-icon { font-size: 1.15rem; color: var(--text-secondary); transition: var(--transition); flex-shrink: 0; }
    .topic-card.open .expand-icon { transform: rotate(180deg); }

    .topic-content { max-height: 0; overflow: hidden; transition: max-height 1.2s ease, padding 1.2s ease; background: var(--bg-tertiary); }
    .topic-card.open .topic-content { max-height: 6500px; padding: 1.6rem 2rem 1.8rem 2rem; border-top: 1px solid var(--border-color); }
    .topic-content h4 { color: var(--text-primary); margin-top: 1.5rem; margin-bottom: 0.55rem; font-size: 1.08rem; border-left: 4px solid var(--accent-primary); padding-left: 12px; }
    .topic-content h4:first-child { margin-top: 0; }
    .topic-content p, .topic-content li { color: var(--text-secondary); line-height: 1.8; margin-bottom: 0.95rem; }
    .topic-content strong { color: var(--text-primary); font-weight: 700; }

    .topic-content blockquote { text-align: center; border-left: 4px solid var(--accent-primary); margin: 1.4rem 0; font-style: normal; color: var(--text-secondary); background: rgba(127, 140, 141, 0.05); border-radius: 0 10px 10px 0; padding: 1rem 1.5rem; overflow-x: auto; }

    footer { text-align: center; margin-top: 4rem; padding-top: 2rem; border-top: 1px solid var(--border-color); }
    footer p { color: var(--text-secondary); font-size: 0.95rem; opacity: 0.9; }

    @media (max-width: 768px) {
      .container { padding: 1rem; }
      .topic-header { padding: 1.1rem 1.3rem; }
      .topic-card.open .topic-content { padding: 1.1rem 1.3rem 1.3rem 1.3rem; }
      .theme-toggle { width: 54px; height: 54px; top: 1rem; right: 1rem; }
    }
  </style>
</head>

<body data-theme="dark">
  <div class="progress-wrapper">
    <div class="progress-bar" id="progressBar"></div>
  </div>

  <div class="particles" id="particles-container"></div>

  <div class="theme-toggle" id="themeToggleButton" title="Cambiar tema">
    <i class="fas fa-moon" id="theme-icon"></i>
  </div>

  <div class="container">
    <header class="header">
      <div class="header-badge">
        <i class="fas fa-project-diagram"></i>
        <span>Estadística Aplicada a Data Science · Repositorio</span>
      </div>
      <h1 class="main-title">{{MAIN_TITLE}}</h1>
      <p class="subtitle">
        La red de equivalencias teóricas que conecta a los GP con Modelos Lineales Bayesianos, Splines, Kernel Methods y Redes Neuronales.
      </p>
    </header>

    {{INTRO_HTML}}

    <section class="quick-nav">
      <h3>Navegación rápida</h3>
      <div class="nav-grid">
        {{NAV_HTML}}
      </div>
    </section>

    <section class="control-bar">
      <button class="control-btn" id="expandAllBtn" type="button"><i class="fas fa-plus-circle"></i> Expandir todo</button>
      <button class="control-btn" id="collapseAllBtn" type="button"><i class="fas fa-minus-circle"></i> Contraer todo</button>
      <button class="control-btn" id="topBtn" type="button"><i class="fas fa-arrow-up"></i> Volver arriba</button>
    </section>

    <div class="lesson-container">
      {{DYNAMIC_HTML}}
    </div>

    <footer>
      <p>{{FOOTER_TEXT}}</p>
    </footer>
  </div>

  <script>
    (function() {
      const themeToggleButton = document.getElementById('themeToggleButton');
      const themeIcon = document.getElementById('theme-icon');
      const bodyEl = document.body;
      const expandAllBtn = document.getElementById('expandAllBtn');
      const collapseAllBtn = document.getElementById('collapseAllBtn');
      const topBtn = document.getElementById('topBtn');
      const progressBar = document.getElementById('progressBar');

      function setTheme(theme) {
        bodyEl.setAttribute('data-theme', theme);
        try { localStorage.setItem('gp_theme', theme); } catch (e) {}
        if (themeIcon) { themeIcon.className = theme === 'dark' ? 'fas fa-sun' : 'fas fa-moon'; }
      }

      if (themeToggleButton) {
        themeToggleButton.addEventListener('click', () => {
          const newTheme = (bodyEl.getAttribute('data-theme') || 'dark') === 'dark' ? 'light' : 'dark';
          setTheme(newTheme);
        });
      }

      let preferredTheme = 'dark';
      try { preferredTheme = localStorage.getItem('gp_theme') || 'dark'; } catch (e) {}
      setTheme(preferredTheme);

      const cards = Array.from(document.querySelectorAll('.topic-card'));

      cards.forEach(card => {
        const header = card.querySelector('.topic-header');
        if (header) {
          header.addEventListener('click', () => {
            card.classList.toggle('open');
          });
        }
      });

      if (expandAllBtn) { expandAllBtn.addEventListener('click', () => { cards.forEach(card => card.classList.add('open')); }); }
      if (collapseAllBtn) { collapseAllBtn.addEventListener('click', () => { cards.forEach(card => card.classList.remove('open')); }); }
      if (topBtn) { topBtn.addEventListener('click', () => { window.scrollTo({ top: 0, behavior: 'smooth' }); }); }

      const navButtons = Array.from(document.querySelectorAll('.nav-chip'));
      navButtons.forEach(btn => {
        btn.addEventListener('click', () => {
          const targetId = btn.getAttribute('data-target');
          const target = document.getElementById(targetId);
          if (target) {
            target.classList.add('open');
            setTimeout(() => { target.scrollIntoView({ behavior: 'smooth', block: 'start' }); }, 120);
          }
        });
      });

      const firstTopic = document.querySelector('.topic-card');
      if (firstTopic) { firstTopic.classList.add('open'); }

      const container = document.getElementById('particles-container');
      if (container) {
        for (let i = 0; i < 34; i++) {
          const p = document.createElement('div');
          p.className = 'particle';
          p.style.left = Math.random() * 100 + 'vw';
          const size = (Math.random() * 6 + 3);
          p.style.width = size + 'px';
          p.style.height = size + 'px';
          p.style.animationDelay = Math.random() * -20 + 's';
          p.style.animationDuration = (15 + Math.random() * 12) + 's';
          container.appendChild(p);
        }
      }

      function updateProgress() {
        const scrollTop = document.documentElement.scrollTop || document.body.scrollTop;
        const scrollHeight = document.documentElement.scrollHeight - document.documentElement.clientHeight;
        const progress = scrollHeight > 0 ? (scrollTop / scrollHeight) * 100 : 0;
        progressBar.style.width = progress + '%';
      }

      window.addEventListener('scroll', updateProgress);
      updateProgress();

      if (window.MathJax) {
          MathJax.Hub.Queue(["Typeset", MathJax.Hub]);
      }
    })();
  </script>
</body>
</html>
"""

# =========================================================
# 5. ENSAMBLADO
# =========================================================

final_html = (
    plantilla_profesional
    .replace("{{MAIN_TITLE}}", "8. Relación con otros métodos")
    .replace("{{INTRO_HTML}}", intro_html_content)
    .replace("{{NAV_HTML}}", navegacion_html)
    .replace("{{DYNAMIC_HTML}}", contenido_dinamico_html)
    .replace("{{FOOTER_TEXT}}", "Material elaborado por el profesor Sergio Gevatschnaider.")
)

display(HTML(final_html))

In [ ]:
# @title Índice Interactivo: 9. Limitaciones prácticas y costo computacional
from IPython.display import display, HTML
import html

# =========================================================
# 1. INTRODUCCIÓN
# =========================================================

intro_html_content = r"""
<div class="content-block" style="margin-top:0;">
    <h2>Resumen de la Sección</h2>
    <p>
        A pesar de su elegancia matemática y de su gran capacidad para modelar funciones con incertidumbre,
        los procesos gaussianos presentan una limitación práctica fundamental: su <strong>costo computacional
        crece muy rápidamente con el tamaño de la muestra</strong>.
    </p>
    <p>
        Esta dificultad no es un detalle de implementación, sino una consecuencia estructural de la propia
        formulación del modelo, que exige capturar todas las interacciones de covarianza entre todos los pares
        de observaciones. En esta sección analizaremos la raíz matemática de este problema, su impacto en
        escenarios de <em>Big Data</em> y la principal estrategia moderna para solucionarlo: las aproximaciones
        <em>sparse</em> y los <em>inducing points</em>.
    </p>
</div>
"""

# =========================================================
# 2. SECCIONES (Texto 100% completo y LaTeX corregido)
# =========================================================

secciones_principales_data = [
    {
        "id": "sec-1",
        "titulo": "1. El cuello de botella computacional: O(n³)",
        "contenido": r"""
            <h4>La matriz de covarianza exacta</h4>
            <p>
                En un proceso gaussiano, la dependencia entre todas las observaciones queda codificada en la matriz
                de covarianza construida a partir del kernel. Si $X=\{x_1,\dots,x_n\}$ es el conjunto de entradas de entrenamiento,
                entonces dicha matriz viene dada por:
            </p>
            <blockquote>$$K(X,X) = \left(k(x_i,x_j)\right)_{i,j=1}^n.$$</blockquote>
            <p>
                Cuando además se supone ruido gaussiano homocedástico, la matriz relevante para inferencia pasa a ser:
            </p>
            <blockquote>$$K_y = K(X,X) + \sigma^2 I.$$</blockquote>
            <h4>Predicción y Optimización</h4>
            <p>
                La predicción posterior para un nuevo punto $x_*$ requiere calcular la media y la varianza condicionales:
            </p>
            <blockquote>$$m_*(x_*) = k_*^T K_y^{-1} y, \qquad v_*(x_*) = k(x_*,x_*) - k_*^T K_y^{-1} k_*,$$</blockquote>
            <p>
                donde $k_* = \left(k(x_*,x_1),\dots,k(x_*,x_n)\right)^T$. Estas fórmulas muestran que toda predicción exige
                resolver sistemas lineales que involucran a la matriz $K_y$. Además, para aprender los hiperparámetros,
                debe evaluarse la verosimilitud marginal logarítmica:
            </p>
            <blockquote>$$\log p(y \mid X,\theta) = -\frac{1}{2}y^T K_y^{-1}y -\frac{1}{2}\log|K_y| -\frac{n}{2}\log(2\pi).$$</blockquote>
            <h4>Complejidad Cúbica</h4>
            <p>
                Desde el punto de vista numérico, la operación central se implementa mediante una factorización de Cholesky, $K_y = LL^T$,
                con $L$ triangular inferior. Factorizar una matriz densa de tamaño $n \times n$ tiene un costo temporal del orden de:
            </p>
            <blockquote>$$\mathcal{O}(n^3),$$</blockquote>
            <p>
                y requiere memoria del orden de $\mathcal{O}(n^2)$. Aunque el modelo sea muy expresivo, su uso exacto queda
                severamente restringido cuando el número de observaciones se vuelve grande.
            </p>
        """
    },
    {
        "id": "sec-2",
        "titulo": "2. El impacto en problemas de gran escala",
        "contenido": r"""
            <h4>Pérdida de competitividad en Big Data</h4>
            <p>
                Para tamaños muestrales pequeños o moderados, los GP proporcionan flexibilidad, inferencia bayesiana y cuantificación
                explícita de incertidumbre. En esos regímenes, el costo matricial es manejable. Sin embargo, a medida que $n$ crece,
                la situación cambia drásticamente.
            </p>
            <p>
                El primer problema es de <strong>memoria</strong>. La matriz $K(X,X)$ contiene $n^2$ entradas; su almacenamiento puede
                volverse prohibitivo. El segundo problema es de <strong>tiempo de cómputo</strong>: si el entrenamiento requiere optimizar
                hiperparámetros, la factorización $\mathcal{O}(n^3)$ debe repetirse múltiples veces. El tercero es de <strong>escalabilidad
                metodológica</strong>: el esquema exacto de GP no se adapta naturalmente a datos que llegan en flujos o mini-lotes (<em>mini-batches</em>).
            </p>
            <p>
                En aplicaciones modernas con cientos de miles de observaciones, un modelo $\mathcal{O}(n^3)$ pierde competitividad
                frente a Redes Neuronales o métodos estocásticos.
            </p>
            <h4>La carga de la dependencia global</h4>
            <p>
                La dificultad de los GP exactos no está en su capacidad predictiva, sino en la imposibilidad de sostenerla computacionalmente.
                Los GP exactos son modelos <strong>globales</strong>: cada nueva observación modifica la inferencia sobre toda la función latente.
                Esta interdependencia global es estadísticamente rica, pero numéricamente costosa.
            </p>
        """
    },
    {
        "id": "sec-3",
        "titulo": "3. Aproximaciones Sparse: Reduciendo la dependencia",
        "contenido": r"""
            <h4>La filosofía Sparse</h4>
            <p>
                Las aproximaciones <em>sparse</em> constituyen la respuesta más importante al problema computacional. La idea es reemplazar
                la dependencia completa entre los $n$ datos por una representación reducida que capture la estructura relevante con
                muchos menos grados de libertad.
            </p>
            <h4>Inducing Points (Puntos Inductores)</h4>
            <p>
                Para ello se introducen los llamados <em>inducing points</em>, que son un conjunto auxiliar de puntos:
            </p>
            <blockquote>$$Z = \{z_1,\dots,z_m\}, \qquad m \ll n,$$</blockquote>
            <p>
                elegidos para resumir el comportamiento de la función latente. Asociados a esos puntos se consideran los valores latentes $u = f(Z)$.
            </p>
            <p>
                La idea esencial es que, en lugar de condicionar directamente sobre los $n$ datos, se condiciona sobre este conjunto reducido
                de variables auxiliares. La información global del proceso se canaliza a través de un "cuello de botella" de dimensión $m$.
            </p>
        """
    },
    {
        "id": "sec-4",
        "titulo": "4. Matrices de rango bajo y reducción del costo",
        "contenido": r"""
            <h4>Reemplazo algebraico</h4>
            <p>
                Desde el punto de vista algebraico, se reemplaza la matriz completa $K(X,X)$ por una matriz de rango bajo construida
                a partir de los <em>inducing points</em>:
            </p>
            <blockquote>$$Q_{nn} = K_{nm} K_{mm}^{-1} K_{mn},$$</blockquote>
            <p>
                donde $K_{nm}$ es la matriz de covarianzas entre los datos observados y los <em>inducing points</em>, y $K_{mm}$
                es la matriz de covarianzas entre los propios <em>inducing points</em>.
            </p>
            <h4>Nuevo costo computacional</h4>
            <p>
                Esta representación tiene rango a lo sumo $m$, por lo que condensa la estructura dominante en un subespacio mucho más pequeño.
                La inferencia pasa a depender de bloques manejables, lo que reduce el costo típico a órdenes como:
            </p>
            <blockquote>$$\mathcal{O}(nm^2),$$</blockquote>
            <p>
                bajo el supuesto de que $m \ll n$.
            </p>
            <p>
                Los <em>inducing points</em> pueden interpretarse como un "esqueleto funcional": no describen cada detalle local,
                pero conservan la estructura global necesaria para reconstruir una buena aproximación posterior.
            </p>
        """
    },
    {
        "id": "sec-5",
        "titulo": "5. El compromiso entre Fidelidad y Escalabilidad",
        "contenido": r"""
            <h4>La tensión inevitable</h4>
            <p>
                La ganancia en escalabilidad introduce necesariamente una pérdida de exactitud. Si $m$ es demasiado pequeño, la representación
                puede ser incapaz de capturar variaciones locales importantes (subestima la incertidumbre o deteriora la predicción).
                Si $m$ crece demasiado, se vuelve más fiel al modelo exacto, pero el ahorro computacional desaparece.
            </p>
            <p>
                En la práctica, ya no se trata solo de elegir un buen kernel, sino de decidir qué esquema de aproximación utilizar, cuántos
                <em>inducing points</em> introducir y cómo optimizarlos.
            </p>
            <h4>Conclusión</h4>
            <p>
                Las técnicas <em>sparse</em> muestran una lección clave del aprendizaje estadístico moderno: cuando un modelo probabilístico es
                demasiado costoso para ser usado de forma exacta, basta con encontrar una <strong>representación reducida que conserve su
                geometría esencial</strong>.
            </p>
            <p>
                La limitación práctica de los GP proviene del costo $\mathcal{O}(n^3)$ de las matrices densas. Gracias a las aproximaciones
                <em>sparse</em> y los <em>inducing points</em>, los GP conservan gran parte de su riqueza inferencial reduciendo drásticamente
                el costo, aunque siempre bajo una tensión inevitable entre exactitud y viabilidad numérica.
            </p>
        """
    }
]

# =========================================================
# 3. GENERACIÓN HTML
# =========================================================

def generar_tarjetas_acordeon(datos):
    bloques = []
    for seccion in datos:
        bloques.append(f"""
        <div class="topic-card" id="{html.escape(seccion['id'])}">
            <div class="topic-header">
                <span class="topic-title">{seccion['titulo']}</span>
                <i class="fas fa-chevron-down expand-icon"></i>
            </div>
            <div class="topic-content">
                {seccion['contenido']}
            </div>
        </div>
        """)
    return "\n".join(bloques)

def generar_navegacion(datos):
    chips = []
    for seccion in datos:
        chips.append(
            f'<button class="nav-chip" type="button" data-target="{html.escape(seccion["id"])}">{seccion["titulo"]}</button>'
        )
    return "\n".join(chips)

contenido_dinamico_html = generar_tarjetas_acordeon(secciones_principales_data)
navegacion_html = generar_navegacion(secciones_principales_data)

# =========================================================
# 4. PLANTILLA HTML
# =========================================================

plantilla_profesional = r"""
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{{MAIN_TITLE}}</title>

  <script type="text/x-mathjax-config">
    MathJax.Hub.Config({
      tex2jax: {
        inlineMath: [['$','$']],
        displayMath: [['$$','$$']],
        processEscapes: true
      }
    });
  </script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-MML-AM_CHTML" async></script>

  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
  <link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css" rel="stylesheet">

  <style>
    :root {
      --bg-primary: linear-gradient(135deg, #f0f9ff 0%, #e0f2fe 100%);
      --bg-secondary: rgba(255, 255, 255, 0.85);
      --bg-tertiary: rgba(248, 250, 252, 0.8);
      --text-primary: #0c4a6e;
      --text-secondary: #075985;
      --accent-primary: #0ea5e9;
      --accent-secondary: #38bdf8;
      --accent-gradient: linear-gradient(135deg, var(--accent-primary) 0%, var(--accent-secondary) 100%);
      --border-color: rgba(226, 232, 240, 0.8);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.08);
      --border-radius: 20px;
      --transition: all 0.4s cubic-bezier(0.25, 0.8, 0.25, 1);
      --nav-chip-bg: rgba(255,255,255,0.8);
      --progress-bg: rgba(255,255,255,0.45);
    }

    [data-theme="dark"] {
      --bg-primary: linear-gradient(135deg, #0c4a6e 0%, #1e3a8a 100%);
      --bg-secondary: rgba(30, 58, 138, 0.85);
      --bg-tertiary: rgba(30, 64, 175, 0.7);
      --text-primary: #f0f9ff;
      --text-secondary: #dbeafe;
      --accent-primary: #38bdf8;
      --accent-secondary: #7dd3fc;
      --border-color: rgba(255, 255, 255, 0.15);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.20);
      --nav-chip-bg: rgba(30, 58, 138, 0.65);
      --progress-bg: rgba(255,255,255,0.15);
    }

    * { margin: 0; padding: 0; box-sizing: border-box; }
    html { scroll-behavior: smooth; }
    body { font-family: 'Inter', sans-serif; line-height: 1.8; background: var(--bg-primary); color: var(--text-primary); transition: var(--transition); min-height: 100vh; position: relative; overflow-x: hidden; }

    .particles { position: fixed; top: 0; left: 0; width: 100%; height: 100%; pointer-events: none; z-index: -1; }
    .particle { position: absolute; border-radius: 50%; animation: float 25s infinite linear; opacity: 0; background: rgba(255, 255, 255, 0.55); box-shadow: 0 0 15px rgba(255,255,255,0.25); }
    @keyframes float { 0% { transform: translateY(100vh) rotate(0deg); opacity: 0; } 10%, 90% { opacity: 0.55; } 100% { transform: translateY(-10vh) rotate(360deg); opacity: 0; } }

    .progress-wrapper { position: fixed; top: 0; left: 0; width: 100%; height: 6px; background: var(--progress-bg); z-index: 1200; backdrop-filter: blur(10px); }
    .progress-bar { height: 100%; width: 0%; background: var(--accent-gradient); transition: width 0.15s ease; }

    .theme-toggle { position: fixed; top: 1.2rem; right: 1.2rem; width: 60px; height: 60px; border: 1px solid var(--border-color); border-radius: 50%; background: var(--bg-secondary); backdrop-filter: blur(15px); box-shadow: var(--shadow-card); cursor: pointer; display: flex; align-items: center; justify-content: center; font-size: 1.35rem; color: var(--text-primary); transition: var(--transition); z-index: 1100; }
    .theme-toggle:hover { transform: scale(1.15) rotate(180deg); }

    .container { max-width: 1100px; margin: 0 auto; padding: 2rem; z-index: 1; position: relative; }
    .header { text-align: center; margin-bottom: 2rem; position: relative; padding-top: 1.2rem; }
    .header-badge { display: inline-flex; align-items: center; gap: 0.6rem; background: var(--bg-secondary); border: 1px solid var(--border-color); border-radius: 999px; padding: 0.7rem 1rem; backdrop-filter: blur(15px); box-shadow: var(--shadow-card); margin-bottom: 1rem; color: var(--text-secondary); font-size: 0.95rem; }

    .main-title { font-size: clamp(2.2rem, 5vw, 3.8rem); font-weight: 800; background: var(--accent-gradient); -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text; margin-bottom: 1rem; line-height: 1.15; }
    .subtitle { color: var(--text-secondary); max-width: 900px; margin: 0 auto; font-size: 1.03rem; }

    .content-block, .quick-nav, .control-bar, .topic-card { background: var(--bg-secondary); backdrop-filter: blur(20px); border-radius: var(--border-radius); box-shadow: var(--shadow-card); border: 2px solid var(--border-color); }
    .content-block { padding: 2rem; margin-top: 2rem; }
    .content-block h2 { color: var(--text-primary); margin-bottom: 1rem; font-size: 1.55rem; }
    .content-block p, .content-block li, .subtitle { color: var(--text-secondary); }

    .quick-nav { margin-top: 2rem; padding: 1.4rem; }
    .quick-nav h3 { margin-bottom: 1rem; color: var(--text-primary); font-size: 1.2rem; }
    .nav-grid { display: flex; flex-wrap: wrap; gap: 0.75rem; }
    .nav-chip { padding: 0.65rem 0.95rem; border-radius: 999px; background: var(--nav-chip-bg); color: var(--text-primary); border: 1px solid var(--border-color); font-size: 0.92rem; transition: var(--transition); box-shadow: 0 8px 18px rgba(0,0,0,0.06); cursor: pointer; font-family: inherit; text-align: left; }
    .nav-chip:hover { transform: translateY(-2px); background: var(--bg-tertiary); }

    .control-bar { margin-top: 1.5rem; padding: 1rem 1.2rem; display: flex; gap: 0.8rem; flex-wrap: wrap; justify-content: center; }
    .control-btn { border: 1px solid var(--border-color); background: var(--nav-chip-bg); color: var(--text-primary); border-radius: 999px; padding: 0.75rem 1rem; cursor: pointer; font-family: inherit; font-size: 0.95rem; transition: var(--transition); }
    .control-btn:hover { transform: translateY(-2px); background: var(--bg-tertiary); }

    .lesson-container { display: flex; flex-direction: column; gap: 1.3rem; margin-top: 1.6rem; }
    .topic-card { overflow: hidden; transition: var(--transition); scroll-margin-top: 90px; }
    .topic-card:hover { transform: translateY(-2px); }
    .topic-header { cursor: pointer; padding: 1.5rem 2rem; display: flex; justify-content: space-between; align-items: center; gap: 1rem; user-select: none; }
    .topic-title { font-size: 1.22rem; font-weight: 700; color: var(--text-primary); line-height: 1.4; }
    .expand-icon { font-size: 1.15rem; color: var(--text-secondary); transition: var(--transition); flex-shrink: 0; }
    .topic-card.open .expand-icon { transform: rotate(180deg); }

    .topic-content { max-height: 0; overflow: hidden; transition: max-height 1.2s ease, padding 1.2s ease; background: var(--bg-tertiary); }
    .topic-card.open .topic-content { max-height: 6500px; padding: 1.6rem 2rem 1.8rem 2rem; border-top: 1px solid var(--border-color); }
    .topic-content h4 { color: var(--text-primary); margin-top: 1.5rem; margin-bottom: 0.55rem; font-size: 1.08rem; border-left: 4px solid var(--accent-primary); padding-left: 12px; }
    .topic-content h4:first-child { margin-top: 0; }
    .topic-content p, .topic-content li { color: var(--text-secondary); line-height: 1.8; margin-bottom: 0.95rem; }
    .topic-content strong { color: var(--text-primary); font-weight: 700; }

    .topic-content blockquote { text-align: center; border-left: 4px solid var(--accent-primary); margin: 1.4rem 0; font-style: normal; color: var(--text-secondary); background: rgba(127, 140, 141, 0.05); border-radius: 0 10px 10px 0; padding: 1rem 1.5rem; overflow-x: auto; }

    footer { text-align: center; margin-top: 4rem; padding-top: 2rem; border-top: 1px solid var(--border-color); }
    footer p { color: var(--text-secondary); font-size: 0.95rem; opacity: 0.9; }

    @media (max-width: 768px) {
      .container { padding: 1rem; }
      .topic-header { padding: 1.1rem 1.3rem; }
      .topic-card.open .topic-content { padding: 1.1rem 1.3rem 1.3rem 1.3rem; }
      .theme-toggle { width: 54px; height: 54px; top: 1rem; right: 1rem; }
    }
  </style>
</head>

<body data-theme="dark">
  <div class="progress-wrapper">
    <div class="progress-bar" id="progressBar"></div>
  </div>

  <div class="particles" id="particles-container"></div>

  <div class="theme-toggle" id="themeToggleButton" title="Cambiar tema">
    <i class="fas fa-moon" id="theme-icon"></i>
  </div>

  <div class="container">
    <header class="header">
      <div class="header-badge">
        <i class="fas fa-project-diagram"></i>
        <span>Estadística Aplicada a Data Science · Repositorio</span>
      </div>
      <h1 class="main-title">{{MAIN_TITLE}}</h1>
      <p class="subtitle">
        Análisis del cuello de botella computacional de $\mathcal{O}(n^3)$ y la solución metodológica a través de aproximaciones Sparse y Puntos Inductores.
      </p>
    </header>

    {{INTRO_HTML}}

    <section class="quick-nav">
      <h3>Navegación rápida</h3>
      <div class="nav-grid">
        {{NAV_HTML}}
      </div>
    </section>

    <section class="control-bar">
      <button class="control-btn" id="expandAllBtn" type="button"><i class="fas fa-plus-circle"></i> Expandir todo</button>
      <button class="control-btn" id="collapseAllBtn" type="button"><i class="fas fa-minus-circle"></i> Contraer todo</button>
      <button class="control-btn" id="topBtn" type="button"><i class="fas fa-arrow-up"></i> Volver arriba</button>
    </section>

    <div class="lesson-container">
      {{DYNAMIC_HTML}}
    </div>

    <footer>
      <p>{{FOOTER_TEXT}}</p>
    </footer>
  </div>

  <script>
    (function() {
      const themeToggleButton = document.getElementById('themeToggleButton');
      const themeIcon = document.getElementById('theme-icon');
      const bodyEl = document.body;
      const expandAllBtn = document.getElementById('expandAllBtn');
      const collapseAllBtn = document.getElementById('collapseAllBtn');
      const topBtn = document.getElementById('topBtn');
      const progressBar = document.getElementById('progressBar');

      function setTheme(theme) {
        bodyEl.setAttribute('data-theme', theme);
        try { localStorage.setItem('gp_theme', theme); } catch (e) {}
        if (themeIcon) { themeIcon.className = theme === 'dark' ? 'fas fa-sun' : 'fas fa-moon'; }
      }

      if (themeToggleButton) {
        themeToggleButton.addEventListener('click', () => {
          const newTheme = (bodyEl.getAttribute('data-theme') || 'dark') === 'dark' ? 'light' : 'dark';
          setTheme(newTheme);
        });
      }

      let preferredTheme = 'dark';
      try { preferredTheme = localStorage.getItem('gp_theme') || 'dark'; } catch (e) {}
      setTheme(preferredTheme);

      const cards = Array.from(document.querySelectorAll('.topic-card'));

      cards.forEach(card => {
        const header = card.querySelector('.topic-header');
        if (header) {
          header.addEventListener('click', () => {
            card.classList.toggle('open');
          });
        }
      });

      if (expandAllBtn) { expandAllBtn.addEventListener('click', () => { cards.forEach(card => card.classList.add('open')); }); }
      if (collapseAllBtn) { collapseAllBtn.addEventListener('click', () => { cards.forEach(card => card.classList.remove('open')); }); }
      if (topBtn) { topBtn.addEventListener('click', () => { window.scrollTo({ top: 0, behavior: 'smooth' }); }); }

      const navButtons = Array.from(document.querySelectorAll('.nav-chip'));
      navButtons.forEach(btn => {
        btn.addEventListener('click', () => {
          const targetId = btn.getAttribute('data-target');
          const target = document.getElementById(targetId);
          if (target) {
            target.classList.add('open');
            setTimeout(() => { target.scrollIntoView({ behavior: 'smooth', block: 'start' }); }, 120);
          }
        });
      });

      const firstTopic = document.querySelector('.topic-card');
      if (firstTopic) { firstTopic.classList.add('open'); }

      const container = document.getElementById('particles-container');
      if (container) {
        for (let i = 0; i < 34; i++) {
          const p = document.createElement('div');
          p.className = 'particle';
          p.style.left = Math.random() * 100 + 'vw';
          const size = (Math.random() * 6 + 3);
          p.style.width = size + 'px';
          p.style.height = size + 'px';
          p.style.animationDelay = Math.random() * -20 + 's';
          p.style.animationDuration = (15 + Math.random() * 12) + 's';
          container.appendChild(p);
        }
      }

      function updateProgress() {
        const scrollTop = document.documentElement.scrollTop || document.body.scrollTop;
        const scrollHeight = document.documentElement.scrollHeight - document.documentElement.clientHeight;
        const progress = scrollHeight > 0 ? (scrollTop / scrollHeight) * 100 : 0;
        progressBar.style.width = progress + '%';
      }

      window.addEventListener('scroll', updateProgress);
      updateProgress();

      if (window.MathJax) {
          MathJax.Hub.Queue(["Typeset", MathJax.Hub]);
      }
    })();
  </script>
</body>
</html>
"""

# =========================================================
# 5. ENSAMBLADO
# =========================================================

final_html = (
    plantilla_profesional
    .replace("{{MAIN_TITLE}}", "9. Limitaciones prácticas")
    .replace("{{INTRO_HTML}}", intro_html_content)
    .replace("{{NAV_HTML}}", navegacion_html)
    .replace("{{DYNAMIC_HTML}}", contenido_dinamico_html)
    .replace("{{FOOTER_TEXT}}", "Material elaborado por el profesor Sergio Gevatschnaider.")
)

display(HTML(final_html))

In [ ]:
# @title Índice Interactivo: 10. Aplicaciones
from IPython.display import display, HTML
import html

# =========================================================
# 1. INTRODUCCIÓN
# =========================================================

intro_html_content = r"""
<div class="content-block" style="margin-top:0;">
    <h2>Resumen de la Sección</h2>
    <p>
        La relevancia de los Procesos Gaussianos no proviene únicamente de su elegancia teórica, sino también de su enorme
        versatilidad aplicada. Su valor distintivo frente a muchos otros métodos de aprendizaje automático es que no solo
        producen una predicción puntual, sino una <strong>distribución posterior sobre funciones</strong>.
    </p>
    <p>
        Esto significa que, en cada problema, el modelo entrega simultáneamente una estimación y una <strong>medida explícita
        de incertidumbre</strong>. Esa combinación los vuelve especialmente adecuados en contextos donde predecir bien no es
        suficiente y resulta igual de importante saber cuán confiable es dicha predicción. A continuación exploraremos sus
        principales campos de aplicación en la ciencia y la industria.
    </p>
</div>
"""

# =========================================================
# 2. SECCIONES (Texto 100% completo y LaTeX corregido)
# =========================================================

secciones_principales_data = [
    {
        "id": "sec-1",
        "titulo": "1. Regresión probabilística",
        "contenido": r"""
            <p>
                La aplicación más inmediata y clásica de los procesos gaussianos es la regresión probabilística. En este contexto, se observa un conjunto de pares $(x_i,y_i)$, y se asume que las respuestas fueron generadas según
            </p>
            <blockquote>$$y_i = f(x_i) + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0,\sigma^2),$$</blockquote>
            <p>
                donde $f$ es una función desconocida sobre la cual se coloca un prior de proceso gaussiano,
            </p>
            <blockquote>$$f \sim \mathcal{GP}(m,k).$$</blockquote>
            <p>
                A partir de esta formulación, el objetivo no es simplemente ajustar una curva, sino inferir una distribución posterior sobre todas las funciones plausibles compatibles con los datos observados. La ventaja práctica de este enfoque es que la predicción en un nuevo punto $x_*$ no se resume en un único valor estimado, sino en una distribución normal condicional con media y varianza dadas por
            </p>
            <blockquote>$$m_*(x_*) = k_*^T(K+\sigma^2 I)^{-1}y,$$</blockquote>
            <blockquote>$$v_*(x_*) = k(x_*,x_*) - k_*^T(K+\sigma^2 I)^{-1}k_*.$$</blockquote>
            <p>
                La media posterior actúa como estimación de la respuesta, mientras que la varianza posterior cuantifica la incertidumbre residual del modelo en ese punto. Esta propiedad es particularmente valiosa cuando los datos son escasos, ruidosos o se encuentran distribuidos de manera desigual en el espacio de entrada.
            </p>
            <p>
                Un ejemplo simple es la estimación de una relación no lineal entre temperatura y consumo energético. Un modelo lineal puede imponer una estructura demasiado rígida, mientras que un proceso gaussiano permite capturar curvaturas suaves, zonas de saturación o comportamientos locales sin necesidad de especificar una forma funcional cerrada. Además, en regiones con pocas observaciones, el modelo no solo predice con más cautela, sino que lo hace explícitamente a través de una banda de incertidumbre más amplia. Esa capacidad de distinguir entre "sé bastante" y "sé poco" es central en aplicaciones científicas, médicas e ingenieriles.
            </p>
            <p>
                En términos más generales, la regresión con GP resulta útil cuando se desea interpolar o extrapolar de manera controlada, preservando una interpretación bayesiana clara. A diferencia de muchos métodos puramente predictivos, aquí la incertidumbre no es un agregado posterior, sino parte constitutiva del modelo.
            </p>
        """
    },
    {
        "id": "sec-2",
        "titulo": "2. Optimización Bayesiana (Bayesian Optimization)",
        "contenido": r"""
            <p>
                Una de las aplicaciones más influyentes de los procesos gaussianos en <em>machine learning</em> moderno es la optimización bayesiana. Este enfoque se utiliza cuando se desea optimizar una función desconocida $f(x)$, pero cada evaluación de esa función es costosa, lenta o experimentalmente difícil. En ese escenario, no es viable explorar el dominio de manera exhaustiva, y se vuelve necesario elegir secuencialmente puntos de evaluación que aporten la mayor información posible.
            </p>
            <p>
                La idea central consiste en usar un proceso gaussiano como modelo probabilístico de la función objetivo. Después de observar algunos pares $(x_i,f(x_i))$, el GP produce una media posterior $m(x)$ y una varianza posterior $v(x)$ sobre cada punto candidato del dominio. Esa información se combina mediante una <strong>función de adquisición</strong>, cuyo propósito es decidir dónde evaluar a continuación. Entre las funciones de adquisición más conocidas se encuentran <em>Expected Improvement</em>, <em>Probability of Improvement</em> y <em>Upper Confidence Bound</em>. Todas comparten la misma filosofía: balancear explotación y exploración. La explotación favorece regiones donde la media posterior sugiere buenos valores; la exploración favorece regiones donde la incertidumbre es alta.
            </p>
            <p>
                Un ejemplo clásico es el ajuste de hiperparámetros de un modelo de aprendizaje automático. Entrenar una red neuronal profunda para cada combinación posible de tasa de aprendizaje, profundidad, regularización o tamaño de <em>batch</em> puede ser prohibitivamente caro. En lugar de eso, la optimización bayesiana construye un GP sobre el espacio de hiperparámetros y usa la función de adquisición para decidir qué configuración evaluar en cada iteración. De este modo, se puede encontrar una buena configuración con muchas menos evaluaciones que una búsqueda en grilla o aleatoria.
            </p>
            <p>
                Otro ejemplo importante aparece en experimentación científica y diseño de materiales. Si cada evaluación implica correr una simulación compleja o realizar un experimento de laboratorio costoso, resulta crucial no desperdiciar evaluaciones. El proceso gaussiano actúa allí como un modelo de creencias actualizado secuencialmente, y la optimización bayesiana convierte esa creencia en una estrategia racional de muestreo. Esta aplicación ilustra de forma especialmente clara por qué los GP son valiosos: no solo predicen, sino que <strong>guían decisiones bajo incertidumbre</strong>.
            </p>
        """
    },
    {
        "id": "sec-3",
        "titulo": "3. Series temporales",
        "contenido": r"""
            <p>
                Los procesos gaussianos también son especialmente útiles en el modelado de series temporales. En este contexto, el tiempo se trata como variable de entrada, y el objetivo es inferir una función latente que describa la evolución del fenómeno observado. A diferencia de modelos puramente autoregresivos, los GP permiten incorporar de manera flexible estructuras como suavidad, periodicidad, tendencias de largo plazo, cambios de escala y combinaciones de varios patrones simultáneamente a través del diseño del kernel.
            </p>
            <p>
                Si $t$ representa el tiempo y $y(t)$ la observación correspondiente, un proceso gaussiano sobre el tiempo se escribe como
            </p>
            <blockquote>$$f(t) \sim \mathcal{GP}(m(t),k(t,t')).$$</blockquote>
            <p>
                La clave aplicada está en elegir un kernel que refleje propiedades temporales plausibles. Un kernel RBF modela trayectorias suaves; un kernel periódico capta oscilaciones repetitivas; un kernel lineal puede representar tendencia global; sumas y productos de kernels permiten construir modelos temporales más ricos. Por ejemplo,
            </p>
            <blockquote>$$k(t,t') = k_{\text{tendencia}}(t,t') + k_{\text{periódico}}(t,t') + k_{\text{ruido}}(t,t')$$</blockquote>
            <p>
                permite describir una serie con crecimiento suave, estacionalidad y fluctuaciones residuales.
            </p>
            <p>
                Un ejemplo natural es la modelización de demanda eléctrica diaria. En este tipo de datos suele haber una tendencia de mediano plazo, patrones semanales o estacionales y ruido asociado a condiciones coyunturales. Un proceso gaussiano permite combinar estos componentes de manera unificada y generar no solo una predicción futura, sino una distribución posterior que informa cuánta incertidumbre existe en cada horizonte temporal. Esto es particularmente relevante cuando la serie presenta zonas con pocas observaciones, interrupciones o mediciones irregulares.
            </p>
            <p>
                Otra aplicación importante es el modelado de series biomédicas, como frecuencia cardíaca, glucosa o actividad cerebral. En esos contextos, la regularidad temporal existe, pero rara vez sigue una forma lineal simple. Los GP ofrecen una forma elegante de capturar dependencia temporal sin necesidad de imponer una estructura paramétrica rígida. Además, su flexibilidad para trabajar con datos faltantes o desigualmente espaciados los vuelve especialmente atractivos frente a métodos tradicionales que suponen muestreo uniforme.
            </p>
        """
    },
    {
        "id": "sec-4",
        "titulo": "4. Geostatística y datos espaciales",
        "contenido": r"""
            <p>
                La relación entre procesos gaussianos y geostatística es profunda y natural. De hecho, gran parte del desarrollo histórico de estos modelos está íntimamente conectado con el problema de inferir valores de una variable espacial a partir de observaciones dispersas en el territorio. En geostatística, una variable como temperatura, precipitación, contaminación, concentración mineral o humedad del suelo se interpreta como una realización de un campo aleatorio espacial. El proceso gaussiano proporciona exactamente ese marco.
            </p>
            <p>
                Si $s \in \mathbb{R}^d$ representa una ubicación espacial, puede modelarse
            </p>
            <blockquote>$$f(s) \sim \mathcal{GP}(m(s),k(s,s')).$$</blockquote>
            <p>
                El kernel codifica aquí la idea de autocorrelación espacial: puntos cercanos tienden a comportarse de manera más similar que puntos lejanos. Esta suposición se corresponde con muchas regularidades físicas y ambientales. La tarea de predecir el valor de la variable en una ubicación no observada, utilizando datos en ubicaciones vecinas, es precisamente la lógica del <em>kriging</em>, que puede interpretarse como una forma de inferencia bajo proceso gaussiano.
            </p>
            <p>
                Un ejemplo clásico es la estimación de concentración de contaminantes en una ciudad a partir de mediciones tomadas por un conjunto limitado de sensores. Como no es factible colocar sensores en cada punto del territorio, se utilizan las observaciones disponibles para inferir un mapa continuo. El proceso gaussiano no solo genera la superficie estimada, sino también un mapa de incertidumbre espacial. Esto permite identificar regiones donde la predicción es confiable y regiones donde convendría instalar nuevos sensores.
            </p>
            <p>
                En agricultura de precisión ocurre algo similar con variables como pH del suelo, humedad o presencia de nutrientes. En minería, con leyes minerales. En climatología, con temperatura o precipitación. En epidemiología espacial, con tasas de incidencia. En todos estos casos, el proceso gaussiano resulta valioso porque combina interpolación espacial, estructura correlacional e incertidumbre cuantificada en una misma herramienta. Desde un punto de vista metodológico, esta aplicación muestra que los GP no son solo un método de <em>machine learning</em> abstracto, sino también una extensión natural del razonamiento espacial clásico.
            </p>
        """
    },
    {
        "id": "sec-5",
        "titulo": "5. Robótica y control",
        "contenido": r"""
            <p>
                En robótica y control, los procesos gaussianos se utilizan para aprender dinámicas, modelar incertidumbre del entorno, construir mapas y apoyar la toma de decisiones en sistemas que deben interactuar con el mundo físico. En estos contextos, la principal ventaja del GP no es únicamente su capacidad de aproximación no lineal, sino el hecho de que la incertidumbre asociada a sus predicciones puede incorporarse directamente en el control o en la planificación.
            </p>
            <p>
                Un caso importante es el aprendizaje de la dinámica de un sistema. Supóngase que un robot tiene un estado $x_t$ y ejecuta una acción $u_t$. La transición al siguiente estado puede escribirse, de manera simplificada, como
            </p>
            <blockquote>$$x_{t+1} = f(x_t,u_t) + \varepsilon_t.$$</blockquote>
            <p>
                Cuando la función $f$ no se conoce con precisión o es demasiado compleja para modelarse desde primeros principios, puede aprenderse mediante un proceso gaussiano. El resultado es un modelo dinámico flexible que entrega no solo una predicción del próximo estado, sino también una estimación de cuán incierta es esa transición. Esto es crucial cuando el sistema debe tomar decisiones seguras y evitar zonas del espacio de estados donde el comportamiento es mal conocido.
            </p>
            <p>
                Un ejemplo típico es el control de un brazo robótico cuyas dinámicas reales difieren levemente del modelo teórico debido a fricción, desgaste o flexibilidad no modelada. Un GP puede aprender ese residuo entre el modelo físico ideal y el comportamiento observado, mejorando la precisión del control. En navegación autónoma, los GP también se usan para construir mapas continuos del entorno (mapas de ocupación, rugosidad del terreno, etc.).
            </p>
            <p>
                En planificación de trayectorias, la incertidumbre del GP puede emplearse para penalizar regiones donde el modelo sabe poco, favoreciendo decisiones prudentes. En control y robótica, no alcanza con una buena predicción media: es necesario conocer la confiabilidad de esa predicción, porque errores de modelado pueden traducirse en colisiones o fallas físicas.
            </p>
        """
    },
    {
        "id": "sec-6",
        "titulo": "6. Detección de anomalías",
        "contenido": r"""
            <p>
                Otra aplicación importante de los procesos gaussianos es la detección de anomalías o comportamientos atípicos. En estos problemas, el objetivo no es solo predecir una variable, sino identificar observaciones que se desvían significativamente del patrón normal esperado. Un GP resulta especialmente adecuado para esta tarea cuando las anomalías deben definirse en relación con una estructura funcional o correlacional subyacente, y no simplemente mediante distancia geométrica en el espacio de datos.
            </p>
            <p>
                La lógica es la siguiente: se entrena un proceso gaussiano sobre observaciones consideradas normales y se utiliza luego la distribución predictiva para evaluar nuevas observaciones. Si para una entrada $x_*$ el valor observado $y_*$ resulta muy improbable bajo la distribución posterior
            </p>
            <blockquote>$$y_* \mid x_*, \mathcal{D} \sim \mathcal{N}(m_*(x_*), v_*(x_*) + \sigma^2),$$</blockquote>
            <p>
                entonces esa observación puede ser marcada como potencial anomalía. El criterio puede basarse en residuos estandarizados, densidad predictiva baja o intervalos de credibilidad incumplidos.
            </p>
            <p>
                Un ejemplo claro aparece en monitoreo industrial. Supongamos que se registra la temperatura de una máquina en función del tiempo y de ciertas condiciones operativas. Si en un momento dado la temperatura observada excede de manera notoria el rango de incertidumbre esperado para esas condiciones, el sistema puede señalar una anomalía.
            </p>
            <p>
                La ventaja frente a métodos de umbral fijo es clara: el GP adapta el criterio de normalidad al régimen local de la señal. Una misma desviación puede ser normal en una región de alta incertidumbre y anómala en una región muy bien conocida.
            </p>
        """
    },
    {
        "id": "sec-7",
        "titulo": "7. Modelos sustitutos (Surrogate models) de simuladores costosos",
        "contenido": r"""
            <p>
                Una de las aplicaciones más estratégicas de los procesos gaussianos en ciencia computacional e ingeniería es la construcción de <em>surrogate models</em> o metamodelos de simuladores costosos. En muchos problemas, la función de interés proviene de un simulador físico o numérico cuya evaluación puede requerir horas de cómputo. En ese contexto, se utiliza un proceso gaussiano como aproximación probabilística del simulador.
            </p>
            <p>
                Sea $y = f(x)$ la salida de un simulador determinista, donde $x$ representa parámetros de diseño. Tras evaluar el simulador en un conjunto reducido de puntos, se ajusta un proceso gaussiano que aproxima la función subyacente. Este GP actúa como un emulador: predice la salida del simulador en puntos nuevos a un costo ínfimo. Además, informa incertidumbre sobre la calidad de esa aproximación.
            </p>
            <p>
                Esta estrategia aparece en diseño aerodinámico, ingeniería estructural, dinámica de fluidos y calibración de modelos climáticos. En este contexto, la incertidumbre posterior del GP cumple un papel crucial: informa dónde la aproximación es confiable y dónde conviene consultar al simulador real, guiando el diseño secuencial de experimentos.
            </p>
            <h4>Conclusión General</h4>
            <p>
                En conjunto, estas aplicaciones muestran que los procesos gaussianos ocupan un lugar singular dentro del aprendizaje estadístico. Son especialmente poderosos cuando el problema exige combinar flexibilidad funcional, estructura de dependencia e incertidumbre cuantificada. Su presencia en regresión probabilística, optimización bayesiana, modelado temporal, inferencia espacial, robótica, detección de anomalías y emulación de simuladores costosos no es accidental: en todos esos dominios, <strong>el hecho de modelar distribuciones sobre funciones, en lugar de solo predictores puntuales, constituye una ventaja decisiva.</strong>
            </p>
        """
    }
]

# =========================================================
# 3. GENERACIÓN HTML
# =========================================================

def generar_tarjetas_acordeon(datos):
    bloques = []
    for seccion in datos:
        bloques.append(f"""
        <div class="topic-card" id="{html.escape(seccion['id'])}">
            <div class="topic-header">
                <span class="topic-title">{seccion['titulo']}</span>
                <i class="fas fa-chevron-down expand-icon"></i>
            </div>
            <div class="topic-content">
                {seccion['contenido']}
            </div>
        </div>
        """)
    return "\n".join(bloques)

def generar_navegacion(datos):
    chips = []
    for seccion in datos:
        chips.append(
            f'<button class="nav-chip" type="button" data-target="{html.escape(seccion["id"])}">{seccion["titulo"]}</button>'
        )
    return "\n".join(chips)

contenido_dinamico_html = generar_tarjetas_acordeon(secciones_principales_data)
navegacion_html = generar_navegacion(secciones_principales_data)

# =========================================================
# 4. PLANTILLA HTML
# =========================================================

plantilla_profesional = r"""
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{{MAIN_TITLE}}</title>

  <script type="text/x-mathjax-config">
    MathJax.Hub.Config({
      tex2jax: {
        inlineMath: [['$','$']],
        displayMath: [['$$','$$']],
        processEscapes: true
      }
    });
  </script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-MML-AM_CHTML" async></script>

  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
  <link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css" rel="stylesheet">

  <style>
    :root {
      --bg-primary: linear-gradient(135deg, #f0f9ff 0%, #e0f2fe 100%);
      --bg-secondary: rgba(255, 255, 255, 0.85);
      --bg-tertiary: rgba(248, 250, 252, 0.8);
      --text-primary: #0c4a6e;
      --text-secondary: #075985;
      --accent-primary: #0ea5e9;
      --accent-secondary: #38bdf8;
      --accent-gradient: linear-gradient(135deg, var(--accent-primary) 0%, var(--accent-secondary) 100%);
      --border-color: rgba(226, 232, 240, 0.8);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.08);
      --border-radius: 20px;
      --transition: all 0.4s cubic-bezier(0.25, 0.8, 0.25, 1);
      --nav-chip-bg: rgba(255,255,255,0.8);
      --progress-bg: rgba(255,255,255,0.45);
    }

    [data-theme="dark"] {
      --bg-primary: linear-gradient(135deg, #0c4a6e 0%, #1e3a8a 100%);
      --bg-secondary: rgba(30, 58, 138, 0.85);
      --bg-tertiary: rgba(30, 64, 175, 0.7);
      --text-primary: #f0f9ff;
      --text-secondary: #dbeafe;
      --accent-primary: #38bdf8;
      --accent-secondary: #7dd3fc;
      --border-color: rgba(255, 255, 255, 0.15);
      --shadow-card: 0 15px 35px rgba(0, 0, 0, 0.20);
      --nav-chip-bg: rgba(30, 58, 138, 0.65);
      --progress-bg: rgba(255,255,255,0.15);
    }

    * { margin: 0; padding: 0; box-sizing: border-box; }
    html { scroll-behavior: smooth; }
    body { font-family: 'Inter', sans-serif; line-height: 1.8; background: var(--bg-primary); color: var(--text-primary); transition: var(--transition); min-height: 100vh; position: relative; overflow-x: hidden; }

    .particles { position: fixed; top: 0; left: 0; width: 100%; height: 100%; pointer-events: none; z-index: -1; }
    .particle { position: absolute; border-radius: 50%; animation: float 25s infinite linear; opacity: 0; background: rgba(255, 255, 255, 0.55); box-shadow: 0 0 15px rgba(255,255,255,0.25); }
    @keyframes float { 0% { transform: translateY(100vh) rotate(0deg); opacity: 0; } 10%, 90% { opacity: 0.55; } 100% { transform: translateY(-10vh) rotate(360deg); opacity: 0; } }

    .progress-wrapper { position: fixed; top: 0; left: 0; width: 100%; height: 6px; background: var(--progress-bg); z-index: 1200; backdrop-filter: blur(10px); }
    .progress-bar { height: 100%; width: 0%; background: var(--accent-gradient); transition: width 0.15s ease; }

    .theme-toggle { position: fixed; top: 1.2rem; right: 1.2rem; width: 60px; height: 60px; border: 1px solid var(--border-color); border-radius: 50%; background: var(--bg-secondary); backdrop-filter: blur(15px); box-shadow: var(--shadow-card); cursor: pointer; display: flex; align-items: center; justify-content: center; font-size: 1.35rem; color: var(--text-primary); transition: var(--transition); z-index: 1100; }
    .theme-toggle:hover { transform: scale(1.15) rotate(180deg); }

    .container { max-width: 1100px; margin: 0 auto; padding: 2rem; z-index: 1; position: relative; }
    .header { text-align: center; margin-bottom: 2rem; position: relative; padding-top: 1.2rem; }
    .header-badge { display: inline-flex; align-items: center; gap: 0.6rem; background: var(--bg-secondary); border: 1px solid var(--border-color); border-radius: 999px; padding: 0.7rem 1rem; backdrop-filter: blur(15px); box-shadow: var(--shadow-card); margin-bottom: 1rem; color: var(--text-secondary); font-size: 0.95rem; }

    .main-title { font-size: clamp(2.2rem, 5vw, 3.8rem); font-weight: 800; background: var(--accent-gradient); -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text; margin-bottom: 1rem; line-height: 1.15; }
    .subtitle { color: var(--text-secondary); max-width: 900px; margin: 0 auto; font-size: 1.03rem; }

    .content-block, .quick-nav, .control-bar, .topic-card { background: var(--bg-secondary); backdrop-filter: blur(20px); border-radius: var(--border-radius); box-shadow: var(--shadow-card); border: 2px solid var(--border-color); }
    .content-block { padding: 2rem; margin-top: 2rem; }
    .content-block h2 { color: var(--text-primary); margin-bottom: 1rem; font-size: 1.55rem; }
    .content-block p, .content-block li, .subtitle { color: var(--text-secondary); }

    .quick-nav { margin-top: 2rem; padding: 1.4rem; }
    .quick-nav h3 { margin-bottom: 1rem; color: var(--text-primary); font-size: 1.2rem; }
    .nav-grid { display: flex; flex-wrap: wrap; gap: 0.75rem; }
    .nav-chip { padding: 0.65rem 0.95rem; border-radius: 999px; background: var(--nav-chip-bg); color: var(--text-primary); border: 1px solid var(--border-color); font-size: 0.92rem; transition: var(--transition); box-shadow: 0 8px 18px rgba(0,0,0,0.06); cursor: pointer; font-family: inherit; text-align: left; }
    .nav-chip:hover { transform: translateY(-2px); background: var(--bg-tertiary); }

    .control-bar { margin-top: 1.5rem; padding: 1rem 1.2rem; display: flex; gap: 0.8rem; flex-wrap: wrap; justify-content: center; }
    .control-btn { border: 1px solid var(--border-color); background: var(--nav-chip-bg); color: var(--text-primary); border-radius: 999px; padding: 0.75rem 1rem; cursor: pointer; font-family: inherit; font-size: 0.95rem; transition: var(--transition); }
    .control-btn:hover { transform: translateY(-2px); background: var(--bg-tertiary); }

    .lesson-container { display: flex; flex-direction: column; gap: 1.3rem; margin-top: 1.6rem; }
    .topic-card { overflow: hidden; transition: var(--transition); scroll-margin-top: 90px; }
    .topic-card:hover { transform: translateY(-2px); }
    .topic-header { cursor: pointer; padding: 1.5rem 2rem; display: flex; justify-content: space-between; align-items: center; gap: 1rem; user-select: none; }
    .topic-title { font-size: 1.22rem; font-weight: 700; color: var(--text-primary); line-height: 1.4; }
    .expand-icon { font-size: 1.15rem; color: var(--text-secondary); transition: var(--transition); flex-shrink: 0; }
    .topic-card.open .expand-icon { transform: rotate(180deg); }

    .topic-content { max-height: 0; overflow: hidden; transition: max-height 1.2s ease, padding 1.2s ease; background: var(--bg-tertiary); }
    .topic-card.open .topic-content { max-height: 6500px; padding: 1.6rem 2rem 1.8rem 2rem; border-top: 1px solid var(--border-color); }
    .topic-content h4 { color: var(--text-primary); margin-top: 1.5rem; margin-bottom: 0.55rem; font-size: 1.08rem; border-left: 4px solid var(--accent-primary); padding-left: 12px; }
    .topic-content h4:first-child { margin-top: 0; }
    .topic-content p, .topic-content li { color: var(--text-secondary); line-height: 1.8; margin-bottom: 0.95rem; }
    .topic-content strong { color: var(--text-primary); font-weight: 700; }

    .topic-content blockquote { text-align: center; border-left: 4px solid var(--accent-primary); margin: 1.4rem 0; font-style: normal; color: var(--text-secondary); background: rgba(127, 140, 141, 0.05); border-radius: 0 10px 10px 0; padding: 1rem 1.5rem; overflow-x: auto; }

    footer { text-align: center; margin-top: 4rem; padding-top: 2rem; border-top: 1px solid var(--border-color); }
    footer p { color: var(--text-secondary); font-size: 0.95rem; opacity: 0.9; }

    @media (max-width: 768px) {
      .container { padding: 1rem; }
      .topic-header { padding: 1.1rem 1.3rem; }
      .topic-card.open .topic-content { padding: 1.1rem 1.3rem 1.3rem 1.3rem; }
      .theme-toggle { width: 54px; height: 54px; top: 1rem; right: 1rem; }
    }
  </style>
</head>

<body data-theme="dark">
  <div class="progress-wrapper">
    <div class="progress-bar" id="progressBar"></div>
  </div>

  <div class="particles" id="particles-container"></div>

  <div class="theme-toggle" id="themeToggleButton" title="Cambiar tema">
    <i class="fas fa-moon" id="theme-icon"></i>
  </div>

  <div class="container">
    <header class="header">
      <div class="header-badge">
        <i class="fas fa-project-diagram"></i>
        <span>Estadística Aplicada a Data Science · Repositorio</span>
      </div>
      <h1 class="main-title">{{MAIN_TITLE}}</h1>
      <p class="subtitle">
        Desde la optimización bayesiana hasta la detección de anomalías: el impacto de la incertidumbre cuantificada en problemas reales de Data Science e Ingeniería.
      </p>
    </header>

    {{INTRO_HTML}}

    <section class="quick-nav">
      <h3>Navegación rápida</h3>
      <div class="nav-grid">
        {{NAV_HTML}}
      </div>
    </section>

    <section class="control-bar">
      <button class="control-btn" id="expandAllBtn" type="button"><i class="fas fa-plus-circle"></i> Expandir todo</button>
      <button class="control-btn" id="collapseAllBtn" type="button"><i class="fas fa-minus-circle"></i> Contraer todo</button>
      <button class="control-btn" id="topBtn" type="button"><i class="fas fa-arrow-up"></i> Volver arriba</button>
    </section>

    <div class="lesson-container">
      {{DYNAMIC_HTML}}
    </div>

    <footer>
      <p>{{FOOTER_TEXT}}</p>
    </footer>
  </div>

  <script>
    (function() {
      const themeToggleButton = document.getElementById('themeToggleButton');
      const themeIcon = document.getElementById('theme-icon');
      const bodyEl = document.body;
      const expandAllBtn = document.getElementById('expandAllBtn');
      const collapseAllBtn = document.getElementById('collapseAllBtn');
      const topBtn = document.getElementById('topBtn');
      const progressBar = document.getElementById('progressBar');

      function setTheme(theme) {
        bodyEl.setAttribute('data-theme', theme);
        try { localStorage.setItem('gp_theme', theme); } catch (e) {}
        if (themeIcon) { themeIcon.className = theme === 'dark' ? 'fas fa-sun' : 'fas fa-moon'; }
      }

      if (themeToggleButton) {
        themeToggleButton.addEventListener('click', () => {
          const newTheme = (bodyEl.getAttribute('data-theme') || 'dark') === 'dark' ? 'light' : 'dark';
          setTheme(newTheme);
        });
      }

      let preferredTheme = 'dark';
      try { preferredTheme = localStorage.getItem('gp_theme') || 'dark'; } catch (e) {}
      setTheme(preferredTheme);

      const cards = Array.from(document.querySelectorAll('.topic-card'));

      cards.forEach(card => {
        const header = card.querySelector('.topic-header');
        if (header) {
          header.addEventListener('click', () => {
            card.classList.toggle('open');
          });
        }
      });

      if (expandAllBtn) { expandAllBtn.addEventListener('click', () => { cards.forEach(card => card.classList.add('open')); }); }
      if (collapseAllBtn) { collapseAllBtn.addEventListener('click', () => { cards.forEach(card => card.classList.remove('open')); }); }
      if (topBtn) { topBtn.addEventListener('click', () => { window.scrollTo({ top: 0, behavior: 'smooth' }); }); }

      const navButtons = Array.from(document.querySelectorAll('.nav-chip'));
      navButtons.forEach(btn => {
        btn.addEventListener('click', () => {
          const targetId = btn.getAttribute('data-target');
          const target = document.getElementById(targetId);
          if (target) {
            target.classList.add('open');
            setTimeout(() => { target.scrollIntoView({ behavior: 'smooth', block: 'start' }); }, 120);
          }
        });
      });

      const firstTopic = document.querySelector('.topic-card');
      if (firstTopic) { firstTopic.classList.add('open'); }

      const container = document.getElementById('particles-container');
      if (container) {
        for (let i = 0; i < 34; i++) {
          const p = document.createElement('div');
          p.className = 'particle';
          p.style.left = Math.random() * 100 + 'vw';
          const size = (Math.random() * 6 + 3);
          p.style.width = size + 'px';
          p.style.height = size + 'px';
          p.style.animationDelay = Math.random() * -20 + 's';
          p.style.animationDuration = (15 + Math.random() * 12) + 's';
          container.appendChild(p);
        }
      }

      function updateProgress() {
        const scrollTop = document.documentElement.scrollTop || document.body.scrollTop;
        const scrollHeight = document.documentElement.scrollHeight - document.documentElement.clientHeight;
        const progress = scrollHeight > 0 ? (scrollTop / scrollHeight) * 100 : 0;
        progressBar.style.width = progress + '%';
      }

      window.addEventListener('scroll', updateProgress);
      updateProgress();

      if (window.MathJax) {
          MathJax.Hub.Queue(["Typeset", MathJax.Hub]);
      }
    })();
  </script>
</body>
</html>
"""

# =========================================================
# 5. ENSAMBLADO
# =========================================================

final_html = (
    plantilla_profesional
    .replace("{{MAIN_TITLE}}", "10. Aplicaciones")
    .replace("{{INTRO_HTML}}", intro_html_content)
    .replace("{{NAV_HTML}}", navegacion_html)
    .replace("{{DYNAMIC_HTML}}", contenido_dinamico_html)
    .replace("{{FOOTER_TEXT}}", "Material elaborado por el profesor Sergio Gevatschnaider.")
)

display(HTML(final_html))